In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — LOAD ROUND 5 DATA
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROUND = 5
DAYS = [2, 3, 4]

DATA_DIR = Path("Data/round5")

ALGO_PRODUCTS = [
    # Galaxy Sounds Recorders
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_WINDS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",

    # Vertical Sleeping Pods
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",

    # Organic Microchips
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",

    # Purification Pebbles
    "PEBBLES_XS",
    "PEBBLES_S",
    "PEBBLES_M",
    "PEBBLES_L",
    "PEBBLES_XL",

    # Domestic Robots
    "ROBOT_VACUUMING",
    "ROBOT_MOPPING",
    "ROBOT_DISHES",
    "ROBOT_LAUNDRY",
    "ROBOT_IRONING",

    # UV-Visors
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",

    # Instant Translators
    "TRANSLATOR_SPACE_GRAY",
    "TRANSLATOR_ASTRO_BLACK",
    "TRANSLATOR_ECLIPSE_CHARCOAL",
    "TRANSLATOR_GRAPHITE_MIST",
    "TRANSLATOR_VOID_BLUE",

    # Construction Panels
    "PANEL_1X2",
    "PANEL_2X2",
    "PANEL_1X4",
    "PANEL_2X4",
    "PANEL_4X4",

    # Liquid Breath Oxygen Shakes
    "OXYGEN_SHAKE_MORNING_BREATH",
    "OXYGEN_SHAKE_EVENING_BREATH",
    "OXYGEN_SHAKE_MINT",
    "OXYGEN_SHAKE_CHOCOLATE",
    "OXYGEN_SHAKE_GARLIC",

    # Protein Snack Packs
    "SNACKPACK_CHOCOLATE",
    "SNACKPACK_VANILLA",
    "SNACKPACK_PISTACHIO",
    "SNACKPACK_STRAWBERRY",
    "SNACKPACK_RASPBERRY",
]

POSITION_LIMITS = {product: 10 for product in ALGO_PRODUCTS}

prices_parts = []
trades_parts = []

for day in DAYS:
    price_path = DATA_DIR / f"prices_round_{ROUND}_day_{day}.csv"
    trade_path = DATA_DIR / f"trades_round_{ROUND}_day_{day}.csv"

    p = pd.read_csv(price_path, sep=";")
    t = pd.read_csv(trade_path, sep=";")

    p["file_day"] = day
    t["file_day"] = day

    prices_parts.append(p)
    trades_parts.append(t)

prices = pd.concat(prices_parts, ignore_index=True)
trades = pd.concat(trades_parts, ignore_index=True)

# Standardise trade product column name.
if "symbol" in trades.columns and "product" not in trades.columns:
    trades = trades.rename(columns={"symbol": "product"})

# Keep only valid Round 5 algorithmic products.
prices = prices[prices["product"].isin(ALGO_PRODUCTS)].copy()
trades = trades[trades["product"].isin(ALGO_PRODUCTS)].copy()

# Useful global time index across days.
# Assumes timestamp resets each day.
min_day = min(DAYS)
prices["global_ts"] = (prices["file_day"] - min_day) * 1_000_000 + prices["timestamp"]
trades["global_ts"] = (trades["file_day"] - min_day) * 1_000_000 + trades["timestamp"]

prices = prices.sort_values(["product", "global_ts"]).reset_index(drop=True)
trades = trades.sort_values(["product", "global_ts"]).reset_index(drop=True)

# Basic sanity checks.
price_products = sorted(prices["product"].unique())
trade_products = sorted(trades["product"].unique())

missing_in_prices = sorted(set(ALGO_PRODUCTS) - set(price_products))
missing_in_trades = sorted(set(ALGO_PRODUCTS) - set(trade_products))

print("prices shape:", prices.shape)
print("trades shape:", trades.shape)
print()
print("Price products:", price_products)
print("Trade products:", trade_products)
print()
print("Missing in prices:", missing_in_prices)
print("Missing in trades:", missing_in_trades)
print()
print("Price days:", sorted(prices["file_day"].unique()))
print("Trade days:", sorted(trades["file_day"].unique()))
print()
print("Position limits:", POSITION_LIMITS)

display(prices.head())
display(trades.head())

prices shape: (1500000, 19)
trades shape: (35385, 9)

Price products: ['GALAXY_SOUNDS_BLACK_HOLES', 'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_FLAMES', 'GALAXY_SOUNDS_SOLAR_WINDS', 'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_EVENING_BREATH', 'OXYGEN_SHAKE_GARLIC', 'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_MORNING_BREATH', 'PANEL_1X2', 'PANEL_1X4', 'PANEL_2X2', 'PANEL_2X4', 'PANEL_4X4', 'PEBBLES_L', 'PEBBLES_M', 'PEBBLES_S', 'PEBBLES_XL', 'PEBBLES_XS', 'ROBOT_DISHES', 'ROBOT_IRONING', 'ROBOT_LAUNDRY', 'ROBOT_MOPPING', 'ROBOT_VACUUMING', 'SLEEP_POD_COTTON', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_NYLON', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_SUEDE', 'SNACKPACK_CHOCOLATE', 'SNACKPACK_PISTACHIO', 'SNACKPACK_RASPBERRY', 'SNACKPACK_STRAWBERRY', 'SNACKPACK_VANILLA', 'TRANSLATOR_ASTRO_BLACK', 'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST', 'TRANSLATOR_SPACE_GRAY'

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,file_day,global_ts
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,2,0
1,2,100,GALAXY_SOUNDS_BLACK_HOLES,10001,18,10000.0,25.0,NaN,NaN,10014,18,10016.0,25.0,NaN,NaN,10007.5,0.0,2,100
2,2,200,GALAXY_SOUNDS_BLACK_HOLES,9996,19,9995.0,31.0,NaN,NaN,10009,19,10011.0,31.0,NaN,NaN,10002.5,0.0,2,200
3,2,300,GALAXY_SOUNDS_BLACK_HOLES,9994,25,9993.0,33.0,NaN,NaN,10007,25,10009.0,33.0,NaN,NaN,10000.5,0.0,2,300
4,2,400,GALAXY_SOUNDS_BLACK_HOLES,9999,14,9997.0,32.0,NaN,NaN,10012,14,10013.0,32.0,NaN,NaN,10005.5,0.0,2,400


,timestamp,buyer,seller,product,currency,price,quantity,file_day,global_ts
0,1700,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9969.0,4,2,1700
1,14500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9749.0,1,2,14500
2,15100,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9764.0,2,2,15100
3,26500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9656.0,4,2,26500
4,36400,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9675.0,4,2,36400


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — MICROCHIPS CONFIG + FEATURES
# ════════════════════════════════════════════════════════════════════════════

import time
import math
import ast
from pathlib import Path
from itertools import combinations, product as iter_product

import numpy as np
import pandas as pd

MICRO_OUT = Path("outputs_microchips_stage1")
MICRO_OUT.mkdir(exist_ok=True)

MICRO_PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

MICRO_META = pd.DataFrame([
    {"product": "MICROCHIP_CIRCLE", "shape_family": "curved", "edges_proxy": 0, "is_curved": 1, "is_angular": 0, "is_quadrilateral": 0},
    {"product": "MICROCHIP_OVAL", "shape_family": "curved", "edges_proxy": 0, "is_curved": 1, "is_angular": 0, "is_quadrilateral": 0},
    {"product": "MICROCHIP_SQUARE", "shape_family": "angular", "edges_proxy": 4, "is_curved": 0, "is_angular": 1, "is_quadrilateral": 1},
    {"product": "MICROCHIP_RECTANGLE", "shape_family": "angular", "edges_proxy": 4, "is_curved": 0, "is_angular": 1, "is_quadrilateral": 1},
    {"product": "MICROCHIP_TRIANGLE", "shape_family": "angular", "edges_proxy": 3, "is_curved": 0, "is_angular": 1, "is_quadrilateral": 0},
])

display(MICRO_META)

MICRO_WINDOWS = [250, 500, 1000, 2500]
MICRO_HORIZONS = [50, 100, 250, 500, 1000]
MICRO_THRESHOLDS = [1.0, 1.5, 2.0, 2.5]
MICRO_MODES = ["meanrev", "breakout"]

MICRO_PAST_HORIZONS = [10, 25, 50, 100, 250, 500]
MICRO_LEADLAG_HORIZONS = [10, 25, 50, 100, 250, 500]
MICRO_FLOW_WINDOWS = [500, 1000, 2500, 5000, 10000]

POS_LIMIT = 10


def add_book_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["mid"] = df["mid_price"]
    df["spread"] = df["ask_price_1"] - df["bid_price_1"]

    for col in [
        "bid_volume_1", "bid_volume_2", "bid_volume_3",
        "ask_volume_1", "ask_volume_2", "ask_volume_3",
    ]:
        df[col] = df[col].fillna(0)

    df["bid_depth_l1"] = df["bid_volume_1"]
    df["ask_depth_l1"] = df["ask_volume_1"]

    df["bid_depth_total"] = df["bid_volume_1"] + df["bid_volume_2"] + df["bid_volume_3"]
    df["ask_depth_total"] = df["ask_volume_1"] + df["ask_volume_2"] + df["ask_volume_3"]

    df["depth_total"] = df["bid_depth_total"] + df["ask_depth_total"]

    df["imbalance_l1"] = (
        (df["bid_depth_l1"] - df["ask_depth_l1"])
        / (df["bid_depth_l1"] + df["ask_depth_l1"]).replace(0, np.nan)
    )

    df["imbalance_total"] = (
        (df["bid_depth_total"] - df["ask_depth_total"])
        / (df["bid_depth_total"] + df["ask_depth_total"]).replace(0, np.nan)
    )

    df["microprice_l1"] = (
        (df["ask_price_1"] * df["bid_depth_l1"] + df["bid_price_1"] * df["ask_depth_l1"])
        / (df["bid_depth_l1"] + df["ask_depth_l1"]).replace(0, np.nan)
    )

    df["microprice_edge"] = df["microprice_l1"] - df["mid"]
    df["microprice_edge_norm"] = df["microprice_edge"] / df["spread"].replace(0, np.nan)

    return df


prices_f = add_book_features(prices)

micro_prices = (
    prices_f[prices_f["product"].isin(MICRO_PRODUCTS)]
    .sort_values(["file_day", "timestamp", "product"])
    .reset_index(drop=True)
)

micro_trades = (
    trades[trades["product"].isin(MICRO_PRODUCTS)]
    .sort_values(["file_day", "timestamp", "product"])
    .reset_index(drop=True)
)

print("micro_prices:", micro_prices.shape)
print("micro_trades:", micro_trades.shape)
print("products:", sorted(micro_prices["product"].unique()))

,product,shape_family,edges_proxy,is_curved,is_angular,is_quadrilateral
0,MICROCHIP_CIRCLE,curved,0,1,0,0
1,MICROCHIP_OVAL,curved,0,1,0,0
2,MICROCHIP_SQUARE,angular,4,0,1,1
3,MICROCHIP_RECTANGLE,angular,4,0,1,1
4,MICROCHIP_TRIANGLE,angular,3,0,1,0


micro_prices: (150000, 31)
micro_trades: (2845, 9)
products: ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE']


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — PIVOT ONCE + DAY ARRAYS
# ════════════════════════════════════════════════════════════════════════════

MICRO_START = time.perf_counter()

def pivot_micro(value_col: str) -> pd.DataFrame:
    return (
        micro_prices
        .pivot_table(
            index=["file_day", "timestamp"],
            columns="product",
            values=value_col,
            aggfunc="last",
        )
        .sort_index()
        .reindex(columns=MICRO_PRODUCTS)
    )


micro_mid = pivot_micro("mid_price")
micro_bid = pivot_micro("bid_price_1")
micro_ask = pivot_micro("ask_price_1")
micro_spread = pivot_micro("spread")
micro_imb_l1 = pivot_micro("imbalance_l1")
micro_imb_total = pivot_micro("imbalance_total")
micro_microprice_edge_norm = pivot_micro("microprice_edge_norm")

valid_rows = ~(
    micro_mid.isna().any(axis=1)
    | micro_bid.isna().any(axis=1)
    | micro_ask.isna().any(axis=1)
)

micro_mid = micro_mid.loc[valid_rows]
micro_bid = micro_bid.loc[valid_rows]
micro_ask = micro_ask.loc[valid_rows]
micro_spread = micro_spread.loc[valid_rows]
micro_imb_l1 = micro_imb_l1.loc[valid_rows]
micro_imb_total = micro_imb_total.loc[valid_rows]
micro_microprice_edge_norm = micro_microprice_edge_norm.loc[valid_rows]

assert list(micro_mid.columns) == MICRO_PRODUCTS
assert list(micro_bid.columns) == MICRO_PRODUCTS
assert list(micro_ask.columns) == MICRO_PRODUCTS

print("Aligned wide shape:", micro_mid.shape)
print("Products:", MICRO_PRODUCTS)

micro_product_to_idx = {p: i for i, p in enumerate(MICRO_PRODUCTS)}

MICRO_FAST_DAY_BASE = {}

for day in DAYS:
    X_df = micro_mid.loc[day]
    B_df = micro_bid.loc[day]
    A_df = micro_ask.loc[day]

    assert X_df.index.min() >= 0
    assert X_df.index.max() <= 999900
    assert X_df.shape == B_df.shape == A_df.shape

    MICRO_FAST_DAY_BASE[day] = {
        "timestamps": X_df.index.to_numpy(),
        "X": X_df.to_numpy(float),
        "B": B_df.to_numpy(float),
        "A": A_df.to_numpy(float),
        "spread": micro_spread.loc[day].to_numpy(float),
        "imb_l1": micro_imb_l1.loc[day].to_numpy(float),
        "imb_total": micro_imb_total.loc[day].to_numpy(float),
        "microprice_edge_norm": micro_microprice_edge_norm.loc[day].to_numpy(float),
    }

    print(f"Day {day}: X shape={MICRO_FAST_DAY_BASE[day]['X'].shape}")

# Basic diagnostics
micro_diag = (
    micro_prices
    .groupby(["file_day", "product"])
    .agg(
        rows=("timestamp", "count"),
        first_mid=("mid", "first"),
        last_mid=("mid", "last"),
        min_mid=("mid", "min"),
        max_mid=("mid", "max"),
        mid_range=("mid", lambda x: x.max() - x.min()),
        mean_spread=("spread", "mean"),
        median_spread=("spread", "median"),
        max_spread=("spread", "max"),
        mean_depth=("depth_total", "mean"),
        median_depth=("depth_total", "median"),
        std_l1_imb=("imbalance_l1", "std"),
    )
    .reset_index()
)

micro_trade_diag = (
    micro_trades
    .groupby(["file_day", "product"])
    .agg(
        trades=("price", "count"),
        total_qty=("quantity", "sum"),
        mean_trade_price=("price", "mean"),
        min_trade_price=("price", "min"),
        max_trade_price=("price", "max"),
    )
    .reset_index()
)

display(micro_diag)
display(micro_trade_diag)

micro_diag.to_csv(MICRO_OUT / "MICRO_diagnostics.csv", index=False)
micro_trade_diag.to_csv(MICRO_OUT / "MICRO_trade_diagnostics.csv", index=False)

Aligned wide shape: (30000, 5)
Products: ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE', 'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']
Day 2: X shape=(10000, 5)
Day 3: X shape=(10000, 5)
Day 4: X shape=(10000, 5)


,file_day,product,rows,first_mid,last_mid,min_mid,max_mid,mid_range,mean_spread,median_spread,max_spread,mean_depth,median_depth,std_l1_imb
0,2,MICROCHIP_CIRCLE,10000,10000.0,8875.0,8572.0,10082.5,1510.5,8.2583,8.0,10,32.1045,32.0,0.181626
1,2,MICROCHIP_OVAL,10000,10000.0,9256.0,9015.0,10433.0,1418.0,8.7157,9.0,10,32.1045,32.0,0.117198
2,2,MICROCHIP_RECTANGLE,10000,10000.0,9420.5,8948.5,10442.5,1494.0,8.5759,9.0,10,32.1045,32.0,0.147005
3,2,MICROCHIP_SQUARE,10000,10000.0,12455.5,9756.0,12508.5,2752.5,9.9049,10.0,11,32.1045,32.0,0.054601
4,2,MICROCHIP_TRIANGLE,10000,10000.0,10704.0,9477.5,10826.5,1349.0,9.0691,9.0,10,32.1045,32.0,0.063712
5,3,MICROCHIP_CIRCLE,10000,8887.5,8541.0,8303.5,9525.5,1222.0,8.0001,8.0,9,32.0181,32.0,0.203310
6,3,MICROCHIP_OVAL,10000,9252.5,7428.5,7342.5,9649.5,2307.0,7.7270,8.0,9,32.0181,32.0,0.228210
7,3,MICROCHIP_RECTANGLE,10000,9411.5,7839.0,7327.5,9490.5,2163.0,7.5291,8.0,9,32.0181,32.0,0.229875
8,3,MICROCHIP_SQUARE,10000,12457.5,15896.0,12247.5,16482.0,4234.5,12.4872,13.0,15,32.0181,32.0,0.054180
9,3,MICROCHIP_TRIANGLE,10000,10714.5,9009.0,8959.5,10872.0,1912.5,9.0501,9.0,10,32.0181,32.0,0.076423


,file_day,product,trades,total_qty,mean_trade_price,min_trade_price,max_trade_price
0,2,MICROCHIP_CIRCLE,174,342,9195.563218,8588.0,10036.0
1,2,MICROCHIP_OVAL,174,342,9774.626437,9109.0,10411.0
2,2,MICROCHIP_RECTANGLE,174,342,9590.367816,9006.0,10413.0
3,2,MICROCHIP_SQUARE,174,342,11240.672414,9778.0,12473.0
4,2,MICROCHIP_TRIANGLE,174,342,10236.310345,9517.0,10814.0
5,3,MICROCHIP_CIRCLE,194,383,8876.628866,8334.0,9522.0
6,3,MICROCHIP_OVAL,194,383,8490.128866,7388.0,9585.0
7,3,MICROCHIP_RECTANGLE,194,383,8267.628866,7497.0,9315.0
8,3,MICROCHIP_SQUARE,194,383,14678.185567,12293.0,16336.0
9,3,MICROCHIP_TRIANGLE,194,383,10212.788660,8964.0,10815.0


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — SHARED VECTORISED HELPERS
# ════════════════════════════════════════════════════════════════════════════

def micro_make_q(coefs: dict) -> np.ndarray:
    q = np.zeros(len(MICRO_PRODUCTS), dtype=float)
    for p, c in coefs.items():
        q[micro_product_to_idx[p]] = float(c)
    return q


def micro_scale_q_to_position_limit(q):
    q = np.asarray(q, dtype=float)
    nz = np.abs(q[np.abs(q) > 1e-12])

    if len(nz) == 0:
        return q.copy()

    if np.allclose(q, np.round(q), atol=1e-9):
        q_int = np.round(q).astype(int)
        max_abs = np.max(np.abs(q_int[q_int != 0]))
        units = max(1, POS_LIMIT // max_abs)
        return q_int.astype(float) * units

    return q / np.max(np.abs(q)) * POS_LIMIT


def micro_rolling_z_np(spread, window):
    s = pd.Series(spread)
    min_periods = max(50, window // 5)
    mean = s.rolling(window, min_periods=min_periods).mean().to_numpy(float)
    std = s.rolling(window, min_periods=min_periods).std().to_numpy(float)
    z = (spread - mean) / std
    z[~np.isfinite(z)] = 0.0
    return z


def micro_fit_ols_arrays(y, x):
    valid = np.isfinite(y) & np.isfinite(x)
    yv = y[valid]
    xv = x[valid]

    if len(yv) < 200 or np.var(xv) == 0:
        return 0.0, 1.0

    beta = np.cov(yv, xv, ddof=0)[0, 1] / np.var(xv)
    alpha = np.mean(yv) - beta * np.mean(xv)
    return alpha, beta


def micro_vectorised_exec_arrays(X, B, A, q_trade, h):
    assert h > 0
    assert X.shape == B.shape == A.shape
    assert X.shape[0] > h

    X0 = X[:-h]
    X1 = X[h:]
    B0 = B[:-h]
    A0 = A[:-h]
    B1 = B[h:]
    A1 = A[h:]

    q = q_trade.reshape(1, -1)

    entry_px = np.where(q > 0, A0, B0)
    exit_px = np.where(q > 0, B1, A1)

    exec_long = ((exit_px - entry_px) * q).sum(axis=1)
    mid_long = ((X1 - X0) * q).sum(axis=1)
    cost_long = mid_long - exec_long

    if np.nanmin(cost_long) < -1e-7:
        raise AssertionError(f"Cost identity failed: min cost={np.nanmin(cost_long)}")

    return exec_long, mid_long, cost_long


def micro_score_event_signal(z, exec_long, mid_long, cost_long, threshold, mode):
    active = np.abs(z) >= threshold

    if not np.any(active):
        return {
            "event_count": 0,
            "win_events": 0,
            "total_exec_pnl": 0.0,
            "total_mid_pnl": 0.0,
            "total_cost": 0.0,
            "avg_exec_pnl": np.nan,
            "median_exec_pnl": np.nan,
            "hit_rate": np.nan,
            "avg_cost": np.nan,
            "avg_abs_z": np.nan,
        }

    z_active = z[active]
    exec_active_long = exec_long[active]
    mid_active_long = mid_long[active]
    cost_active_long = cost_long[active]

    assert np.all(cost_active_long >= -1e-7)

    if mode == "meanrev":
        side = -np.sign(z_active)
    elif mode == "breakout":
        side = np.sign(z_active)
    elif mode == "follow":
        side = np.sign(z_active)
    elif mode == "inverse":
        side = -np.sign(z_active)
    else:
        raise ValueError(mode)

    exec_pnl = side * exec_active_long
    mid_pnl = side * mid_active_long

    identity_err = np.nanmax(np.abs(np.abs(mid_pnl - exec_pnl) - cost_active_long))
    assert identity_err < 1e-6, f"Cost identity failed: {identity_err}"

    event_count = len(exec_pnl)
    win_events = int((exec_pnl > 0).sum())

    return {
        "event_count": int(event_count),
        "win_events": win_events,
        "total_exec_pnl": float(np.nansum(exec_pnl)),
        "total_mid_pnl": float(np.nansum(mid_pnl)),
        "total_cost": float(np.nansum(cost_active_long)),
        "avg_exec_pnl": float(np.nanmean(exec_pnl)),
        "median_exec_pnl": float(np.nanmedian(exec_pnl)),
        "hit_rate": float(win_events / event_count),
        "avg_cost": float(np.nanmean(cost_active_long)),
        "avg_abs_z": float(np.nanmean(np.abs(z_active))),
    }


def micro_summarise_day_results(day_df, group_cols):
    summary = (
        day_df
        .groupby(group_cols)
        .agg(
            rows=("day", "count"),
            unique_days=("day", "nunique"),
            active_days=("event_count", lambda x: int((x > 0).sum())),
            event_count=("event_count", "sum"),
            win_events=("win_events", "sum"),
            total_exec_pnl=("total_exec_pnl", "sum"),
            total_mid_pnl=("total_mid_pnl", "sum"),
            total_cost=("total_cost", "sum"),
            day_avg_exec_pnl_mean=("avg_exec_pnl", "mean"),
            day_avg_exec_pnl_median=("avg_exec_pnl", "median"),
            worst_day_avg_exec_pnl=("avg_exec_pnl", "min"),
            best_day_avg_exec_pnl=("avg_exec_pnl", "max"),
            positive_avg_edge_days=("avg_exec_pnl", lambda x: int((x > 0).sum())),
            mean_hit_rate=("hit_rate", "mean"),
            min_hit_rate=("hit_rate", "min"),
            mean_avg_cost=("avg_cost", "mean"),
            mean_abs_z=("avg_abs_z", "mean"),
            worst_day_total_exec_pnl=("total_exec_pnl", "min"),
            best_day_total_exec_pnl=("total_exec_pnl", "max"),
            positive_total_pnl_days=("total_exec_pnl", lambda x: int((x > 0).sum())),
        )
        .reset_index()
    )

    summary["overall_hit_rate"] = summary["win_events"] / summary["event_count"].replace(0, np.nan)

    summary["signal_quality_score"] = (
        summary["positive_avg_edge_days"] * 1000
        + summary["unique_days"] * 100
        + summary["day_avg_exec_pnl_mean"].fillna(0)
        + (summary["mean_hit_rate"].fillna(0.5) - 0.5) * 1000
    )

    summary = summary.sort_values(
        ["positive_avg_edge_days", "unique_days", "day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
        ascending=[False, False, False, False, False],
    )

    return summary

In [5]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 — FAST BASKET / PAIR / SUBBASKET SCAN
# Tests standalone, pair, OLS pair, subbasket, and integer-combo z-score signals.
# ════════════════════════════════════════════════════════════════════════════

cell_start = time.perf_counter()

def micro_add_candidate(candidates, name, family, q_signal_by_day, note=""):
    q_trade_by_day = {}
    for day, q in q_signal_by_day.items():
        q_trade_by_day[day] = micro_scale_q_to_position_limit(q)

    candidates.append({
        "candidate": name,
        "family": family,
        "q_signal_by_day": q_signal_by_day,
        "q_trade_by_day": q_trade_by_day,
        "note": note,
    })


MICRO_CANDIDATES = []

# A. Standalone products.
for p in MICRO_PRODUCTS:
    q = micro_make_q({p: 1})
    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"single::{p}",
        family="single",
        q_signal_by_day={day: q for day in DAYS},
        note="single product z-score",
    )

# B. Simple pair differences.
for a, b in combinations(MICRO_PRODUCTS, 2):
    q = micro_make_q({a: 1, b: -1})
    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"pair_simple::{a}-{b}",
        family="pair_simple",
        q_signal_by_day={day: q for day in DAYS},
        note="simple pair difference",
    )

# C. Pair sums, directional sub-basket diagnostics.
for a, b in combinations(MICRO_PRODUCTS, 2):
    q = micro_make_q({a: 1, b: 1})
    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"pair_sum::{a}+{b}",
        family="pair_sum",
        q_signal_by_day={day: q for day in DAYS},
        note="positive pair basket, not market neutral",
    )

# D. Shape-family subbaskets.
SUBBASKETS = {
    "curved_sum": {"MICROCHIP_CIRCLE": 1, "MICROCHIP_OVAL": 1},
    "angular_sum": {"MICROCHIP_SQUARE": 1, "MICROCHIP_RECTANGLE": 1, "MICROCHIP_TRIANGLE": 1},
    "quadrilateral_sum": {"MICROCHIP_SQUARE": 1, "MICROCHIP_RECTANGLE": 1},
    "all_sum": {
        "MICROCHIP_CIRCLE": 1,
        "MICROCHIP_OVAL": 1,
        "MICROCHIP_SQUARE": 1,
        "MICROCHIP_RECTANGLE": 1,
        "MICROCHIP_TRIANGLE": 1,
    },
    "curved_minus_angular_balanced": {
        "MICROCHIP_CIRCLE": 1,
        "MICROCHIP_OVAL": 1,
        "MICROCHIP_SQUARE": -2/3,
        "MICROCHIP_RECTANGLE": -2/3,
        "MICROCHIP_TRIANGLE": -2/3,
    },
    "curved_minus_quadrilateral": {
        "MICROCHIP_CIRCLE": 1,
        "MICROCHIP_OVAL": 1,
        "MICROCHIP_SQUARE": -1,
        "MICROCHIP_RECTANGLE": -1,
    },
    "rectangle_triangle_sum": {
        "MICROCHIP_RECTANGLE": 1,
        "MICROCHIP_TRIANGLE": 1,
    },
    "oval_triangle_sum": {
        "MICROCHIP_OVAL": 1,
        "MICROCHIP_TRIANGLE": 1,
    },
}

for name, coefs in SUBBASKETS.items():
    q = micro_make_q(coefs)
    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"subbasket::{name}",
        family="subbasket",
        q_signal_by_day={day: q for day in DAYS},
        note=str(coefs),
    )

# E. Global OLS pair residuals.
for a, b in combinations(MICRO_PRODUCTS, 2):
    y = micro_mid[a].to_numpy(float)
    x = micro_mid[b].to_numpy(float)

    alpha, beta = micro_fit_ols_arrays(y, x)
    q = micro_make_q({a: 1, b: -beta})

    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"pair_global_ols::{a}~{b}",
        family="pair_global_ols",
        q_signal_by_day={day: q for day in DAYS},
        note=f"global beta={beta:.6f}",
    )

# F. Day-specific OLS pair residuals, diagnostic only.
for a, b in combinations(MICRO_PRODUCTS, 2):
    q_by_day = {}

    for day in DAYS:
        X = MICRO_FAST_DAY_BASE[day]["X"]
        ia = micro_product_to_idx[a]
        ib = micro_product_to_idx[b]
        alpha, beta = micro_fit_ols_arrays(X[:, ia], X[:, ib])
        q_by_day[day] = micro_make_q({a: 1, b: -beta})

    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"pair_day_ols::{a}~{b}",
        family="pair_day_ols",
        q_signal_by_day=q_by_day,
        note="day-specific beta; diagnostic/in-sample",
    )

# G. Integer combos, broad but limited.
def micro_normalise_coefs(coefs):
    coefs = list(coefs)

    if all(c == 0 for c in coefs):
        return None

    if sum(c != 0 for c in coefs) < 2:
        return None

    gcd = 0
    for c in coefs:
        gcd = math.gcd(gcd, abs(c))

    if gcd > 1:
        coefs = [c // gcd for c in coefs]

    first_nonzero = next(c for c in coefs if c != 0)
    if first_nonzero < 0:
        coefs = [-c for c in coefs]

    return tuple(coefs)


coef_values = [-2, -1, 0, 1, 2]
seen = set()
added_integer = 0

for raw in iter_product(coef_values, repeat=len(MICRO_PRODUCTS)):
    coefs = micro_normalise_coefs(raw)
    if coefs is None or coefs in seen:
        continue
    seen.add(coefs)

    gross_abs = sum(abs(c) for c in coefs)
    max_abs = max(abs(c) for c in coefs)

    if gross_abs > 6:
        continue

    q = np.array(coefs, dtype=float)

    micro_add_candidate(
        MICRO_CANDIDATES,
        name=f"integer_combo::{list(coefs)}",
        family="integer_combo",
        q_signal_by_day={day: q for day in DAYS},
        note=f"gross_abs={gross_abs}, max_abs={max_abs}",
    )
    added_integer += 1

print("Integer combos added:", added_integer)
print("Total MICRO_CANDIDATES:", len(MICRO_CANDIDATES))

# Run vectorised scan.
basket_rows = []

for ci, cand in enumerate(MICRO_CANDIDATES, start=1):
    if ci == 1 or ci % 10 == 0:
        print(f"[{time.perf_counter() - cell_start:7.2f}s] candidate {ci}/{len(MICRO_CANDIDATES)}: {cand['candidate']}")

    for day in DAYS:
        base = MICRO_FAST_DAY_BASE[day]
        X = base["X"]
        B = base["B"]
        A = base["A"]

        q_signal = np.asarray(cand["q_signal_by_day"][day], dtype=float)
        q_trade = np.asarray(cand["q_trade_by_day"][day], dtype=float)

        spread = X @ q_signal

        for window in MICRO_WINDOWS:
            z_full = micro_rolling_z_np(spread, window)

            for h in MICRO_HORIZONS:
                if X.shape[0] <= h + window:
                    continue

                exec_long, mid_long, cost_long = micro_vectorised_exec_arrays(X, B, A, q_trade, h)
                z = z_full[:-h]

                assert len(z) == len(exec_long)
                assert len(exec_long) == X.shape[0] - h

                for threshold in MICRO_THRESHOLDS:
                    for mode in MICRO_MODES:
                        stats = micro_score_event_signal(
                            z=z,
                            exec_long=exec_long,
                            mid_long=mid_long,
                            cost_long=cost_long,
                            threshold=threshold,
                            mode=mode,
                        )

                        basket_rows.append({
                            "candidate": cand["candidate"],
                            "family": cand["family"],
                            "note": cand["note"],
                            "day": day,
                            "window": window,
                            "horizon": h,
                            "threshold": threshold,
                            "mode": mode,
                            "q_signal": str(np.round(q_signal, 6).tolist()),
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                            **stats,
                        })

MICRO_FAST_DAY_RESULTS = pd.DataFrame(basket_rows)

# Dedupe just in case identical integer combos slipped in.
dedupe_cols = [
    "candidate", "family", "note", "day", "window", "horizon", "threshold", "mode", "q_signal", "q_trade"
]
MICRO_FAST_DAY_RESULTS = MICRO_FAST_DAY_RESULTS.drop_duplicates(subset=dedupe_cols).copy()

MICRO_FAST_BASE_SUMMARY = micro_summarise_day_results(
    MICRO_FAST_DAY_RESULTS,
    group_cols=["candidate", "family", "note", "window", "horizon", "threshold", "mode", "q_signal", "q_trade"],
)

MICRO_FAST_TRADABLE = MICRO_FAST_BASE_SUMMARY[
    ~MICRO_FAST_BASE_SUMMARY["family"].eq("pair_day_ols")
].copy()

MICRO_FAST_ROBUST = MICRO_FAST_TRADABLE[
    (MICRO_FAST_TRADABLE["unique_days"] == 3)
    & (MICRO_FAST_TRADABLE["positive_avg_edge_days"] == 3)
    & (MICRO_FAST_TRADABLE["event_count"] >= 300)
    & (MICRO_FAST_TRADABLE["day_avg_exec_pnl_mean"] > 0)
].copy()

MICRO_FAST_ROBUST = MICRO_FAST_ROBUST.sort_values(
    ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
    ascending=[False, False, False],
)

MICRO_FAST_BEST_PER_BASKET = (
    MICRO_FAST_TRADABLE
    .groupby("candidate", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

print("\nTop MICRO_FAST_ROBUST:")
display(MICRO_FAST_ROBUST.head(80))

print("\nTop MICRO_FAST_TRADABLE:")
display(MICRO_FAST_TRADABLE.head(80))

print("\nMICRO_FAST_BEST_PER_BASKET:")
display(MICRO_FAST_BEST_PER_BASKET.head(80))

MICRO_FAST_DAY_RESULTS.to_csv(MICRO_OUT / "MICRO_FAST_DAY_RESULTS.csv", index=False)
MICRO_FAST_BASE_SUMMARY.to_csv(MICRO_OUT / "MICRO_FAST_BASE_SUMMARY.csv", index=False)
MICRO_FAST_TRADABLE.to_csv(MICRO_OUT / "MICRO_FAST_TRADABLE.csv", index=False)
MICRO_FAST_ROBUST.to_csv(MICRO_OUT / "MICRO_FAST_ROBUST.csv", index=False)
MICRO_FAST_BEST_PER_BASKET.to_csv(MICRO_OUT / "MICRO_FAST_BEST_PER_BASKET.csv", index=False)

print(f"Runtime Cell 5: {time.perf_counter() - cell_start:.2f}s")

Integer combos added: 876
Total MICRO_CANDIDATES: 929
[   0.06s] candidate 1/929: single::MICROCHIP_CIRCLE
[   0.95s] candidate 10/929: pair_simple::MICROCHIP_OVAL-MICROCHIP_SQUARE
[   1.68s] candidate 20/929: pair_sum::MICROCHIP_OVAL+MICROCHIP_SQUARE
[   2.39s] candidate 30/929: subbasket::curved_minus_angular_balanced
[   3.08s] candidate 40/929: pair_global_ols::MICROCHIP_OVAL~MICROCHIP_TRIANGLE
[   3.77s] candidate 50/929: pair_day_ols::MICROCHIP_OVAL~MICROCHIP_TRIANGLE
[   4.46s] candidate 60/929: integer_combo::[1, 1, 1, -1, 1]
[   5.15s] candidate 70/929: integer_combo::[1, 1, 0, 1, -1]
[   5.85s] candidate 80/929: integer_combo::[2, 2, 0, -1, 0]
[   6.58s] candidate 90/929: integer_combo::[1, 1, -1, 1, 1]
[   7.50s] candidate 100/929: integer_combo::[2, 1, 2, 0, 1]
[   8.19s] candidate 110/929: integer_combo::[2, 1, 1, 0, 0]
[   8.93s] candidate 120/929: integer_combo::[2, 1, 0, 1, 2]
[   9.74s] candidate 130/929: integer_combo::[2, 1, 0, -1, 2]
[  10.49s] candidate 140/929: in

,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
15359,"integer_combo::[0, 1, 0, -1, 1]",integer_combo,"gross_abs=3, max_abs=1",2500,1000,2.5,meanrev,"[0.0, 1.0, 0.0, -1.0, 1.0]","[0.0, 10.0, 0.0, -10.0, 10.0]",3,...,3,0.802992,0.433628,243.261924,2.777966,484710.0,6219000.0,3,0.930481,12501.341927
109599,"integer_combo::[2, -1, 0, 1, -2]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.5,meanrev,"[2.0, -1.0, 0.0, 1.0, -2.0]","[10.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.890260,0.771429,243.906099,2.784770,585775.0,3979835.0,3,0.934540,12587.323911
45919,"integer_combo::[1, -1, 0, 1, -2]",integer_combo,"gross_abs=5, max_abs=2",2500,1000,2.5,meanrev,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.952087,0.867347,203.673155,2.789253,665570.0,2856900.0,3,0.975968,12351.368626
49439,"integer_combo::[1, -1, 1, 1, -1]",integer_combo,"gross_abs=5, max_abs=1",2500,1000,2.5,meanrev,"[1.0, -1.0, 1.0, 1.0, -1.0]","[10.0, -10.0, 10.0, 10.0, -10.0]",3,...,3,0.816985,0.522639,442.705026,2.809839,308830.0,5784850.0,3,0.755090,11557.672750
80959,"integer_combo::[1, 1, -1, -1, 1]",integer_combo,"gross_abs=5, max_abs=1",2500,1000,2.5,meanrev,"[1.0, 1.0, -1.0, -1.0, 1.0]","[10.0, 10.0, -10.0, -10.0, 10.0]",3,...,3,0.939134,0.911175,442.153308,2.818409,4982280.0,5654210.0,3,0.937623,11550.934408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29599,"integer_combo::[0, 2, -2, -1, 0]",integer_combo,"gross_abs=5, max_abs=2",2500,1000,2.5,meanrev,"[0.0, 2.0, -2.0, -1.0, 0.0]","[0.0, 10.0, -10.0, -5.0, 0.0]",3,...,3,0.916757,0.769394,233.365735,2.919434,1722960.0,3811725.0,3,0.882719,9205.808176
75517,"integer_combo::[1, 0, 1, 1, -1]",integer_combo,"gross_abs=4, max_abs=1",2500,1000,2.0,meanrev,"[1.0, 0.0, 1.0, 1.0, -1.0]","[10.0, 0.0, 10.0, 10.0, -10.0]",3,...,3,0.748316,0.587811,368.395586,2.371368,3047050.0,5948640.0,3,0.694157,9019.739913
52159,"integer_combo::[1, -1, 2, 1, -1]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.5,meanrev,"[1.0, -1.0, 2.0, 1.0, -1.0]","[5.0, -5.0, 10.0, 5.0, -5.0]",3,...,3,0.860388,0.677852,281.286768,2.929898,980560.0,4092635.0,3,0.868583,9130.855809
75997,"integer_combo::[1, 0, 1, 1, 1]",integer_combo,"gross_abs=4, max_abs=1",2500,1000,2.0,meanrev,"[1.0, 0.0, 1.0, 1.0, 1.0]","[10.0, 0.0, 10.0, 10.0, 10.0]",3,...,3,0.749249,0.720175,367.459488,2.480160,2657020.0,7191540.0,3,0.747547,9019.150982



Top MICRO_FAST_TRADABLE:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
15359,"integer_combo::[0, 1, 0, -1, 1]",integer_combo,"gross_abs=3, max_abs=1",2500,1000,2.5,meanrev,"[0.0, 1.0, 0.0, -1.0, 1.0]","[0.0, 10.0, 0.0, -10.0, 10.0]",3,...,3,0.802992,0.433628,243.261924,2.777966,484710.0,6219000.0,3,0.930481,12501.341927
109599,"integer_combo::[2, -1, 0, 1, -2]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.5,meanrev,"[2.0, -1.0, 0.0, 1.0, -2.0]","[10.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.890260,0.771429,243.906099,2.784770,585775.0,3979835.0,3,0.934540,12587.323911
45919,"integer_combo::[1, -1, 0, 1, -2]",integer_combo,"gross_abs=5, max_abs=2",2500,1000,2.5,meanrev,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.952087,0.867347,203.673155,2.789253,665570.0,2856900.0,3,0.975968,12351.368626
49439,"integer_combo::[1, -1, 1, 1, -1]",integer_combo,"gross_abs=5, max_abs=1",2500,1000,2.5,meanrev,"[1.0, -1.0, 1.0, 1.0, -1.0]","[10.0, -10.0, 10.0, 10.0, -10.0]",3,...,3,0.816985,0.522639,442.705026,2.809839,308830.0,5784850.0,3,0.755090,11557.672750
80959,"integer_combo::[1, 1, -1, -1, 1]",integer_combo,"gross_abs=5, max_abs=1",2500,1000,2.5,meanrev,"[1.0, 1.0, -1.0, -1.0, 1.0]","[10.0, 10.0, -10.0, -10.0, 10.0]",3,...,3,0.939134,0.911175,442.153308,2.818409,4982280.0,5654210.0,3,0.937623,11550.934408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29599,"integer_combo::[0, 2, -2, -1, 0]",integer_combo,"gross_abs=5, max_abs=2",2500,1000,2.5,meanrev,"[0.0, 2.0, -2.0, -1.0, 0.0]","[0.0, 10.0, -10.0, -5.0, 0.0]",3,...,3,0.916757,0.769394,233.365735,2.919434,1722960.0,3811725.0,3,0.882719,9205.808176
75517,"integer_combo::[1, 0, 1, 1, -1]",integer_combo,"gross_abs=4, max_abs=1",2500,1000,2.0,meanrev,"[1.0, 0.0, 1.0, 1.0, -1.0]","[10.0, 0.0, 10.0, 10.0, -10.0]",3,...,3,0.748316,0.587811,368.395586,2.371368,3047050.0,5948640.0,3,0.694157,9019.739913
52159,"integer_combo::[1, -1, 2, 1, -1]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.5,meanrev,"[1.0, -1.0, 2.0, 1.0, -1.0]","[5.0, -5.0, 10.0, 5.0, -5.0]",3,...,3,0.860388,0.677852,281.286768,2.929898,980560.0,4092635.0,3,0.868583,9130.855809
75997,"integer_combo::[1, 0, 1, 1, 1]",integer_combo,"gross_abs=4, max_abs=1",2500,1000,2.0,meanrev,"[1.0, 0.0, 1.0, 1.0, 1.0]","[10.0, 0.0, 10.0, 10.0, 10.0]",3,...,3,0.749249,0.720175,367.459488,2.480160,2657020.0,7191540.0,3,0.747547,9019.150982



MICRO_FAST_BEST_PER_BASKET:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
0,"integer_combo::[0, 1, 0, -1, 1]",integer_combo,"gross_abs=3, max_abs=1",2500,1000,2.5,meanrev,"[0.0, 1.0, 0.0, -1.0, 1.0]","[0.0, 10.0, 0.0, -10.0, 10.0]",3,...,3,0.802992,0.433628,243.261924,2.777966,484710.0,6219000.0,3,0.930481,12501.341927
1,"integer_combo::[2, -1, 0, 1, -2]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.5,meanrev,"[2.0, -1.0, 0.0, 1.0, -2.0]","[10.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.890260,0.771429,243.906099,2.784770,585775.0,3979835.0,3,0.934540,12587.323911
2,"integer_combo::[1, -1, 0, 1, -2]",integer_combo,"gross_abs=5, max_abs=2",2500,1000,2.5,meanrev,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.952087,0.867347,203.673155,2.789253,665570.0,2856900.0,3,0.975968,12351.368626
3,"integer_combo::[1, -1, 1, 1, -1]",integer_combo,"gross_abs=5, max_abs=1",2500,1000,2.5,meanrev,"[1.0, -1.0, 1.0, 1.0, -1.0]","[10.0, -10.0, 10.0, 10.0, -10.0]",3,...,3,0.816985,0.522639,442.705026,2.809839,308830.0,5784850.0,3,0.755090,11557.672750
4,"integer_combo::[1, 1, -1, -1, 1]",integer_combo,"gross_abs=5, max_abs=1",2500,1000,2.5,meanrev,"[1.0, 1.0, -1.0, -1.0, 1.0]","[10.0, 10.0, -10.0, -10.0, 10.0]",3,...,3,0.939134,0.911175,442.153308,2.818409,4982280.0,5654210.0,3,0.937623,11550.934408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,"integer_combo::[1, 0, 2, 2, 1]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.0,meanrev,"[1.0, 0.0, 2.0, 2.0, 1.0]","[5.0, 0.0, 10.0, 10.0, 5.0]",3,...,3,0.760083,0.662582,283.265610,2.528941,2090140.0,8597520.0,3,0.753717,8770.357406
76,"integer_combo::[0, 0, 1, 2, -2]",integer_combo,"gross_abs=5, max_abs=2",2500,1000,2.0,meanrev,"[0.0, 0.0, 1.0, 2.0, -2.0]","[0.0, 0.0, 5.0, 10.0, -10.0]",3,...,3,0.858979,0.646166,225.336297,2.411718,1914170.0,9034840.0,3,0.843703,8865.580999
77,"integer_combo::[1, 1, 1, 1, 2]",integer_combo,"gross_abs=6, max_abs=2",2500,1000,2.5,meanrev,"[1.0, 1.0, 1.0, 1.0, 2.0]","[5.0, 5.0, 5.0, 5.0, 10.0]",3,...,3,0.941230,0.939394,264.941518,2.972389,1159745.0,2112900.0,3,0.941234,8944.514947
78,"integer_combo::[1, 0, 0, 1, -2]",integer_combo,"gross_abs=4, max_abs=2",2500,1000,2.5,meanrev,"[1.0, 0.0, 0.0, 1.0, -2.0]","[5.0, 0.0, 0.0, 5.0, -10.0]",3,...,3,0.830236,0.670940,166.538954,2.879413,695505.0,3441500.0,3,0.819512,8832.877856


Runtime Cell 5: 79.96s


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — MICROCHIP LEAD-LAG SCAN
# Tests A past move predicting B future move with executable bid/ask PnL.
# ════════════════════════════════════════════════════════════════════════════

cell_start = time.perf_counter()

leadlag_rows = []

for leader in MICRO_PRODUCTS:
    for follower in MICRO_PRODUCTS:
        if leader == follower:
            continue

        q_follower = micro_make_q({follower: 1})
        q_trade = micro_scale_q_to_position_limit(q_follower)

        for past_h in MICRO_PAST_HORIZONS:
            for future_h in MICRO_LEADLAG_HORIZONS:
                for threshold in [1.0, 1.5, 2.0, 2.5]:
                    for mode in ["follow", "inverse"]:
                        for day in DAYS:
                            base = MICRO_FAST_DAY_BASE[day]
                            X = base["X"]
                            B = base["B"]
                            A = base["A"]

                            il = micro_product_to_idx[leader]

                            leader_move = np.full(X.shape[0], np.nan)
                            leader_move[past_h:] = X[past_h:, il] - X[:-past_h, il]

                            z_full = micro_rolling_z_np(leader_move, window=500)

                            if X.shape[0] <= future_h + past_h:
                                continue

                            exec_long, mid_long, cost_long = micro_vectorised_exec_arrays(X, B, A, q_trade, future_h)
                            z = z_full[:-future_h]

                            assert len(z) == len(exec_long)

                            stats = micro_score_event_signal(
                                z=z,
                                exec_long=exec_long,
                                mid_long=mid_long,
                                cost_long=cost_long,
                                threshold=threshold,
                                mode=mode,
                            )

                            leadlag_rows.append({
                                "leader": leader,
                                "follower": follower,
                                "day": day,
                                "past_h": past_h,
                                "future_h": future_h,
                                "threshold": threshold,
                                "mode": mode,
                                "q_trade": str(np.round(q_trade, 6).tolist()),
                                **stats,
                            })

MICRO_LEADLAG_DAY_RESULTS = pd.DataFrame(leadlag_rows)

MICRO_LEADLAG_SUMMARY = micro_summarise_day_results(
    MICRO_LEADLAG_DAY_RESULTS,
    group_cols=["leader", "follower", "past_h", "future_h", "threshold", "mode", "q_trade"],
)

MICRO_LEADLAG_ROBUST = MICRO_LEADLAG_SUMMARY[
    (MICRO_LEADLAG_SUMMARY["unique_days"] == 3)
    & (MICRO_LEADLAG_SUMMARY["positive_avg_edge_days"] == 3)
    & (MICRO_LEADLAG_SUMMARY["event_count"] >= 200)
    & (MICRO_LEADLAG_SUMMARY["day_avg_exec_pnl_mean"] > 0)
].copy()

MICRO_LEADLAG_ROBUST = MICRO_LEADLAG_ROBUST.sort_values(
    ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
    ascending=[False, False, False],
)

print("\nTop MICRO_LEADLAG_ROBUST:")
display(MICRO_LEADLAG_ROBUST.head(80))

print("\nTop MICRO_LEADLAG_SUMMARY:")
display(MICRO_LEADLAG_SUMMARY.head(80))

MICRO_LEADLAG_DAY_RESULTS.to_csv(MICRO_OUT / "MICRO_LEADLAG_DAY_RESULTS.csv", index=False)
MICRO_LEADLAG_SUMMARY.to_csv(MICRO_OUT / "MICRO_LEADLAG_SUMMARY.csv", index=False)
MICRO_LEADLAG_ROBUST.to_csv(MICRO_OUT / "MICRO_LEADLAG_ROBUST.csv", index=False)

print(f"Runtime Cell 6: {time.perf_counter() - cell_start:.2f}s")


Top MICRO_LEADLAG_ROBUST:


,leader,follower,past_h,future_h,threshold,mode,q_trade,rows,unique_days,active_days,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
719,MICROCHIP_CIRCLE,MICROCHIP_SQUARE,50,500,2.5,inverse,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,...,3,0.644439,0.440000,118.227611,2.841393,56440.0,490980.0,3,0.656716,4832.956578
2300,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,500,500,2.0,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,...,3,0.669414,0.640292,86.015360,2.385483,777480.0,3406240.0,3,0.667756,4852.288615
815,MICROCHIP_CIRCLE,MICROCHIP_SQUARE,250,500,2.5,inverse,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,...,3,0.627386,0.409341,118.403315,2.862896,288880.0,671220.0,3,0.627561,4762.725055
2158,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,50,500,2.5,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,...,3,0.678359,0.544444,87.027612,2.820921,182260.0,311060.0,3,0.680431,4745.126478
813,MICROCHIP_CIRCLE,MICROCHIP_SQUARE,250,500,2.0,inverse,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,...,3,0.650146,0.595969,118.207823,2.427318,277470.0,1752200.0,3,0.648257,4648.367011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2102,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,25,250,2.5,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,...,3,0.563427,0.466165,86.973875,2.802783,26160.0,155940.0,3,0.569106,3832.746145
3695,MICROCHIP_SQUARE,MICROCHIP_CIRCLE,250,500,2.5,inverse,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,3,...,3,0.541770,0.484029,82.060820,2.826747,112360.0,241260.0,3,0.535986,3805.679666
430,MICROCHIP_CIRCLE,MICROCHIP_RECTANGLE,50,500,2.5,follow,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,...,3,0.600870,0.546610,79.417837,2.841393,11310.0,181040.0,3,0.597015,3862.167587
562,MICROCHIP_CIRCLE,MICROCHIP_RECTANGLE,500,250,1.5,follow,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,...,3,0.602660,0.583988,78.643794,2.091509,1076160.0,1597520.0,3,0.601867,3861.644982



Top MICRO_LEADLAG_SUMMARY:


,leader,follower,past_h,future_h,threshold,mode,q_trade,rows,unique_days,active_days,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
719,MICROCHIP_CIRCLE,MICROCHIP_SQUARE,50,500,2.5,inverse,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,...,3,0.644439,0.440000,118.227611,2.841393,56440.0,490980.0,3,0.656716,4832.956578
2300,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,500,500,2.0,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,...,3,0.669414,0.640292,86.015360,2.385483,777480.0,3406240.0,3,0.667756,4852.288615
815,MICROCHIP_CIRCLE,MICROCHIP_SQUARE,250,500,2.5,inverse,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,...,3,0.627386,0.409341,118.403315,2.862896,288880.0,671220.0,3,0.627561,4762.725055
2158,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,50,500,2.5,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,...,3,0.678359,0.544444,87.027612,2.820921,182260.0,311060.0,3,0.680431,4745.126478
813,MICROCHIP_CIRCLE,MICROCHIP_SQUARE,250,500,2.0,inverse,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,...,3,0.650146,0.595969,118.207823,2.427318,277470.0,1752200.0,3,0.648257,4648.367011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2102,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,25,250,2.5,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,...,3,0.563427,0.466165,86.973875,2.802783,26160.0,155940.0,3,0.569106,3832.746145
3695,MICROCHIP_SQUARE,MICROCHIP_CIRCLE,250,500,2.5,inverse,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,3,...,3,0.541770,0.484029,82.060820,2.826747,112360.0,241260.0,3,0.535986,3805.679666
430,MICROCHIP_CIRCLE,MICROCHIP_RECTANGLE,50,500,2.5,follow,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,...,3,0.600870,0.546610,79.417837,2.841393,11310.0,181040.0,3,0.597015,3862.167587
562,MICROCHIP_CIRCLE,MICROCHIP_RECTANGLE,500,250,1.5,follow,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,...,3,0.602660,0.583988,78.643794,2.091509,1076160.0,1597520.0,3,0.601867,3861.644982


Runtime Cell 6: 11.35s


In [7]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — MICROCHIP OWN MOMENTUM / REVERSAL SCAN
# Tests own past move predicting own future move.
# ════════════════════════════════════════════════════════════════════════════

cell_start = time.perf_counter()

own_rows = []

for p in MICRO_PRODUCTS:
    q = micro_make_q({p: 1})
    q_trade = micro_scale_q_to_position_limit(q)
    ip = micro_product_to_idx[p]

    for past_h in MICRO_PAST_HORIZONS:
        for future_h in MICRO_LEADLAG_HORIZONS:
            for threshold in [1.0, 1.5, 2.0, 2.5]:
                for mode in ["follow", "inverse"]:
                    for day in DAYS:
                        base = MICRO_FAST_DAY_BASE[day]
                        X = base["X"]
                        B = base["B"]
                        A = base["A"]

                        past_move = np.full(X.shape[0], np.nan)
                        past_move[past_h:] = X[past_h:, ip] - X[:-past_h, ip]

                        z_full = micro_rolling_z_np(past_move, window=500)

                        if X.shape[0] <= future_h + past_h:
                            continue

                        exec_long, mid_long, cost_long = micro_vectorised_exec_arrays(X, B, A, q_trade, future_h)
                        z = z_full[:-future_h]

                        stats = micro_score_event_signal(
                            z=z,
                            exec_long=exec_long,
                            mid_long=mid_long,
                            cost_long=cost_long,
                            threshold=threshold,
                            mode=mode,
                        )

                        own_rows.append({
                            "product": p,
                            "day": day,
                            "past_h": past_h,
                            "future_h": future_h,
                            "threshold": threshold,
                            "mode": mode,
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                            **stats,
                        })

MICRO_OWN_DAY_RESULTS = pd.DataFrame(own_rows)

MICRO_OWN_SUMMARY = micro_summarise_day_results(
    MICRO_OWN_DAY_RESULTS,
    group_cols=["product", "past_h", "future_h", "threshold", "mode", "q_trade"],
)

MICRO_OWN_ROBUST = MICRO_OWN_SUMMARY[
    (MICRO_OWN_SUMMARY["unique_days"] == 3)
    & (MICRO_OWN_SUMMARY["positive_avg_edge_days"] == 3)
    & (MICRO_OWN_SUMMARY["event_count"] >= 200)
    & (MICRO_OWN_SUMMARY["day_avg_exec_pnl_mean"] > 0)
].copy()

MICRO_OWN_ROBUST = MICRO_OWN_ROBUST.sort_values(
    ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
    ascending=[False, False, False],
)

print("\nTop MICRO_OWN_ROBUST:")
display(MICRO_OWN_ROBUST.head(80))

print("\nTop MICRO_OWN_SUMMARY:")
display(MICRO_OWN_SUMMARY.head(80))

MICRO_OWN_DAY_RESULTS.to_csv(MICRO_OUT / "MICRO_OWN_DAY_RESULTS.csv", index=False)
MICRO_OWN_SUMMARY.to_csv(MICRO_OUT / "MICRO_OWN_SUMMARY.csv", index=False)
MICRO_OWN_ROBUST.to_csv(MICRO_OUT / "MICRO_OWN_ROBUST.csv", index=False)

print(f"Runtime Cell 7: {time.perf_counter() - cell_start:.2f}s")


Top MICRO_OWN_ROBUST:


,product,past_h,future_h,threshold,mode,q_trade,rows,unique_days,active_days,event_count,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
998,MICROCHIP_SQUARE,50,250,2.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,545,...,3,0.647611,0.550725,118.437182,2.891463,69760.0,328280.0,3,0.662385,4420.887752
863,MICROCHIP_RECTANGLE,500,500,2.5,inverse,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,1432,...,3,0.637172,0.494949,78.772292,2.885273,200540.0,641660.0,3,0.631285,4404.696354
861,MICROCHIP_RECTANGLE,500,500,2.0,inverse,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,3939,...,3,0.622368,0.486936,78.615938,2.463419,177210.0,2061620.0,3,0.617669,4340.837070
1046,MICROCHIP_SQUARE,100,250,2.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,989,...,3,0.595720,0.574405,117.555499,2.903025,174300.0,407380.0,3,0.593529,4221.999800
287,MICROCHIP_CIRCLE,500,500,2.5,inverse,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,3,1640,...,3,0.603283,0.500907,82.319851,2.876519,107060.0,469940.0,3,0.596951,4050.089325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,MICROCHIP_OVAL,25,250,1.5,follow,"[0.0, 10.0, 0.0, 0.0, 0.0]",3,3,3,4395,...,3,0.510728,0.478922,74.577683,1.963077,194420.0,363200.0,3,0.510808,3481.991355
185,MICROCHIP_CIRCLE,100,500,1.0,inverse,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,3,11024,...,3,0.535464,0.526861,82.725374,1.617386,308920.0,1012650.0,3,0.535559,3504.485823
1014,MICROCHIP_SQUARE,100,10,2.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,1011,...,3,0.605693,0.560000,117.747024,2.901270,34010.0,80870.0,3,0.603363,3572.395815
1359,MICROCHIP_TRIANGLE,250,25,2.5,inverse,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,1162,...,3,0.583811,0.461089,86.739000,2.832500,9130.0,99100.0,3,0.563683,3547.786414



Top MICRO_OWN_SUMMARY:


,product,past_h,future_h,threshold,mode,q_trade,rows,unique_days,active_days,event_count,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
998,MICROCHIP_SQUARE,50,250,2.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,545,...,3,0.647611,0.550725,118.437182,2.891463,69760.0,328280.0,3,0.662385,4420.887752
863,MICROCHIP_RECTANGLE,500,500,2.5,inverse,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,1432,...,3,0.637172,0.494949,78.772292,2.885273,200540.0,641660.0,3,0.631285,4404.696354
861,MICROCHIP_RECTANGLE,500,500,2.0,inverse,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,3,3939,...,3,0.622368,0.486936,78.615938,2.463419,177210.0,2061620.0,3,0.617669,4340.837070
1046,MICROCHIP_SQUARE,100,250,2.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,989,...,3,0.595720,0.574405,117.555499,2.903025,174300.0,407380.0,3,0.593529,4221.999800
287,MICROCHIP_CIRCLE,500,500,2.5,inverse,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,3,1640,...,3,0.603283,0.500907,82.319851,2.876519,107060.0,469940.0,3,0.596951,4050.089325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,MICROCHIP_OVAL,25,250,1.5,follow,"[0.0, 10.0, 0.0, 0.0, 0.0]",3,3,3,4395,...,3,0.510728,0.478922,74.577683,1.963077,194420.0,363200.0,3,0.510808,3481.991355
185,MICROCHIP_CIRCLE,100,500,1.0,inverse,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,3,11024,...,3,0.535464,0.526861,82.725374,1.617386,308920.0,1012650.0,3,0.535559,3504.485823
1014,MICROCHIP_SQUARE,100,10,2.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,3,1011,...,3,0.605693,0.560000,117.747024,2.901270,34010.0,80870.0,3,0.603363,3572.395815
1359,MICROCHIP_TRIANGLE,250,25,2.5,inverse,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,3,1162,...,3,0.583811,0.461089,86.739000,2.832500,9130.0,99100.0,3,0.563683,3547.786414


Runtime Cell 7: 2.85s


In [8]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — ORDER BOOK + TRADE FLOW SCAN
# Tests imbalance, microprice, and inferred aggressive trade flow.
# ════════════════════════════════════════════════════════════════════════════

cell_start = time.perf_counter()

# Infer trade side.
quote_cols = ["file_day", "timestamp", "product", "bid_price_1", "ask_price_1", "mid", "spread"]

micro_trade_q = micro_trades.merge(
    micro_prices[quote_cols],
    on=["file_day", "timestamp", "product"],
    how="left",
)

micro_trade_q["inferred_side"] = 0
micro_trade_q.loc[micro_trade_q["price"] >= micro_trade_q["ask_price_1"], "inferred_side"] = 1
micro_trade_q.loc[micro_trade_q["price"] <= micro_trade_q["bid_price_1"], "inferred_side"] = -1
micro_trade_q["signed_qty"] = micro_trade_q["inferred_side"] * micro_trade_q["quantity"]

# Build flow arrays once.
for day in DAYS:
    base = MICRO_FAST_DAY_BASE[day]
    timestamps = base["timestamps"]

    signed = np.zeros((len(timestamps), len(MICRO_PRODUCTS)), dtype=float)
    gross = np.zeros((len(timestamps), len(MICRO_PRODUCTS)), dtype=float)

    ts_index = {ts: i for i, ts in enumerate(timestamps)}

    day_trades = micro_trade_q[micro_trade_q["file_day"] == day]

    for p in MICRO_PRODUCTS:
        ip = micro_product_to_idx[p]

        g = (
            day_trades[day_trades["product"] == p]
            .groupby("timestamp")
            .agg(
                signed_qty=("signed_qty", "sum"),
                gross_qty=("quantity", "sum"),
            )
        )

        for ts, row in g.iterrows():
            if ts in ts_index:
                idx = ts_index[ts]
                signed[idx, ip] = row["signed_qty"]
                gross[idx, ip] = row["gross_qty"]

    base["trade_signed_qty"] = signed
    base["trade_gross_qty"] = gross

book_flow_rows = []

BOOK_FLOW_SIGNALS = ["imb_l1", "imb_total", "microprice_edge_norm"]

for signal_name in BOOK_FLOW_SIGNALS:
    for p in MICRO_PRODUCTS:
        ip = micro_product_to_idx[p]
        q = micro_make_q({p: 1})
        q_trade = micro_scale_q_to_position_limit(q)

        for future_h in [1, 2, 5, 10, 25, 50, 100]:
            for threshold in [1.0, 1.5, 2.0]:
                for mode in ["follow", "inverse"]:
                    for day in DAYS:
                        base = MICRO_FAST_DAY_BASE[day]
                        X = base["X"]
                        B = base["B"]
                        A = base["A"]

                        raw_signal = base[signal_name][:, ip]
                        z_full = micro_rolling_z_np(raw_signal, window=500)

                        if X.shape[0] <= future_h:
                            continue

                        exec_long, mid_long, cost_long = micro_vectorised_exec_arrays(X, B, A, q_trade, future_h)
                        z = z_full[:-future_h]

                        stats = micro_score_event_signal(
                            z=z,
                            exec_long=exec_long,
                            mid_long=mid_long,
                            cost_long=cost_long,
                            threshold=threshold,
                            mode=mode,
                        )

                        book_flow_rows.append({
                            "signal_family": "book",
                            "signal_name": signal_name,
                            "product": p,
                            "day": day,
                            "future_h": future_h,
                            "flow_window": np.nan,
                            "threshold": threshold,
                            "mode": mode,
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                            **stats,
                        })

# Trade flow imbalance.
for fw in MICRO_FLOW_WINDOWS:
    grid_w = max(1, fw // 100)

    for p in MICRO_PRODUCTS:
        ip = micro_product_to_idx[p]
        q = micro_make_q({p: 1})
        q_trade = micro_scale_q_to_position_limit(q)

        for future_h in [5, 10, 25, 50, 100, 250]:
            for threshold in [1.0, 1.5, 2.0]:
                for mode in ["follow", "inverse"]:
                    for day in DAYS:
                        base = MICRO_FAST_DAY_BASE[day]
                        X = base["X"]
                        B = base["B"]
                        A = base["A"]

                        signed_s = pd.Series(base["trade_signed_qty"][:, ip])
                        gross_s = pd.Series(base["trade_gross_qty"][:, ip])

                        signed_roll = signed_s.rolling(grid_w, min_periods=1).sum().to_numpy(float)
                        gross_roll = gross_s.rolling(grid_w, min_periods=1).sum().to_numpy(float)

                        flow_imb = signed_roll / np.where(gross_roll == 0, np.nan, gross_roll)
                        flow_imb[~np.isfinite(flow_imb)] = 0.0

                        z_full = micro_rolling_z_np(flow_imb, window=500)

                        if X.shape[0] <= future_h:
                            continue

                        exec_long, mid_long, cost_long = micro_vectorised_exec_arrays(X, B, A, q_trade, future_h)
                        z = z_full[:-future_h]

                        stats = micro_score_event_signal(
                            z=z,
                            exec_long=exec_long,
                            mid_long=mid_long,
                            cost_long=cost_long,
                            threshold=threshold,
                            mode=mode,
                        )

                        book_flow_rows.append({
                            "signal_family": "flow",
                            "signal_name": f"flow_imb_{fw}",
                            "product": p,
                            "day": day,
                            "future_h": future_h,
                            "flow_window": fw,
                            "threshold": threshold,
                            "mode": mode,
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                            **stats,
                        })

MICRO_BOOK_FLOW_DAY_RESULTS = pd.DataFrame(book_flow_rows)

MICRO_BOOK_FLOW_SUMMARY = micro_summarise_day_results(
    MICRO_BOOK_FLOW_DAY_RESULTS,
    group_cols=["signal_family", "signal_name", "product", "future_h", "flow_window", "threshold", "mode", "q_trade"],
)

MICRO_BOOK_FLOW_ROBUST = MICRO_BOOK_FLOW_SUMMARY[
    (MICRO_BOOK_FLOW_SUMMARY["unique_days"] == 3)
    & (MICRO_BOOK_FLOW_SUMMARY["positive_avg_edge_days"] == 3)
    & (MICRO_BOOK_FLOW_SUMMARY["event_count"] >= 100)
    & (MICRO_BOOK_FLOW_SUMMARY["day_avg_exec_pnl_mean"] > 0)
].copy()

MICRO_BOOK_FLOW_ROBUST = MICRO_BOOK_FLOW_ROBUST.sort_values(
    ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
    ascending=[False, False, False],
)

print("\nTop MICRO_BOOK_FLOW_ROBUST:")
display(MICRO_BOOK_FLOW_ROBUST.head(80))

print("\nTop MICRO_BOOK_FLOW_SUMMARY:")
display(MICRO_BOOK_FLOW_SUMMARY.head(80))

MICRO_BOOK_FLOW_DAY_RESULTS.to_csv(MICRO_OUT / "MICRO_BOOK_FLOW_DAY_RESULTS.csv", index=False)
MICRO_BOOK_FLOW_SUMMARY.to_csv(MICRO_OUT / "MICRO_BOOK_FLOW_SUMMARY.csv", index=False)
MICRO_BOOK_FLOW_ROBUST.to_csv(MICRO_OUT / "MICRO_BOOK_FLOW_ROBUST.csv", index=False)

print(f"Runtime Cell 8: {time.perf_counter() - cell_start:.2f}s")


Top MICRO_BOOK_FLOW_ROBUST:


,signal_family,signal_name,product,future_h,flow_window,threshold,mode,q_trade,rows,unique_days,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
862,flow,flow_imb_5000,MICROCHIP_SQUARE,250,5000.0,2.0,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,...,3,0.630099,0.575835,117.875929,2.272947,131670.0,225180.0,3,0.614152,4309.827980
322,flow,flow_imb_10000,MICROCHIP_SQUARE,250,10000.0,2.0,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,...,3,0.654556,0.597444,117.577239,2.421265,124340.0,471670.0,3,0.645137,4253.041803
358,flow,flow_imb_10000,MICROCHIP_TRIANGLE,250,10000.0,2.0,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.603914,0.512739,86.178496,2.421265,257280.0,312980.0,3,0.600304,4149.295855
898,flow,flow_imb_5000,MICROCHIP_TRIANGLE,250,5000.0,2.0,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.500027,0.454545,86.608395,2.272947,14930.0,168230.0,3,0.495327,3934.972608
314,flow,flow_imb_10000,MICROCHIP_SQUARE,100,10000.0,1.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,...,3,0.607773,0.509763,116.840829,1.901446,12920.0,1775920.0,3,0.613883,3862.063392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,flow,flow_imb_2500,MICROCHIP_TRIANGLE,5,2500.0,1.0,inverse,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.515415,0.498688,86.234724,1.700396,26050.0,88750.0,3,0.515574,3329.992025
364,flow,flow_imb_2500,MICROCHIP_CIRCLE,5,2500.0,2.0,follow,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,...,3,0.530973,0.497496,81.915778,2.219510,2280.0,13620.0,3,0.531306,3344.905037
434,flow,flow_imb_2500,MICROCHIP_RECTANGLE,5,2500.0,1.5,follow,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,...,3,0.509412,0.499198,78.628801,1.826486,14620.0,56400.0,3,0.509737,3322.137925
507,flow,flow_imb_2500,MICROCHIP_TRIANGLE,5,2500.0,1.5,inverse,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.507971,0.488772,86.390768,1.826486,3400.0,56950.0,3,0.507515,3320.379512



Top MICRO_BOOK_FLOW_SUMMARY:


,signal_family,signal_name,product,future_h,flow_window,threshold,mode,q_trade,rows,unique_days,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
862,flow,flow_imb_5000,MICROCHIP_SQUARE,250,5000.0,2.0,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,...,3,0.630099,0.575835,117.875929,2.272947,131670.0,225180.0,3,0.614152,4309.827980
322,flow,flow_imb_10000,MICROCHIP_SQUARE,250,10000.0,2.0,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,...,3,0.654556,0.597444,117.577239,2.421265,124340.0,471670.0,3,0.645137,4253.041803
358,flow,flow_imb_10000,MICROCHIP_TRIANGLE,250,10000.0,2.0,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.603914,0.512739,86.178496,2.421265,257280.0,312980.0,3,0.600304,4149.295855
898,flow,flow_imb_5000,MICROCHIP_TRIANGLE,250,5000.0,2.0,follow,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.500027,0.454545,86.608395,2.272947,14930.0,168230.0,3,0.495327,3934.972608
314,flow,flow_imb_10000,MICROCHIP_SQUARE,100,10000.0,1.5,follow,"[0.0, 0.0, 10.0, 0.0, 0.0]",3,3,...,3,0.607773,0.509763,116.840829,1.901446,12920.0,1775920.0,3,0.613883,3862.063392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,flow,flow_imb_2500,MICROCHIP_TRIANGLE,5,2500.0,1.0,inverse,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.515415,0.498688,86.234724,1.700396,26050.0,88750.0,3,0.515574,3329.992025
364,flow,flow_imb_2500,MICROCHIP_CIRCLE,5,2500.0,2.0,follow,"[10.0, 0.0, 0.0, 0.0, 0.0]",3,3,...,3,0.530973,0.497496,81.915778,2.219510,2280.0,13620.0,3,0.531306,3344.905037
434,flow,flow_imb_2500,MICROCHIP_RECTANGLE,5,2500.0,1.5,follow,"[0.0, 0.0, 0.0, 10.0, 0.0]",3,3,...,3,0.509412,0.499198,78.628801,1.826486,14620.0,56400.0,3,0.509737,3322.137925
507,flow,flow_imb_2500,MICROCHIP_TRIANGLE,5,2500.0,1.5,inverse,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,3,...,3,0.507971,0.488772,86.390768,1.826486,3400.0,56950.0,3,0.507515,3320.379512


Runtime Cell 8: 3.66s


In [9]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — MICROCHIPS PRIORITY REPORT EXPORT
# ════════════════════════════════════════════════════════════════════════════

MICRO_PRIORITY_TABLES = []

def add_priority(df, source_name, n=40):
    if df is None or len(df) == 0:
        return
    temp = df.head(n).copy()
    temp["source"] = source_name
    MICRO_PRIORITY_TABLES.append(temp)

add_priority(MICRO_FAST_ROBUST, "basket_pair_subbasket_scan", 60)
add_priority(MICRO_LEADLAG_ROBUST, "leadlag_scan", 60)
add_priority(MICRO_OWN_ROBUST, "own_momentum_reversal_scan", 60)
add_priority(MICRO_BOOK_FLOW_ROBUST, "book_flow_scan", 60)

if MICRO_PRIORITY_TABLES:
    MICRO_PRIORITY = pd.concat(MICRO_PRIORITY_TABLES, ignore_index=True, sort=False)
else:
    MICRO_PRIORITY = pd.DataFrame()

# Generic ranking columns may differ by source, so sort on common useful fields.
sort_cols = [c for c in ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"] if c in MICRO_PRIORITY.columns]
if sort_cols:
    MICRO_PRIORITY = MICRO_PRIORITY.sort_values(sort_cols, ascending=[False] * len(sort_cols))

display(MICRO_PRIORITY.head(120))

MICRO_PRIORITY.to_csv(MICRO_OUT / "MICRO_PRIORITY.csv", index=False)

report = []
report.append("# Microchips Stage 1 Discovery Report\n")
report.append(f"Products: {MICRO_PRODUCTS}\n")
report.append(f"Runtime so far: {time.perf_counter() - MICRO_START:.2f}s\n")

def add_table(title, df, n=60):
    report.append(f"\n## {title}\n")
    if df is None or len(df) == 0:
        report.append("No rows.\n")
        return
    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_table("Diagnostics", micro_diag, 30)
add_table("Trade Diagnostics", micro_trade_diag, 30)
add_table("Priority Combined Table", MICRO_PRIORITY, 120)
add_table("Basket / Pair / Subbasket Robust", MICRO_FAST_ROBUST, 80)
add_table("Lead-Lag Robust", MICRO_LEADLAG_ROBUST, 80)
add_table("Own Momentum / Reversal Robust", MICRO_OWN_ROBUST, 80)
add_table("Book / Flow Robust", MICRO_BOOK_FLOW_ROBUST, 80)
add_table("Basket / Pair / Subbasket Best Per Basket", MICRO_FAST_BEST_PER_BASKET, 80)

report_path = MICRO_OUT / "MICROCHIPS_stage1_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print("Saved:")
print((MICRO_OUT / "MICROCHIPS_stage1_report.md").resolve())
print((MICRO_OUT / "MICRO_PRIORITY.csv").resolve())
print((MICRO_OUT / "MICRO_FAST_ROBUST.csv").resolve())
print((MICRO_OUT / "MICRO_LEADLAG_ROBUST.csv").resolve())
print((MICRO_OUT / "MICRO_OWN_ROBUST.csv").resolve())
print((MICRO_OUT / "MICRO_BOOK_FLOW_ROBUST.csv").resolve())

,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,signal_quality_score,source,leader,follower,past_h,future_h,product,signal_family,signal_name,flow_window
0,"integer_combo::[0, 1, 0, -1, 1]",integer_combo,"gross_abs=3, max_abs=1",2500.0,1000.0,2.5,meanrev,"[0.0, 1.0, 0.0, -1.0, 1.0]","[0.0, 10.0, 0.0, -10.0, 10.0]",3,...,12501.341927,basket_pair_subbasket_scan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"integer_combo::[2, -1, 0, 1, -2]",integer_combo,"gross_abs=6, max_abs=2",2500.0,1000.0,2.5,meanrev,"[2.0, -1.0, 0.0, 1.0, -2.0]","[10.0, -5.0, 0.0, 5.0, -10.0]",3,...,12587.323911,basket_pair_subbasket_scan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"integer_combo::[1, -1, 0, 1, -2]",integer_combo,"gross_abs=5, max_abs=2",2500.0,1000.0,2.5,meanrev,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",3,...,12351.368626,basket_pair_subbasket_scan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"integer_combo::[1, -1, 1, 1, -1]",integer_combo,"gross_abs=5, max_abs=1",2500.0,1000.0,2.5,meanrev,"[1.0, -1.0, 1.0, 1.0, -1.0]","[10.0, -10.0, 10.0, 10.0, -10.0]",3,...,11557.672750,basket_pair_subbasket_scan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"integer_combo::[1, 1, -1, -1, 1]",integer_combo,"gross_abs=5, max_abs=1",2500.0,1000.0,2.5,meanrev,"[1.0, 1.0, -1.0, -1.0, 1.0]","[10.0, 10.0, -10.0, -10.0, 10.0]",3,...,11550.934408,basket_pair_subbasket_scan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,NaN,NaN,NaN,NaN,NaN,1.5,follow,NaN,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,...,4008.209156,leadlag_scan,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,50.0,500.0,NaN,NaN,NaN,NaN
105,NaN,NaN,NaN,NaN,NaN,2.5,follow,NaN,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,...,3962.519961,leadlag_scan,MICROCHIP_CIRCLE,MICROCHIP_TRIANGLE,250.0,500.0,NaN,NaN,NaN,NaN
106,NaN,NaN,NaN,NaN,NaN,2.5,follow,NaN,"[0.0, 10.0, 0.0, 0.0, 0.0]",3,...,4063.486463,leadlag_scan,MICROCHIP_TRIANGLE,MICROCHIP_OVAL,500.0,250.0,NaN,NaN,NaN,NaN
107,NaN,NaN,NaN,NaN,NaN,1.0,follow,NaN,"[0.0, 0.0, 0.0, 0.0, 10.0]",3,...,3996.686487,leadlag_scan,MICROCHIP_OVAL,MICROCHIP_TRIANGLE,500.0,250.0,NaN,NaN,NaN,NaN


Saved:
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage1/MICROCHIPS_stage1_report.md
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage1/MICRO_PRIORITY.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage1/MICRO_FAST_ROBUST.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage1/MICRO_LEADLAG_ROBUST.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage1/MICRO_OWN_ROBUST.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage1/MICRO_BOOK_FLOW_ROBUST.csv


In [10]:
MICRO_PRODUCTS
MICRO_FAST_DAY_BASE
micro_make_q
micro_scale_q_to_position_limit
micro_rolling_z_np
micro_vectorised_exec_arrays
micro_score_event_signal
micro_summarise_day_results
DAYS

[2, 3, 4]

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 10 — MICROCHIPS SHAPE-AWARE DISCOVERY
# Tests interpretable shape/context relationships:
# curved vs angular, circle vs oval, square vs rectangle, triangle vs quads,
# regular vs elongated, side-count baskets, and hand-picked shape residuals.
# ════════════════════════════════════════════════════════════════════════════

import time
from pathlib import Path
import numpy as np
import pandas as pd

SHAPE_OUT = Path("outputs_microchips_shape")
SHAPE_OUT.mkdir(exist_ok=True)

shape_start = time.perf_counter()

required = [
    "MICRO_PRODUCTS",
    "MICRO_FAST_DAY_BASE",
    "micro_make_q",
    "micro_scale_q_to_position_limit",
    "micro_rolling_z_np",
    "micro_vectorised_exec_arrays",
    "micro_score_event_signal",
    "micro_summarise_day_results",
    "DAYS",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables/functions: {missing}. Run the earlier Microchips cells first.")

print("Products:", MICRO_PRODUCTS)

# Expected product order:
# ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE',
#  'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']

C = "MICROCHIP_CIRCLE"
O = "MICROCHIP_OVAL"
S = "MICROCHIP_SQUARE"
R = "MICROCHIP_RECTANGLE"
T = "MICROCHIP_TRIANGLE"

SHAPE_WINDOWS = [250, 500, 1000, 2500, 5000]
SHAPE_HORIZONS = [100, 250, 500, 1000, 1500, 2500]
SHAPE_THRESHOLDS = [1.0, 1.5, 2.0, 2.5, 3.0]
SHAPE_MODES = ["meanrev", "breakout"]


def add_shape_candidate(cands, name, family, coefs, note):
    q = micro_make_q(coefs)
    q_trade = micro_scale_q_to_position_limit(q)

    cands.append({
        "candidate": name,
        "family": family,
        "note": note,
        "q_signal": q,
        "q_trade": q_trade,
    })


SHAPE_CANDIDATES = []

# ───────────────────────────────────────────────────────────────────────────
# A. Direct same-family relationships
# ───────────────────────────────────────────────────────────────────────────

add_shape_candidate(
    SHAPE_CANDIDATES,
    "circle_minus_oval",
    "same_family_pair",
    {C: 1, O: -1},
    "Curved pair: circle vs oval.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "square_minus_rectangle",
    "same_family_pair",
    {S: 1, R: -1},
    "Quadrilateral pair: square vs rectangle.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "circle_plus_oval",
    "curved_basket",
    {C: 1, O: 1},
    "Curved-chip basket.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "square_plus_rectangle",
    "quadrilateral_basket",
    {S: 1, R: 1},
    "Four-sided/quadrilateral basket.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "square_plus_rectangle_plus_triangle",
    "angular_basket",
    {S: 1, R: 1, T: 1},
    "All angular/polygon chips.",
)


# ───────────────────────────────────────────────────────────────────────────
# B. Curved vs angular / polygon relationships
# ───────────────────────────────────────────────────────────────────────────

add_shape_candidate(
    SHAPE_CANDIDATES,
    "curved_minus_angular_balanced",
    "curved_vs_angular",
    {C: 1, O: 1, S: -2/3, R: -2/3, T: -2/3},
    "Curved chips versus balanced angular chips.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "curved_minus_quadrilateral",
    "curved_vs_quadrilateral",
    {C: 1, O: 1, S: -1, R: -1},
    "Curved chips versus quadrilateral chips.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "curved_plus_triangle_minus_quads",
    "shape_group",
    {C: 1, O: 1, T: 1, S: -1, R: -1},
    "Curved + triangle versus square + rectangle. Matches one top integer idea.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "quads_minus_curved_plus_triangle",
    "shape_group",
    {S: 1, R: 1, C: -1, O: -1, T: -1},
    "Opposite of curved + triangle versus quads.",
)


# ───────────────────────────────────────────────────────────────────────────
# C. Regular vs elongated shapes
# Circle and square are more regular/symmetric.
# Oval and rectangle are elongated.
# Triangle is separate.
# ───────────────────────────────────────────────────────────────────────────

add_shape_candidate(
    SHAPE_CANDIDATES,
    "regular_minus_elongated",
    "regular_vs_elongated",
    {C: 1, S: 1, O: -1, R: -1},
    "Regular/symmetric shapes versus elongated shapes.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "regular_plus_triangle_minus_elongated",
    "regular_vs_elongated",
    {C: 1, S: 1, T: 1, O: -1, R: -1},
    "Regular + triangle versus elongated shapes.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "elongated_minus_regular",
    "regular_vs_elongated",
    {O: 1, R: 1, C: -1, S: -1},
    "Elongated shapes versus regular/symmetric shapes.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "oval_rectangle_minus_circle_square_triangle",
    "regular_vs_elongated",
    {O: 1, R: 1, C: -2/3, S: -2/3, T: -2/3},
    "Elongated shapes versus balanced non-elongated chips.",
)


# ───────────────────────────────────────────────────────────────────────────
# D. Side-count / angle-count inspired relationships
# Curved = 0 sides, triangle = 3, square/rectangle = 4.
# These are not direct price formulas, but they can create explainable factors.
# ───────────────────────────────────────────────────────────────────────────

add_shape_candidate(
    SHAPE_CANDIDATES,
    "side_count_angular_factor",
    "side_count",
    {S: 4, R: 4, T: 3},
    "Raw angular side-count factor.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "side_count_minus_curved_balanced",
    "side_count",
    {S: 4, R: 4, T: 3, C: -11/2, O: -11/2},
    "Side-count angular factor versus curved chips, balanced gross notionally.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "four_sides_minus_triangle_scaled",
    "side_count",
    {S: 1, R: 1, T: -4/3},
    "Four-sided chips versus triangle scaled by side count.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "triangle_minus_four_sides_balanced",
    "side_count",
    {T: 1, S: -0.5, R: -0.5},
    "Triangle versus average quadrilateral.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "triangle_minus_curved",
    "triangle_vs_curved",
    {T: 1, C: -0.5, O: -0.5},
    "Triangle versus average curved chip.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "triangle_minus_all_others",
    "triangle_residual",
    {T: 1, C: -0.25, O: -0.25, S: -0.25, R: -0.25},
    "Triangle residual versus all other chips.",
)


# ───────────────────────────────────────────────────────────────────────────
# E. Top integer combos reframed as shape hypotheses
# These came from Stage 1, but we add them with interpretable labels.
# ───────────────────────────────────────────────────────────────────────────

add_shape_candidate(
    SHAPE_CANDIDATES,
    "stage1_top_oval_rect_triangle",
    "stage1_interpretable",
    {O: 1, R: -1, T: 1},
    "OVAL + TRIANGLE versus RECTANGLE. Top Stage 1 simple 3-leg combo.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "stage1_top_circle_oval_rect_2triangle",
    "stage1_interpretable",
    {C: 1, O: -1, R: 1, T: -2},
    "CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. Very high Stage 1 hit-rate.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "stage1_top_2circle_oval_rect_2triangle",
    "stage1_interpretable",
    {C: 2, O: -1, R: 1, T: -2},
    "2×CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "stage1_curved_triangle_vs_quads",
    "stage1_interpretable",
    {C: 1, O: 1, T: 1, S: -1, R: -1},
    "CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTANGLE.",
)

add_shape_candidate(
    SHAPE_CANDIDATES,
    "stage1_circle_oval_square_rect_triangle_mix",
    "stage1_interpretable",
    {C: 1, O: -1, S: 1, R: 1, T: -1},
    "CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIANGLE.",
)

print(f"Shape-aware candidates: {len(SHAPE_CANDIDATES)}")

for c in SHAPE_CANDIDATES:
    print(
        c["candidate"],
        "| family:", c["family"],
        "| q_signal:", np.round(c["q_signal"], 4).tolist(),
        "| q_trade:", np.round(c["q_trade"], 4).tolist(),
    )


# ───────────────────────────────────────────────────────────────────────────
# Fast overlapping fixed-horizon scan for shape candidates
# ───────────────────────────────────────────────────────────────────────────

shape_rows = []

for ci, cand in enumerate(SHAPE_CANDIDATES, start=1):
    if ci == 1 or ci % 5 == 0:
        print(f"[{time.perf_counter() - shape_start:7.2f}s] Shape candidate {ci}/{len(SHAPE_CANDIDATES)}: {cand['candidate']}")

    q_signal = np.asarray(cand["q_signal"], dtype=float)
    q_trade = np.asarray(cand["q_trade"], dtype=float)

    for day in DAYS:
        base = MICRO_FAST_DAY_BASE[day]
        X = base["X"]
        B = base["B"]
        A = base["A"]

        spread = X @ q_signal

        for window in SHAPE_WINDOWS:
            z_full = micro_rolling_z_np(spread, window)

            for h in SHAPE_HORIZONS:
                if X.shape[0] <= h + window:
                    continue

                exec_long, mid_long, cost_long = micro_vectorised_exec_arrays(X, B, A, q_trade, h)
                z = z_full[:-h]

                assert len(z) == len(exec_long)

                for threshold in SHAPE_THRESHOLDS:
                    for mode in SHAPE_MODES:
                        stats = micro_score_event_signal(
                            z=z,
                            exec_long=exec_long,
                            mid_long=mid_long,
                            cost_long=cost_long,
                            threshold=threshold,
                            mode=mode,
                        )

                        shape_rows.append({
                            "candidate": cand["candidate"],
                            "family": cand["family"],
                            "note": cand["note"],
                            "day": day,
                            "window": window,
                            "horizon": h,
                            "threshold": threshold,
                            "mode": mode,
                            "q_signal": str(np.round(q_signal, 6).tolist()),
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                            **stats,
                        })

MICRO_SHAPE_DAY_RESULTS = pd.DataFrame(shape_rows)

MICRO_SHAPE_SUMMARY = micro_summarise_day_results(
    MICRO_SHAPE_DAY_RESULTS,
    group_cols=[
        "candidate", "family", "note",
        "window", "horizon", "threshold", "mode",
        "q_signal", "q_trade",
    ],
)

MICRO_SHAPE_ROBUST = MICRO_SHAPE_SUMMARY[
    (MICRO_SHAPE_SUMMARY["unique_days"] == 3)
    & (MICRO_SHAPE_SUMMARY["positive_avg_edge_days"] == 3)
    & (MICRO_SHAPE_SUMMARY["event_count"] >= 200)
    & (MICRO_SHAPE_SUMMARY["day_avg_exec_pnl_mean"] > 0)
].copy()

MICRO_SHAPE_ROBUST = MICRO_SHAPE_ROBUST.sort_values(
    ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
    ascending=[False, False, False],
)

MICRO_SHAPE_BEST_PER_CANDIDATE = (
    MICRO_SHAPE_ROBUST
    .groupby("candidate", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

print("\nTop MICRO_SHAPE_ROBUST:")
display(MICRO_SHAPE_ROBUST.head(100))

print("\nMICRO_SHAPE_BEST_PER_CANDIDATE:")
display(MICRO_SHAPE_BEST_PER_CANDIDATE)

MICRO_SHAPE_DAY_RESULTS.to_csv(SHAPE_OUT / "MICRO_SHAPE_DAY_RESULTS.csv", index=False)
MICRO_SHAPE_SUMMARY.to_csv(SHAPE_OUT / "MICRO_SHAPE_SUMMARY.csv", index=False)
MICRO_SHAPE_ROBUST.to_csv(SHAPE_OUT / "MICRO_SHAPE_ROBUST.csv", index=False)
MICRO_SHAPE_BEST_PER_CANDIDATE.to_csv(SHAPE_OUT / "MICRO_SHAPE_BEST_PER_CANDIDATE.csv", index=False)

print("\nSaved:")
print((SHAPE_OUT / "MICRO_SHAPE_ROBUST.csv").resolve())
print((SHAPE_OUT / "MICRO_SHAPE_BEST_PER_CANDIDATE.csv").resolve())
print(f"Runtime: {time.perf_counter() - shape_start:.2f}s")

Products: ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE', 'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']
Shape-aware candidates: 24
circle_minus_oval | family: same_family_pair | q_signal: [1.0, -1.0, 0.0, 0.0, 0.0] | q_trade: [10.0, -10.0, 0.0, 0.0, 0.0]
square_minus_rectangle | family: same_family_pair | q_signal: [0.0, 0.0, 1.0, -1.0, 0.0] | q_trade: [0.0, 0.0, 10.0, -10.0, 0.0]
circle_plus_oval | family: curved_basket | q_signal: [1.0, 1.0, 0.0, 0.0, 0.0] | q_trade: [10.0, 10.0, 0.0, 0.0, 0.0]
square_plus_rectangle | family: quadrilateral_basket | q_signal: [0.0, 0.0, 1.0, 1.0, 0.0] | q_trade: [0.0, 0.0, 10.0, 10.0, 0.0]
square_plus_rectangle_plus_triangle | family: angular_basket | q_signal: [0.0, 0.0, 1.0, 1.0, 1.0] | q_trade: [0.0, 0.0, 10.0, 10.0, 10.0]
curved_minus_angular_balanced | family: curved_vs_angular | q_signal: [1.0, 1.0, -0.6667, -0.6667, -0.6667] | q_trade: [10.0, 10.0, -6.6667, -6.6667, -6.6667]
curved_minus_quadrilateral | family: curved_vs_quadrilateral

,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
6219,stage1_top_oval_rect_triangle,stage1_interpretable,OVAL + TRIANGLE versus RECTANGLE. Top Stage 1 ...,2500,1000,3.0,meanrev,"[0.0, 1.0, 0.0, -1.0, 1.0]","[0.0, 10.0, 0.0, -10.0, 10.0]",3,...,3,1.000000,1.000000,243.623904,3.154611,38570.0,2393260.0,3,1.000000,18278.355263
4489,square_plus_rectangle,quadrilateral_basket,Four-sided/quadrilateral basket.,5000,1500,3.0,meanrev,"[0.0, 0.0, 1.0, 1.0, 0.0]","[0.0, 0.0, 10.0, 10.0, 0.0]",3,...,3,0.918848,0.756545,199.210626,3.169170,415400.0,1430020.0,3,0.809035,15561.627529
1678,elongated_minus_regular,regular_vs_elongated,Elongated shapes versus regular/symmetric shapes.,1000,2500,3.0,breakout,"[-1.0, 1.0, -1.0, 1.0, 0.0]","[-10.0, 10.0, -10.0, 10.0, 0.0]",3,...,3,0.896353,0.728723,353.127855,3.272426,525940.0,1603530.0,3,0.843931,14271.795169
4487,square_plus_rectangle,quadrilateral_basket,Four-sided/quadrilateral basket.,5000,1500,2.5,meanrev,"[0.0, 0.0, 1.0, 1.0, 0.0]","[0.0, 0.0, 10.0, 10.0, 0.0]",3,...,3,0.931017,0.793051,199.016196,2.844143,3205570.0,5832620.0,3,0.910865,14037.355420
5019,stage1_circle_oval_square_rect_triangle_mix,stage1_interpretable,CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIA...,2500,1000,3.0,meanrev,"[1.0, -1.0, 1.0, 1.0, -1.0]","[10.0, -10.0, 10.0, 10.0, -10.0]",3,...,3,0.979167,0.937500,441.141208,3.209805,482350.0,1954680.0,3,0.982206,14069.699819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2994,regular_minus_elongated,regular_vs_elongated,Regular/symmetric shapes versus elongated shapes.,5000,2500,2.0,breakout,"[1.0, -1.0, 1.0, -1.0, 0.0]","[10.0, -10.0, 10.0, -10.0, 0.0]",3,...,3,0.739292,0.655385,353.178266,2.355080,1255530.0,12896470.0,3,0.748644,9597.616524
2075,four_sides_minus_triangle_scaled,side_count,Four-sided chips versus triangle scaled by sid...,5000,1000,2.0,meanrev,"[0.0, 0.0, 1.0, 1.0, -1.333333]","[0.0, 0.0, 7.5, 7.5, -10.0]",3,...,3,0.863400,0.798381,235.666354,2.391286,2875807.5,9388132.5,3,0.846328,9718.511666
1117,curved_minus_quadrilateral,curved_vs_quadrilateral,Curved chips versus quadrilateral chips.,2500,1000,2.5,meanrev,"[1.0, 1.0, -1.0, -1.0, 0.0]","[10.0, 10.0, -10.0, -10.0, 0.0]",3,...,3,0.912603,0.873544,355.609019,2.855863,2106750.0,4819120.0,3,0.918785,9747.729809
525,circle_plus_oval,curved_basket,Curved-chip basket.,2500,1500,2.0,meanrev,"[1.0, 1.0, 0.0, 0.0, 0.0]","[10.0, 10.0, 0.0, 0.0, 0.0]",3,...,3,0.756287,0.492846,157.717534,2.419601,130770.0,6361340.0,3,0.741779,9569.693663



MICRO_SHAPE_BEST_PER_CANDIDATE:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
0,stage1_top_oval_rect_triangle,stage1_interpretable,OVAL + TRIANGLE versus RECTANGLE. Top Stage 1 ...,2500,1000,3.0,meanrev,"[0.0, 1.0, 0.0, -1.0, 1.0]","[0.0, 10.0, 0.0, -10.0, 10.0]",3,...,3,1.000000,1.000000,243.623904,3.154611,38570.000000,2.393260e+06,3,1.000000,18278.355263
1,square_plus_rectangle,quadrilateral_basket,Four-sided/quadrilateral basket.,5000,1500,3.0,meanrev,"[0.0, 0.0, 1.0, 1.0, 0.0]","[0.0, 0.0, 10.0, 10.0, 0.0]",3,...,3,0.918848,0.756545,199.210626,3.169170,415400.000000,1.430020e+06,3,0.809035,15561.627529
2,elongated_minus_regular,regular_vs_elongated,Elongated shapes versus regular/symmetric shapes.,1000,2500,3.0,breakout,"[-1.0, 1.0, -1.0, 1.0, 0.0]","[-10.0, 10.0, -10.0, 10.0, 0.0]",3,...,3,0.896353,0.728723,353.127855,3.272426,525940.000000,1.603530e+06,3,0.843931,14271.795169
3,stage1_circle_oval_square_rect_triangle_mix,stage1_interpretable,CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIA...,2500,1000,3.0,meanrev,"[1.0, -1.0, 1.0, 1.0, -1.0]","[10.0, -10.0, 10.0, 10.0, -10.0]",3,...,3,0.979167,0.937500,441.141208,3.209805,482350.000000,1.954680e+06,3,0.982206,14069.699819
4,regular_minus_elongated,regular_vs_elongated,Regular/symmetric shapes versus elongated shapes.,1000,2500,3.0,breakout,"[1.0, -1.0, 1.0, -1.0, 0.0]","[10.0, -10.0, 10.0, -10.0, 0.0]",3,...,3,0.878623,0.675532,353.127855,3.272426,478090.000000,1.541140e+06,3,0.815029,13597.892370
5,square_plus_rectangle_plus_triangle,angular_basket,All angular/polygon chips.,2500,2500,3.0,meanrev,"[0.0, 0.0, 1.0, 1.0, 1.0]","[0.0, 0.0, 10.0, 10.0, 10.0]",3,...,3,0.966667,0.900000,282.773059,3.227486,216090.000000,3.610490e+06,3,0.948624,13672.881861
6,quads_minus_curved_plus_triangle,shape_group,Opposite of curved + triangle versus quads.,2500,1000,3.0,meanrev,"[-1.0, -1.0, 1.0, 1.0, -1.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]",3,...,3,1.000000,1.000000,443.951635,3.228424,891370.000000,1.698240e+06,3,1.000000,13297.846514
7,stage1_top_circle_oval_rect_2triangle,stage1_interpretable,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. V...,2500,2500,2.5,meanrev,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",3,...,3,0.972222,0.916667,206.380540,2.763721,256925.000000,1.042365e+06,3,0.962733,12896.055780
8,four_sides_minus_triangle_scaled,side_count,Four-sided chips versus triangle scaled by sid...,5000,1500,2.5,meanrev,"[0.0, 0.0, 1.0, 1.0, -1.333333]","[0.0, 0.0, 7.5, 7.5, -10.0]",3,...,3,0.967817,0.916667,235.369476,2.815934,928335.000000,2.589875e+06,3,0.969144,12760.697318
9,regular_plus_triangle_minus_elongated,regular_vs_elongated,Regular + triangle versus elongated shapes.,2500,1000,3.0,meanrev,"[1.0, -1.0, 1.0, -1.0, 1.0]","[10.0, -10.0, 10.0, -10.0, 10.0]",3,...,3,0.990196,0.970588,440.452002,3.226597,178530.000000,1.492510e+06,3,0.995495,12723.659432



Saved:
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_shape/MICRO_SHAPE_ROBUST.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_shape/MICRO_SHAPE_BEST_PER_CANDIDATE.csv
Runtime: 3.44s


In [12]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 11 — MICROCHIPS STAGE 2 OPTIMISER
# Non-overlapping trades, realistic bid/ask execution, fixed-hold and optional
# z-convergence exits.
# ════════════════════════════════════════════════════════════════════════════

import time
from pathlib import Path
import numpy as np
import pandas as pd

STAGE2_OUT = Path("outputs_microchips_stage2")
STAGE2_OUT.mkdir(exist_ok=True)

stage2_start = time.perf_counter()

required = [
    "MICRO_PRODUCTS",
    "MICRO_FAST_DAY_BASE",
    "micro_make_q",
    "micro_scale_q_to_position_limit",
    "micro_rolling_z_np",
    "DAYS",
]

missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables/functions: {missing}. Run previous Microchips cells first.")

C = "MICROCHIP_CIRCLE"
O = "MICROCHIP_OVAL"
S = "MICROCHIP_SQUARE"
R = "MICROCHIP_RECTANGLE"
T = "MICROCHIP_TRIANGLE"

# Parameter grid is smaller than Stage 1, focused on plausible values.
STAGE2_WINDOWS = [500, 1000, 2500, 5000]
STAGE2_ENTRY_ZS = [1.5, 2.0, 2.5, 3.0]
STAGE2_EXIT_ZS = [None, 0.25, 0.5, 0.75]
STAGE2_MAX_HOLDS = [250, 500, 1000, 1500, 2500]


def stage2_pnl_for_trades(X, B, A, q_trade, entry_idx, exit_idx, side):
    entry_idx = np.asarray(entry_idx, dtype=int)
    exit_idx = np.asarray(exit_idx, dtype=int)
    side = np.asarray(side, dtype=float)

    assert len(entry_idx) == len(exit_idx) == len(side)

    if len(entry_idx) == 0:
        return {
            "exec_pnl": np.array([]),
            "mid_pnl": np.array([]),
            "cost": np.array([]),
            "target": np.empty((0, len(q_trade))),
        }

    targets = side[:, None] * q_trade.reshape(1, -1)

    X0 = X[entry_idx]
    X1 = X[exit_idx]
    B0 = B[entry_idx]
    A0 = A[entry_idx]
    B1 = B[exit_idx]
    A1 = A[exit_idx]

    entry_px = np.where(targets > 0, A0, B0)
    exit_px = np.where(targets > 0, B1, A1)

    exec_pnl = ((exit_px - entry_px) * targets).sum(axis=1)
    mid_pnl = ((X1 - X0) * targets).sum(axis=1)
    cost = mid_pnl - exec_pnl

    if len(cost):
        assert np.nanmin(cost) >= -1e-7, f"Cost identity failed. min cost={np.nanmin(cost)}"

    return {
        "exec_pnl": exec_pnl,
        "mid_pnl": mid_pnl,
        "cost": cost,
        "target": targets,
    }


def stage2_select_non_overlapping_trades(z, mode, entry_z, exit_z, max_hold):
    n = len(z)

    entry_candidates = np.flatnonzero(np.abs(z) >= entry_z)
    entry_candidates = entry_candidates[entry_candidates + max_hold < n]

    entries = []
    exits = []
    sides = []
    holds = []

    last_exit = -1

    for i in entry_candidates:
        if i <= last_exit:
            continue

        zi = z[i]
        if not np.isfinite(zi) or zi == 0:
            continue

        if mode == "meanrev":
            side = -np.sign(zi)
        elif mode == "breakout":
            side = np.sign(zi)
        elif mode == "follow":
            side = np.sign(zi)
        elif mode == "inverse":
            side = -np.sign(zi)
        else:
            raise ValueError(mode)

        hard_exit = min(i + max_hold, n - 1)
        exit_i = hard_exit

        if exit_z is not None:
            search_region = np.abs(z[i + 1: hard_exit + 1])
            found = np.flatnonzero(search_region <= exit_z)
            if len(found) > 0:
                exit_i = i + 1 + int(found[0])

        if exit_i <= i:
            continue

        entries.append(i)
        exits.append(exit_i)
        sides.append(side)
        holds.append(exit_i - i)

        last_exit = exit_i

    return (
        np.asarray(entries, dtype=int),
        np.asarray(exits, dtype=int),
        np.asarray(sides, dtype=float),
        np.asarray(holds, dtype=int),
    )


def add_stage2_z_candidate(cands, name, family, coefs, mode, note):
    q_signal = micro_make_q(coefs)
    q_trade = micro_scale_q_to_position_limit(q_signal)

    cands.append({
        "candidate": name,
        "signal_type": "zspread",
        "family": family,
        "mode": mode,
        "note": note,
        "q_signal": q_signal,
        "q_trade": q_trade,
        "extra": {},
    })


def add_stage2_own_move_candidate(cands, name, product, past_h, mode, note):
    q = micro_make_q({product: 1})
    q_trade = micro_scale_q_to_position_limit(q)

    cands.append({
        "candidate": name,
        "signal_type": "own_move",
        "family": "own_momentum_reversal",
        "mode": mode,
        "note": note,
        "q_signal": q,
        "q_trade": q_trade,
        "extra": {"product": product, "past_h": past_h},
    })


def add_stage2_leadlag_candidate(cands, name, leader, follower, past_h, mode, note):
    q = micro_make_q({follower: 1})
    q_trade = micro_scale_q_to_position_limit(q)

    cands.append({
        "candidate": name,
        "signal_type": "leadlag",
        "family": "leadlag",
        "mode": mode,
        "note": note,
        "q_signal": q,
        "q_trade": q_trade,
        "extra": {"leader": leader, "follower": follower, "past_h": past_h},
    })


def add_stage2_flow_candidate(cands, name, product, flow_window, mode, note):
    q = micro_make_q({product: 1})
    q_trade = micro_scale_q_to_position_limit(q)

    cands.append({
        "candidate": name,
        "signal_type": "flow",
        "family": "trade_flow",
        "mode": mode,
        "note": note,
        "q_signal": q,
        "q_trade": q_trade,
        "extra": {"product": product, "flow_window": flow_window},
    })


STAGE2_MICRO_CANDIDATES = []

# ───────────────────────────────────────────────────────────────────────────
# A. Best Stage 1 integer / shape-aware mean-reversion baskets
# ───────────────────────────────────────────────────────────────────────────

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_circle_oval_rect_2tri",
    "integer_shape_mr",
    {C: 1, O: -1, R: 1, T: -2},
    "meanrev",
    "CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. Highest-priority Stage 1 candidate.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_2circle_oval_rect_2tri",
    "integer_shape_mr",
    {C: 2, O: -1, R: 1, T: -2},
    "meanrev",
    "2×CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_oval_rect_triangle",
    "integer_shape_mr",
    {O: 1, R: -1, T: 1},
    "meanrev",
    "OVAL + TRIANGLE versus RECTANGLE.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_curved_triangle_vs_quads",
    "shape_group_mr",
    {C: 1, O: 1, T: 1, S: -1, R: -1},
    "meanrev",
    "CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTANGLE.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_circle_square_rect_vs_oval_triangle",
    "shape_group_mr",
    {C: 1, S: 1, R: 1, O: -1, T: -1},
    "meanrev",
    "CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIANGLE.",
)


# ───────────────────────────────────────────────────────────────────────────
# B. New explicit context-clue candidates
# ───────────────────────────────────────────────────────────────────────────

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_circle_minus_oval",
    "same_family_pair",
    {C: 1, O: -1},
    "meanrev",
    "Curved pair mean reversion.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_square_minus_rectangle",
    "same_family_pair",
    {S: 1, R: -1},
    "meanrev",
    "Quadrilateral pair mean reversion.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_regular_minus_elongated",
    "regular_vs_elongated",
    {C: 1, S: 1, O: -1, R: -1},
    "meanrev",
    "Regular/symmetric shapes versus elongated shapes.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_triangle_minus_quads",
    "triangle_vs_quads",
    {T: 1, S: -0.5, R: -0.5},
    "meanrev",
    "Triangle versus average quadrilateral.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_curved_minus_quads",
    "curved_vs_quadrilateral",
    {C: 1, O: 1, S: -1, R: -1},
    "meanrev",
    "Curved chips versus quadrilateral chips.",
)


# ───────────────────────────────────────────────────────────────────────────
# C. Earlier pair ideas
# ───────────────────────────────────────────────────────────────────────────

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_circle_minus_square",
    "old_pair",
    {C: 1, S: -1},
    "meanrev",
    "Earlier CIRCLE/SQUARE pair idea.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_circle_minus_triangle",
    "old_pair",
    {C: 1, T: -1},
    "meanrev",
    "Earlier CIRCLE/TRIANGLE pair idea.",
)

add_stage2_z_candidate(
    STAGE2_MICRO_CANDIDATES,
    "mr_square_minus_triangle",
    "old_pair",
    {S: 1, T: -1},
    "meanrev",
    "Earlier SQUARE/TRIANGLE pair idea.",
)


# ───────────────────────────────────────────────────────────────────────────
# D. Own momentum/reversal and lead-lag candidates
# ───────────────────────────────────────────────────────────────────────────

add_stage2_own_move_candidate(
    STAGE2_MICRO_CANDIDATES,
    "own_square_momentum_past50",
    S,
    past_h=50,
    mode="follow",
    note="SQUARE own momentum after strong 50-tick move.",
)

add_stage2_own_move_candidate(
    STAGE2_MICRO_CANDIDATES,
    "own_square_momentum_past100",
    S,
    past_h=100,
    mode="follow",
    note="SQUARE own momentum after strong 100-tick move.",
)

add_stage2_own_move_candidate(
    STAGE2_MICRO_CANDIDATES,
    "own_rectangle_reversal_past500",
    R,
    past_h=500,
    mode="inverse",
    note="RECTANGLE own reversal after 500-tick move.",
)

add_stage2_own_move_candidate(
    STAGE2_MICRO_CANDIDATES,
    "own_circle_reversal_past500",
    C,
    past_h=500,
    mode="inverse",
    note="CIRCLE own reversal after 500-tick move.",
)

add_stage2_leadlag_candidate(
    STAGE2_MICRO_CANDIDATES,
    "lead_oval_to_triangle_follow_500",
    O,
    T,
    past_h=500,
    mode="follow",
    note="OVAL move predicts TRIANGLE follow.",
)

add_stage2_leadlag_candidate(
    STAGE2_MICRO_CANDIDATES,
    "lead_circle_to_square_inverse_50",
    C,
    S,
    past_h=50,
    mode="inverse",
    note="CIRCLE move predicts opposite SQUARE move.",
)


# ───────────────────────────────────────────────────────────────────────────
# E. Flow candidates, if trade flow arrays exist
# ───────────────────────────────────────────────────────────────────────────

flow_available = all("trade_signed_qty" in MICRO_FAST_DAY_BASE[day] for day in DAYS)

if flow_available:
    add_stage2_flow_candidate(
        STAGE2_MICRO_CANDIDATES,
        "flow_square_follow_5000",
        S,
        flow_window=5000,
        mode="follow",
        note="SQUARE follows long-window aggressive flow.",
    )

    add_stage2_flow_candidate(
        STAGE2_MICRO_CANDIDATES,
        "flow_square_follow_10000",
        S,
        flow_window=10000,
        mode="follow",
        note="SQUARE follows very-long-window aggressive flow.",
    )

    add_stage2_flow_candidate(
        STAGE2_MICRO_CANDIDATES,
        "flow_triangle_follow_10000",
        T,
        flow_window=10000,
        mode="follow",
        note="TRIANGLE follows long-window aggressive flow.",
    )
else:
    print("Flow arrays not found. Flow candidates skipped. Run Cell 8 first if you want them.")


print("Stage 2 Microchip candidates:", len(STAGE2_MICRO_CANDIDATES))
for c in STAGE2_MICRO_CANDIDATES:
    print(
        c["candidate"],
        "| type:", c["signal_type"],
        "| mode:", c["mode"],
        "| q_trade:", np.round(c["q_trade"], 4).tolist(),
        "| note:", c["note"],
    )


# ───────────────────────────────────────────────────────────────────────────
# Build signal series for each candidate/day/window
# ───────────────────────────────────────────────────────────────────────────

def build_stage2_signal(cand, base, window):
    X = base["X"]

    signal_type = cand["signal_type"]

    if signal_type == "zspread":
        raw = X @ cand["q_signal"]
        return micro_rolling_z_np(raw, window)

    if signal_type == "own_move":
        p = cand["extra"]["product"]
        past_h = cand["extra"]["past_h"]
        ip = MICRO_PRODUCTS.index(p)

        raw = np.full(X.shape[0], np.nan)
        raw[past_h:] = X[past_h:, ip] - X[:-past_h, ip]
        return micro_rolling_z_np(raw, window)

    if signal_type == "leadlag":
        leader = cand["extra"]["leader"]
        past_h = cand["extra"]["past_h"]
        il = MICRO_PRODUCTS.index(leader)

        raw = np.full(X.shape[0], np.nan)
        raw[past_h:] = X[past_h:, il] - X[:-past_h, il]
        return micro_rolling_z_np(raw, window)

    if signal_type == "flow":
        if "trade_signed_qty" not in base:
            return np.zeros(X.shape[0], dtype=float)

        p = cand["extra"]["product"]
        fw = cand["extra"]["flow_window"]
        ip = MICRO_PRODUCTS.index(p)

        grid_w = max(1, fw // 100)

        signed = pd.Series(base["trade_signed_qty"][:, ip])
        gross = pd.Series(base["trade_gross_qty"][:, ip])

        signed_roll = signed.rolling(grid_w, min_periods=1).sum().to_numpy(float)
        gross_roll = gross.rolling(grid_w, min_periods=1).sum().to_numpy(float)

        raw = signed_roll / np.where(gross_roll == 0, np.nan, gross_roll)
        raw[~np.isfinite(raw)] = 0.0

        return micro_rolling_z_np(raw, window)

    raise ValueError(signal_type)


# ───────────────────────────────────────────────────────────────────────────
# Run optimiser
# ───────────────────────────────────────────────────────────────────────────

day_rows = []
trade_rows = []

total_configs = (
    len(STAGE2_MICRO_CANDIDATES)
    * len(STAGE2_WINDOWS)
    * len(STAGE2_ENTRY_ZS)
    * len(STAGE2_EXIT_ZS)
    * len(STAGE2_MAX_HOLDS)
)

print("Total config combos:", total_configs)

for ci, cand in enumerate(STAGE2_MICRO_CANDIDATES, start=1):
    print(f"[{time.perf_counter() - stage2_start:7.2f}s] Candidate {ci}/{len(STAGE2_MICRO_CANDIDATES)}: {cand['candidate']}")

    q_trade = np.asarray(cand["q_trade"], dtype=float)
    mode = cand["mode"]

    for window in STAGE2_WINDOWS:
        for entry_z in STAGE2_ENTRY_ZS:
            for exit_z in STAGE2_EXIT_ZS:
                for max_hold in STAGE2_MAX_HOLDS:
                    for day in DAYS:
                        base = MICRO_FAST_DAY_BASE[day]
                        X = base["X"]
                        B = base["B"]
                        A = base["A"]
                        ts = base["timestamps"]

                        z = build_stage2_signal(cand, base, window)

                        entry_idx, exit_idx, side, holds = stage2_select_non_overlapping_trades(
                            z=z,
                            mode=mode,
                            entry_z=entry_z,
                            exit_z=exit_z,
                            max_hold=max_hold,
                        )

                        pnl = stage2_pnl_for_trades(
                            X=X,
                            B=B,
                            A=A,
                            q_trade=q_trade,
                            entry_idx=entry_idx,
                            exit_idx=exit_idx,
                            side=side,
                        )

                        exec_pnl = pnl["exec_pnl"]
                        mid_pnl = pnl["mid_pnl"]
                        cost = pnl["cost"]

                        trade_count = len(exec_pnl)

                        row = {
                            "candidate": cand["candidate"],
                            "signal_type": cand["signal_type"],
                            "family": cand["family"],
                            "note": cand["note"],
                            "mode": mode,
                            "day": day,
                            "window": window,
                            "entry_z": entry_z,
                            "exit_z": "fixed" if exit_z is None else exit_z,
                            "max_hold": max_hold,
                            "trade_count": trade_count,
                            "total_exec_pnl": float(np.sum(exec_pnl)) if trade_count else 0.0,
                            "total_mid_pnl": float(np.sum(mid_pnl)) if trade_count else 0.0,
                            "total_cost": float(np.sum(cost)) if trade_count else 0.0,
                            "avg_exec_pnl": float(np.mean(exec_pnl)) if trade_count else np.nan,
                            "median_exec_pnl": float(np.median(exec_pnl)) if trade_count else np.nan,
                            "hit_rate": float(np.mean(exec_pnl > 0)) if trade_count else np.nan,
                            "avg_hold": float(np.mean(holds)) if trade_count else 0.0,
                            "max_hold_seen": int(np.max(holds)) if trade_count else 0,
                            "q_signal": str(np.round(cand["q_signal"], 6).tolist()),
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                            "extra": str(cand["extra"]),
                        }

                        day_rows.append(row)

                        if trade_count:
                            trade_rows.append(pd.DataFrame({
                                "candidate": cand["candidate"],
                                "signal_type": cand["signal_type"],
                                "family": cand["family"],
                                "mode": mode,
                                "day": day,
                                "window": window,
                                "entry_z": entry_z,
                                "exit_z": "fixed" if exit_z is None else exit_z,
                                "max_hold": max_hold,
                                "entry_idx": entry_idx,
                                "exit_idx": exit_idx,
                                "entry_ts": ts[entry_idx],
                                "exit_ts": ts[exit_idx],
                                "side": side,
                                "hold": holds,
                                "exec_pnl": exec_pnl,
                                "mid_pnl": mid_pnl,
                                "cost": cost,
                            }))

MICRO_STAGE2_DAY_RESULTS = pd.DataFrame(day_rows)
MICRO_STAGE2_TRADE_LOG = pd.concat(trade_rows, ignore_index=True) if trade_rows else pd.DataFrame()

print("MICRO_STAGE2_DAY_RESULTS:", MICRO_STAGE2_DAY_RESULTS.shape)
print("MICRO_STAGE2_TRADE_LOG:", MICRO_STAGE2_TRADE_LOG.shape)

display(MICRO_STAGE2_DAY_RESULTS.head())
display(MICRO_STAGE2_TRADE_LOG.head())


# ───────────────────────────────────────────────────────────────────────────
# Summaries
# ───────────────────────────────────────────────────────────────────────────

MICRO_STAGE2_SUMMARY = (
    MICRO_STAGE2_DAY_RESULTS
    .groupby([
        "candidate", "signal_type", "family", "note", "mode",
        "window", "entry_z", "exit_z", "max_hold",
        "q_signal", "q_trade", "extra",
    ])
    .agg(
        days=("day", "count"),
        active_days=("trade_count", lambda x: int((x > 0).sum())),
        trade_count=("trade_count", "sum"),
        total_exec_pnl=("total_exec_pnl", "sum"),
        total_mid_pnl=("total_mid_pnl", "sum"),
        total_cost=("total_cost", "sum"),
        avg_day_exec_pnl=("total_exec_pnl", "mean"),
        worst_day_exec_pnl=("total_exec_pnl", "min"),
        best_day_exec_pnl=("total_exec_pnl", "max"),
        positive_days=("total_exec_pnl", lambda x: int((x > 0).sum())),
        avg_trade_pnl=("avg_exec_pnl", "mean"),
        median_trade_pnl=("median_exec_pnl", "mean"),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        avg_hold=("avg_hold", "mean"),
        max_hold_seen=("max_hold_seen", "max"),
    )
    .reset_index()
)

MICRO_STAGE2_SUMMARY["pnl_per_trade"] = (
    MICRO_STAGE2_SUMMARY["total_exec_pnl"]
    / MICRO_STAGE2_SUMMARY["trade_count"].replace(0, np.nan)
)

MICRO_STAGE2_SUMMARY["pnl_per_cost"] = (
    MICRO_STAGE2_SUMMARY["total_exec_pnl"]
    / MICRO_STAGE2_SUMMARY["total_cost"].replace(0, np.nan)
)

MICRO_STAGE2_SUMMARY["one_day_dependency"] = (
    MICRO_STAGE2_SUMMARY["best_day_exec_pnl"]
    / MICRO_STAGE2_SUMMARY["total_exec_pnl"].replace(0, np.nan)
)

MICRO_STAGE2_ROBUST = MICRO_STAGE2_SUMMARY[
    (MICRO_STAGE2_SUMMARY["days"] == len(DAYS))
    & (MICRO_STAGE2_SUMMARY["active_days"] == len(DAYS))
    & (MICRO_STAGE2_SUMMARY["positive_days"] == len(DAYS))
    & (MICRO_STAGE2_SUMMARY["total_exec_pnl"] > 0)
    & (MICRO_STAGE2_SUMMARY["trade_count"] >= 3)
    & (MICRO_STAGE2_SUMMARY["one_day_dependency"] <= 0.80)
].copy()

MICRO_STAGE2_ROBUST = MICRO_STAGE2_ROBUST.sort_values(
    ["total_exec_pnl", "pnl_per_trade", "mean_hit_rate"],
    ascending=[False, False, False],
)

MICRO_STAGE2_BEST_BY_CANDIDATE = (
    MICRO_STAGE2_ROBUST
    .groupby("candidate", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

MICRO_STAGE2_SUMMARY = MICRO_STAGE2_SUMMARY.sort_values(
    ["positive_days", "total_exec_pnl", "pnl_per_trade", "mean_hit_rate"],
    ascending=[False, False, False, False],
)

print("\nTop MICRO_STAGE2_ROBUST:")
display(MICRO_STAGE2_ROBUST.head(120))

print("\nMICRO_STAGE2_BEST_BY_CANDIDATE:")
display(MICRO_STAGE2_BEST_BY_CANDIDATE)

print("\nTop MICRO_STAGE2_SUMMARY:")
display(MICRO_STAGE2_SUMMARY.head(120))


# ───────────────────────────────────────────────────────────────────────────
# Save
# ───────────────────────────────────────────────────────────────────────────

MICRO_STAGE2_DAY_RESULTS.to_csv(STAGE2_OUT / "MICRO_STAGE2_DAY_RESULTS.csv", index=False)
MICRO_STAGE2_SUMMARY.to_csv(STAGE2_OUT / "MICRO_STAGE2_SUMMARY.csv", index=False)
MICRO_STAGE2_ROBUST.to_csv(STAGE2_OUT / "MICRO_STAGE2_ROBUST.csv", index=False)
MICRO_STAGE2_BEST_BY_CANDIDATE.to_csv(STAGE2_OUT / "MICRO_STAGE2_BEST_BY_CANDIDATE.csv", index=False)
MICRO_STAGE2_TRADE_LOG.to_csv(STAGE2_OUT / "MICRO_STAGE2_TRADE_LOG.csv", index=False)

report = []
report.append("# Microchips Stage 2 Report\n")
report.append(f"Candidates tested: {len(STAGE2_MICRO_CANDIDATES)}\n")
report.append(f"Windows: {STAGE2_WINDOWS}\n")
report.append(f"Entry z: {STAGE2_ENTRY_ZS}\n")
report.append(f"Exit z: {STAGE2_EXIT_ZS}\n")
report.append(f"Max holds: {STAGE2_MAX_HOLDS}\n")
report.append(f"Runtime: {time.perf_counter() - stage2_start:.2f}s\n")

def add_report_table(title, df, n=80):
    report.append(f"\n## {title}\n")
    if df is None or len(df) == 0:
        report.append("No rows.\n")
        return
    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_report_table("MICRO_STAGE2_ROBUST", MICRO_STAGE2_ROBUST, 120)
add_report_table("MICRO_STAGE2_BEST_BY_CANDIDATE", MICRO_STAGE2_BEST_BY_CANDIDATE, 120)
add_report_table("Top MICRO_STAGE2_SUMMARY", MICRO_STAGE2_SUMMARY, 120)

report_path = STAGE2_OUT / "MICROCHIPS_stage2_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print("\nSaved:")
print((STAGE2_OUT / "MICROCHIPS_stage2_report.md").resolve())
print((STAGE2_OUT / "MICRO_STAGE2_ROBUST.csv").resolve())
print((STAGE2_OUT / "MICRO_STAGE2_BEST_BY_CANDIDATE.csv").resolve())
print((STAGE2_OUT / "MICRO_STAGE2_SUMMARY.csv").resolve())
print((STAGE2_OUT / "MICRO_STAGE2_TRADE_LOG.csv").resolve())
print(f"Runtime: {time.perf_counter() - stage2_start:.2f}s")

Stage 2 Microchip candidates: 22
mr_circle_oval_rect_2tri | type: zspread | mode: meanrev | q_trade: [5.0, -5.0, 0.0, 5.0, -10.0] | note: CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. Highest-priority Stage 1 candidate.
mr_2circle_oval_rect_2tri | type: zspread | mode: meanrev | q_trade: [10.0, -5.0, 0.0, 5.0, -10.0] | note: 2×CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE.
mr_oval_rect_triangle | type: zspread | mode: meanrev | q_trade: [0.0, 10.0, 0.0, -10.0, 10.0] | note: OVAL + TRIANGLE versus RECTANGLE.
mr_curved_triangle_vs_quads | type: zspread | mode: meanrev | q_trade: [10.0, 10.0, -10.0, -10.0, 10.0] | note: CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTANGLE.
mr_circle_square_rect_vs_oval_triangle | type: zspread | mode: meanrev | q_trade: [10.0, -10.0, 10.0, 10.0, -10.0] | note: CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIANGLE.
mr_circle_minus_oval | type: zspread | mode: meanrev | q_trade: [10.0, -10.0, 0.0, 0.0, 0.0] | note: Curved pair mean reversion.
mr_square_minus_rectan

,candidate,signal_type,family,note,mode,day,window,entry_z,exit_z,max_hold,...,total_mid_pnl,total_cost,avg_exec_pnl,median_exec_pnl,hit_rate,avg_hold,max_hold_seen,q_signal,q_trade,extra
0,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. H...,meanrev,2,500,1.5,fixed,250,...,632.5,7437.5,-200.147059,-37.5,0.500000,250.0,250,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",{}
1,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. H...,meanrev,3,500,1.5,fixed,250,...,-9480.0,6395.0,-512.096774,-230.0,0.419355,250.0,250,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",{}
2,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. H...,meanrev,4,500,1.5,fixed,250,...,6352.5,5712.5,21.333333,45.0,0.500000,250.0,250,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",{}
3,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. H...,meanrev,2,500,1.5,fixed,500,...,14970.0,3945.0,612.500000,1680.0,0.666667,500.0,500,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",{}
4,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. H...,meanrev,3,500,1.5,fixed,500,...,7592.5,3522.5,239.411765,1235.0,0.529412,500.0,500,"[1.0, -1.0, 0.0, 1.0, -2.0]","[5.0, -5.0, 0.0, 5.0, -10.0]",{}


,candidate,signal_type,family,mode,day,window,entry_z,exit_z,max_hold,entry_idx,exit_idx,entry_ts,exit_ts,side,hold,exec_pnl,mid_pnl,cost
0,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,meanrev,2,500,1.5,fixed,250,99,349,9900,34900,-1.0,250,3550.0,3772.5,222.5
1,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,meanrev,2,500,1.5,fixed,250,350,600,35000,60000,1.0,250,-2570.0,-2340.0,230.0
2,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,meanrev,2,500,1.5,fixed,250,601,851,60100,85100,1.0,250,3220.0,3445.0,225.0
3,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,meanrev,2,500,1.5,fixed,250,852,1102,85200,110200,-1.0,250,-415.0,-192.5,222.5
4,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,meanrev,2,500,1.5,fixed,250,1103,1353,110300,135300,-1.0,250,3040.0,3260.0,220.0



Top MICRO_STAGE2_ROBUST:


,candidate,signal_type,family,note,mode,window,entry_z,exit_z,max_hold,q_signal,...,positive_days,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_hold,max_hold_seen,pnl_per_trade,pnl_per_cost,one_day_dependency
4516,mr_regular_minus_elongated,zspread,regular_vs_elongated,Regular/symmetric shapes versus elongated shapes.,meanrev,500,2.0,fixed,500,"[1.0, -1.0, 1.0, -1.0, 0.0]",...,3,2013.791667,2215.000000,0.588889,0.500000,500.000000,500,2006.521739,5.619482,0.695125
3276,mr_circle_square_rect_vs_oval_triangle,zspread,shape_group_mr,CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIA...,meanrev,500,3.0,fixed,500,"[1.0, -1.0, 1.0, 1.0, -1.0]",...,3,3370.555556,3558.333333,0.759259,0.555556,500.000000,500,3564.583333,7.932313,0.560023
997,lead_circle_to_square_inverse_50,leadlag,leadlag,CIRCLE move predicts opposite SQUARE move.,inverse,500,2.0,fixed,1000,"[0.0, 0.0, 1.0, 0.0, 0.0]",...,3,3083.472222,3038.333333,0.541667,0.444444,1000.000000,1000,3062.307692,26.989831,0.456041
5616,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,2500,1.5,fixed,500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,1847.774725,1242.500000,0.584249,0.500000,500.000000,500,1859.634146,10.019054,0.528231
4122,mr_curved_triangle_vs_quads,zspread,shape_group_mr,CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTA...,meanrev,5000,2.5,0.25,1000,"[1.0, 1.0, -1.0, -1.0, 1.0]",...,3,6382.222222,5726.666667,0.933333,0.800000,608.250000,1000,6298.333333,14.074488,0.436094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4247,mr_oval_rect_triangle,zspread,integer_shape_mr,OVAL + TRIANGLE versus RECTANGLE.,meanrev,1000,1.5,0.5,1000,"[0.0, 1.0, 0.0, -1.0, 1.0]",...,3,535.775058,1090.000000,0.824426,0.772727,200.302031,848,561.842105,2.333971,0.588525
5558,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,1000,2.0,fixed,1500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,2844.666667,4065.000000,0.733333,0.600000,1500.000000,1500,2844.666667,15.404332,0.596789
4135,mr_curved_triangle_vs_quads,zspread,shape_group_mr,CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTA...,meanrev,5000,2.5,fixed,250,"[1.0, 1.0, -1.0, -1.0, 1.0]",...,3,1823.108466,2908.333333,0.637566,0.500000,250.000000,250,2132.500000,4.841090,0.501524
5719,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,5000,2.0,fixed,2500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,7105.000000,7105.000000,0.833333,0.500000,2500.000000,2500,7105.000000,38.147651,0.555829



MICRO_STAGE2_BEST_BY_CANDIDATE:


,candidate,signal_type,family,note,mode,window,entry_z,exit_z,max_hold,q_signal,...,positive_days,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_hold,max_hold_seen,pnl_per_trade,pnl_per_cost,one_day_dependency
0,mr_regular_minus_elongated,zspread,regular_vs_elongated,Regular/symmetric shapes versus elongated shapes.,meanrev,500,2.0,fixed,500,"[1.0, -1.0, 1.0, -1.0, 0.0]",...,3,2013.791667,2215.000000,0.588889,0.500000,500.000000,500,2006.521739,5.619482,0.695125
1,mr_circle_square_rect_vs_oval_triangle,zspread,shape_group_mr,CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIA...,meanrev,500,3.0,fixed,500,"[1.0, -1.0, 1.0, 1.0, -1.0]",...,3,3370.555556,3558.333333,0.759259,0.555556,500.000000,500,3564.583333,7.932313,0.560023
2,lead_circle_to_square_inverse_50,leadlag,leadlag,CIRCLE move predicts opposite SQUARE move.,inverse,500,2.0,fixed,1000,"[0.0, 0.0, 1.0, 0.0, 0.0]",...,3,3083.472222,3038.333333,0.541667,0.444444,1000.000000,1000,3062.307692,26.989831,0.456041
3,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,2500,1.5,fixed,500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,1847.774725,1242.500000,0.584249,0.500000,500.000000,500,1859.634146,10.019054,0.528231
4,mr_curved_triangle_vs_quads,zspread,shape_group_mr,CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTA...,meanrev,5000,2.5,0.25,1000,"[1.0, 1.0, -1.0, -1.0, 1.0]",...,3,6382.222222,5726.666667,0.933333,0.800000,608.250000,1000,6298.333333,14.074488,0.436094
5,mr_oval_rect_triangle,zspread,integer_shape_mr,OVAL + TRIANGLE versus RECTANGLE.,meanrev,2500,3.0,fixed,1000,"[0.0, 1.0, 0.0, -1.0, 1.0]",...,3,14223.333333,14223.333333,1.000000,1.000000,1000.000000,1000,13924.000000,55.919679,0.539931
6,mr_circle_oval_rect_2tri,zspread,integer_shape_mr,CIRCLE + RECTANGLE versus OVAL + 2×TRIANGLE. H...,meanrev,2500,2.5,fixed,1000,"[1.0, -1.0, 0.0, 1.0, -2.0]",...,3,7028.944444,5827.500000,0.833333,0.500000,1000.000000,1000,6911.000000,32.370023,0.467877
7,own_square_momentum_past100,own_move,own_momentum_reversal,SQUARE own momentum after strong 100-tick move.,follow,1000,1.5,fixed,2500,"[0.0, 0.0, 1.0, 0.0, 0.0]",...,3,7426.666667,8220.000000,0.888889,0.666667,2500.000000,2500,7426.666667,63.355450,0.528576
8,lead_oval_to_triangle_follow_500,leadlag,leadlag,OVAL move predicts TRIANGLE follow.,follow,1000,1.5,fixed,250,"[0.0, 0.0, 0.0, 0.0, 1.0]",...,3,762.322261,961.666667,0.667444,0.653846,250.000000,250,779.583333,9.075182,0.538393
9,mr_circle_minus_square,zspread,old_pair,Earlier CIRCLE/SQUARE pair idea.,meanrev,500,2.5,0.25,500,"[1.0, 0.0, -1.0, 0.0, 0.0]",...,3,882.216539,1776.666667,0.810742,0.782609,189.805627,500,856.825397,4.301195,0.361801



Top MICRO_STAGE2_SUMMARY:


,candidate,signal_type,family,note,mode,window,entry_z,exit_z,max_hold,q_signal,...,positive_days,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_hold,max_hold_seen,pnl_per_trade,pnl_per_cost,one_day_dependency
4516,mr_regular_minus_elongated,zspread,regular_vs_elongated,Regular/symmetric shapes versus elongated shapes.,meanrev,500,2.0,fixed,500,"[1.0, -1.0, 1.0, -1.0, 0.0]",...,3,2013.791667,2215.000000,0.588889,0.500000,500.000000,500,2006.521739,5.619482,0.695125
3276,mr_circle_square_rect_vs_oval_triangle,zspread,shape_group_mr,CIRCLE + SQUARE + RECTANGLE versus OVAL + TRIA...,meanrev,500,3.0,fixed,500,"[1.0, -1.0, 1.0, 1.0, -1.0]",...,3,3370.555556,3558.333333,0.759259,0.555556,500.000000,500,3564.583333,7.932313,0.560023
997,lead_circle_to_square_inverse_50,leadlag,leadlag,CIRCLE move predicts opposite SQUARE move.,inverse,500,2.0,fixed,1000,"[0.0, 0.0, 1.0, 0.0, 0.0]",...,3,3083.472222,3038.333333,0.541667,0.444444,1000.000000,1000,3062.307692,26.989831,0.456041
5616,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,2500,1.5,fixed,500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,1847.774725,1242.500000,0.584249,0.500000,500.000000,500,1859.634146,10.019054,0.528231
4122,mr_curved_triangle_vs_quads,zspread,shape_group_mr,CIRCLE + OVAL + TRIANGLE versus SQUARE + RECTA...,meanrev,5000,2.5,0.25,1000,"[1.0, 1.0, -1.0, -1.0, 1.0]",...,3,6382.222222,5726.666667,0.933333,0.800000,608.250000,1000,6298.333333,14.074488,0.436094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3799,mr_curved_minus_quads,zspread,curved_vs_quadrilateral,Curved chips versus quadrilateral chips.,meanrev,5000,2.0,fixed,2500,"[1.0, 1.0, -1.0, -1.0, 0.0]",...,3,4879.444444,2058.333333,0.500000,0.333333,2500.000000,2500,5342.500000,15.457505,0.643425
5683,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,5000,1.5,0.25,1500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,2022.777778,2660.833333,0.833333,0.666667,624.236111,1500,2136.500000,11.394667,0.591622
4247,mr_oval_rect_triangle,zspread,integer_shape_mr,OVAL + TRIANGLE versus RECTANGLE.,meanrev,1000,1.5,0.5,1000,"[0.0, 1.0, 0.0, -1.0, 1.0]",...,3,535.775058,1090.000000,0.824426,0.772727,200.302031,848,561.842105,2.333971,0.588525
5558,mr_triangle_minus_quads,zspread,triangle_vs_quads,Triangle versus average quadrilateral.,meanrev,1000,2.0,fixed,1500,"[0.0, 0.0, -0.5, -0.5, 1.0]",...,3,2844.666667,4065.000000,0.733333,0.600000,1500.000000,1500,2844.666667,15.404332,0.596789



Saved:
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage2/MICROCHIPS_stage2_report.md
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage2/MICRO_STAGE2_ROBUST.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage2/MICRO_STAGE2_BEST_BY_CANDIDATE.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage2/MICRO_STAGE2_SUMMARY.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_microchips_stage2/MICRO_STAGE2_TRADE_LOG.csv
Runtime: 14.06s


In [13]:
# ════════════════════════════════════════════════════════════════════════════
# MICROCHIPS — NONLINEAR SHAPE RELATIONSHIP DISCOVERY
# Pure relationship diagnostics, not full strategy backtest
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path
import time
import numpy as np
import pandas as pd

t0 = time.time()

OUT_DIR = Path("analysis_outputs/microchips_nonlinear")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MICRO_PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

P = {p: i for i, p in enumerate(MICRO_PRODUCTS)}

# -------------------------------------------------------------------------
# 1. Build aligned wide matrices once
# -------------------------------------------------------------------------

mid_wide = (
    prices[prices["product"].isin(MICRO_PRODUCTS)]
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="mid_price")
    .sort_index()
)

mid_wide = mid_wide[MICRO_PRODUCTS].dropna()

DAYS_USED = sorted(mid_wide.index.get_level_values(0).unique())

DAY_DATA = {}

for day in DAYS_USED:
    day_mid = mid_wide.loc[day, MICRO_PRODUCTS].copy()
    DAY_DATA[day] = {
        "X": day_mid.to_numpy(float),
        "ts": day_mid.index.to_numpy(),
    }

print("Days:", DAYS_USED)
print("Products:", MICRO_PRODUCTS)
print("Aligned shape:", mid_wide.shape)
for day, d in DAY_DATA.items():
    print(f"Day {day}: X={d['X'].shape}")

# -------------------------------------------------------------------------
# 2. Helper functions
# -------------------------------------------------------------------------

def wvec(weights_dict):
    q = np.zeros(len(MICRO_PRODUCTS), dtype=float)
    for product, weight in weights_dict.items():
        q[P[product]] = weight
    return q

def basket(X, q):
    return X @ np.asarray(q, dtype=float)

def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 5:
        return np.nan
    aa = a[mask]
    bb = b[mask]
    if np.std(aa) < 1e-12 or np.std(bb) < 1e-12:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

def train_test_split_idx(n, train_frac=0.65):
    split = int(n * train_frac)
    split = max(500, min(split, n - 500))
    return split

def design_matrix(x, model, train_idx):
    x = np.asarray(x, dtype=float)
    eps = 1e-9

    if model == "linear":
        x0 = x
        mu = np.nanmean(x0[:train_idx])
        sd = np.nanstd(x0[:train_idx])
        sd = sd if sd > eps else 1.0
        xs = (x0 - mu) / sd
        return np.column_stack([np.ones_like(xs), xs])

    if model == "quadratic":
        x0 = x
        mu = np.nanmean(x0[:train_idx])
        sd = np.nanstd(x0[:train_idx])
        sd = sd if sd > eps else 1.0
        xs = (x0 - mu) / sd
        return np.column_stack([np.ones_like(xs), xs, xs**2])

    if model == "cubic":
        x0 = x
        mu = np.nanmean(x0[:train_idx])
        sd = np.nanstd(x0[:train_idx])
        sd = sd if sd > eps else 1.0
        xs = (x0 - mu) / sd
        return np.column_stack([np.ones_like(xs), xs, xs**2, xs**3])

    if model == "loglinear":
        lx = np.log(np.maximum(x, eps))
        mu = np.nanmean(lx[:train_idx])
        sd = np.nanstd(lx[:train_idx])
        sd = sd if sd > eps else 1.0
        xs = (lx - mu) / sd
        return np.column_stack([np.ones_like(xs), xs])

    raise ValueError(f"Unknown model: {model}")

def fit_residual(x, y, model, train_idx):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    eps = 1e-9

    if model == "spread_fixed":
        pred = x
        resid = y - pred
        return resid, pred, np.nan

    if model == "ratio":
        ratio = y / np.maximum(x, eps)
        ratio_mu = np.nanmean(ratio[:train_idx])
        pred = ratio_mu * x
        resid = ratio - ratio_mu
        # R2 on actual y fit
        test_y = y[train_idx:]
        test_pred = pred[train_idx:]
        sse = np.nansum((test_y - test_pred) ** 2)
        sst = np.nansum((test_y - np.nanmean(test_y)) ** 2)
        r2 = 1 - sse / sst if sst > eps else np.nan
        return resid, pred, r2

    if model == "logratio":
        ly = np.log(np.maximum(y, eps))
        lx = np.log(np.maximum(x, eps))
        resid = ly - lx
        pred = x
        return resid, pred, np.nan

    if model == "loglinear":
        ly = np.log(np.maximum(y, eps))
        Xd = design_matrix(x, model, train_idx)
        beta = np.linalg.lstsq(Xd[:train_idx], ly[:train_idx], rcond=None)[0]
        pred_log = Xd @ beta
        resid = ly - pred_log
        test_y = ly[train_idx:]
        test_pred = pred_log[train_idx:]
        sse = np.nansum((test_y - test_pred) ** 2)
        sst = np.nansum((test_y - np.nanmean(test_y)) ** 2)
        r2 = 1 - sse / sst if sst > eps else np.nan
        return resid, np.exp(pred_log), r2

    Xd = design_matrix(x, model, train_idx)
    beta = np.linalg.lstsq(Xd[:train_idx], y[:train_idx], rcond=None)[0]
    pred = Xd @ beta
    resid = y - pred

    test_y = y[train_idx:]
    test_pred = pred[train_idx:]
    sse = np.nansum((test_y - test_pred) ** 2)
    sst = np.nansum((test_y - np.nanmean(test_y)) ** 2)
    r2 = 1 - sse / sst if sst > eps else np.nan

    return resid, pred, r2

def residual_mr_stats(resid, train_idx, horizon, threshold):
    resid = np.asarray(resid, dtype=float)
    n = len(resid)

    if n <= train_idx + horizon + 10:
        return None

    train = resid[:train_idx]
    mu = np.nanmean(train)
    sd = np.nanstd(train)
    if not np.isfinite(sd) or sd < 1e-9:
        return None

    z = (resid - mu) / sd

    start = train_idx
    end = n - horizon

    z0 = z[start:end]
    r0 = resid[start:end]
    r1 = resid[start + horizon:end + horizon]
    delta = r1 - r0

    valid = np.isfinite(z0) & np.isfinite(delta)
    if valid.sum() < 20:
        return None

    z0 = z0[valid]
    delta = delta[valid]

    signal = np.abs(z0) >= threshold
    event_count = int(signal.sum())

    if event_count == 0:
        return {
            "event_count": 0,
            "hit_rate": np.nan,
            "avg_reversion_edge": np.nan,
            "median_reversion_edge": np.nan,
            "mr_corr": safe_corr(z0, delta),
            "avg_abs_z": np.nan,
        }

    z_sig = z0[signal]
    d_sig = delta[signal]

    # A good mean-reverting residual has:
    # if z positive, future residual should fall => delta negative
    # if z negative, future residual should rise => delta positive
    hit = np.sign(d_sig) == -np.sign(z_sig)
    edge = -np.sign(z_sig) * d_sig

    return {
        "event_count": event_count,
        "hit_rate": float(np.mean(hit)),
        "avg_reversion_edge": float(np.mean(edge)),
        "median_reversion_edge": float(np.median(edge)),
        "mr_corr": safe_corr(z0, delta),
        "avg_abs_z": float(np.mean(np.abs(z_sig))),
    }

# -------------------------------------------------------------------------
# 3. Relationship candidates
# -------------------------------------------------------------------------

RELATIONSHIPS = [
    {
        "name": "oval_plus_triangle_vs_rectangle",
        "note": "OVAL + TRIANGLE explained by RECTANGLE",
        "y": wvec({"MICROCHIP_OVAL": 1, "MICROCHIP_TRIANGLE": 1}),
        "x": wvec({"MICROCHIP_RECTANGLE": 1}),
    },
    {
        "name": "rectangle_vs_oval_plus_triangle",
        "note": "RECTANGLE explained by OVAL + TRIANGLE",
        "y": wvec({"MICROCHIP_RECTANGLE": 1}),
        "x": wvec({"MICROCHIP_OVAL": 1, "MICROCHIP_TRIANGLE": 1}),
    },
    {
        "name": "curved_triangle_vs_quads",
        "note": "CIRCLE + OVAL + TRIANGLE explained by SQUARE + RECTANGLE",
        "y": wvec({"MICROCHIP_CIRCLE": 1, "MICROCHIP_OVAL": 1, "MICROCHIP_TRIANGLE": 1}),
        "x": wvec({"MICROCHIP_SQUARE": 1, "MICROCHIP_RECTANGLE": 1}),
    },
    {
        "name": "quads_vs_curved_triangle",
        "note": "SQUARE + RECTANGLE explained by CIRCLE + OVAL + TRIANGLE",
        "y": wvec({"MICROCHIP_SQUARE": 1, "MICROCHIP_RECTANGLE": 1}),
        "x": wvec({"MICROCHIP_CIRCLE": 1, "MICROCHIP_OVAL": 1, "MICROCHIP_TRIANGLE": 1}),
    },
    {
        "name": "regular_vs_elongated",
        "note": "CIRCLE + SQUARE explained by OVAL + RECTANGLE",
        "y": wvec({"MICROCHIP_CIRCLE": 1, "MICROCHIP_SQUARE": 1}),
        "x": wvec({"MICROCHIP_OVAL": 1, "MICROCHIP_RECTANGLE": 1}),
    },
    {
        "name": "elongated_vs_regular",
        "note": "OVAL + RECTANGLE explained by CIRCLE + SQUARE",
        "y": wvec({"MICROCHIP_OVAL": 1, "MICROCHIP_RECTANGLE": 1}),
        "x": wvec({"MICROCHIP_CIRCLE": 1, "MICROCHIP_SQUARE": 1}),
    },
    {
        "name": "triangle_vs_quads_avg",
        "note": "TRIANGLE explained by average of SQUARE + RECTANGLE",
        "y": wvec({"MICROCHIP_TRIANGLE": 1}),
        "x": wvec({"MICROCHIP_SQUARE": 0.5, "MICROCHIP_RECTANGLE": 0.5}),
    },
    {
        "name": "quads_avg_vs_triangle",
        "note": "Average of SQUARE + RECTANGLE explained by TRIANGLE",
        "y": wvec({"MICROCHIP_SQUARE": 0.5, "MICROCHIP_RECTANGLE": 0.5}),
        "x": wvec({"MICROCHIP_TRIANGLE": 1}),
    },
    {
        "name": "circle_vs_oval",
        "note": "CIRCLE explained by OVAL",
        "y": wvec({"MICROCHIP_CIRCLE": 1}),
        "x": wvec({"MICROCHIP_OVAL": 1}),
    },
    {
        "name": "square_vs_rectangle",
        "note": "SQUARE explained by RECTANGLE",
        "y": wvec({"MICROCHIP_SQUARE": 1}),
        "x": wvec({"MICROCHIP_RECTANGLE": 1}),
    },
    {
        "name": "square_vs_circle",
        "note": "SQUARE explained by CIRCLE",
        "y": wvec({"MICROCHIP_SQUARE": 1}),
        "x": wvec({"MICROCHIP_CIRCLE": 1}),
    },
    {
        "name": "square_vs_triangle",
        "note": "SQUARE explained by TRIANGLE",
        "y": wvec({"MICROCHIP_SQUARE": 1}),
        "x": wvec({"MICROCHIP_TRIANGLE": 1}),
    },
]

MODELS = [
    "spread_fixed",
    "ratio",
    "logratio",
    "linear",
    "quadratic",
    "cubic",
    "loglinear",
]

HORIZONS = [100, 250, 500, 1000, 1500, 2500]
THRESHOLDS = [1.5, 2.0, 2.5, 3.0]

# -------------------------------------------------------------------------
# 4. Run nonlinear relationship scan
# -------------------------------------------------------------------------

rows = []

total_jobs = len(RELATIONSHIPS) * len(MODELS)
job = 0

for rel in RELATIONSHIPS:
    for model in MODELS:
        job += 1
        if job == 1 or job % 10 == 0:
            print(f"[{time.time()-t0:7.2f}s] relationship/model {job}/{total_jobs}: {rel['name']} | {model}")

        for day, d in DAY_DATA.items():
            X = d["X"]
            n = len(X)
            train_idx = train_test_split_idx(n, train_frac=0.65)

            y = basket(X, rel["y"])
            x = basket(X, rel["x"])

            try:
                resid, pred, r2 = fit_residual(x, y, model, train_idx)
            except Exception as e:
                continue

            train_resid_std = float(np.nanstd(resid[:train_idx]))
            test_resid_std = float(np.nanstd(resid[train_idx:]))

            for h in HORIZONS:
                for th in THRESHOLDS:
                    stats = residual_mr_stats(resid, train_idx, h, th)
                    if stats is None:
                        continue

                    rows.append({
                        "relationship": rel["name"],
                        "note": rel["note"],
                        "model": model,
                        "day": day,
                        "horizon": h,
                        "threshold": th,
                        "train_idx": train_idx,
                        "train_resid_std": train_resid_std,
                        "test_resid_std": test_resid_std,
                        "test_r2": r2,
                        **stats,
                    })

NONLINEAR_REL_DAY = pd.DataFrame(rows)

print("NONLINEAR_REL_DAY:", NONLINEAR_REL_DAY.shape)

# -------------------------------------------------------------------------
# 5. Aggregate relationship scan
# -------------------------------------------------------------------------

gcols = ["relationship", "note", "model", "horizon", "threshold"]

NONLINEAR_REL_SUMMARY = (
    NONLINEAR_REL_DAY
    .groupby(gcols, dropna=False)
    .agg(
        rows=("day", "size"),
        active_days=("day", "nunique"),
        total_events=("event_count", "sum"),
        mean_events=("event_count", "mean"),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        mean_reversion_edge=("avg_reversion_edge", "mean"),
        min_reversion_edge=("avg_reversion_edge", "min"),
        median_reversion_edge=("median_reversion_edge", "mean"),
        mean_mr_corr=("mr_corr", "mean"),
        max_mr_corr=("mr_corr", "max"),
        mean_test_r2=("test_r2", "mean"),
        mean_test_resid_std=("test_resid_std", "mean"),
        mean_abs_z=("avg_abs_z", "mean"),
    )
    .reset_index()
)

NONLINEAR_REL_SUMMARY["positive_edge_days"] = (
    NONLINEAR_REL_DAY.assign(pos=lambda d: d["avg_reversion_edge"] > 0)
    .groupby(gcols)["pos"]
    .sum()
    .values
)

# Score favours:
# - all 3 active days
# - positive mean-reversion edge all days
# - high hit rate
# - negative residual MR correlation
# - enough events
NONLINEAR_REL_SUMMARY["nonlinear_score"] = (
    10000 * (NONLINEAR_REL_SUMMARY["active_days"] == 3).astype(int)
    + 3000 * (NONLINEAR_REL_SUMMARY["positive_edge_days"] == 3).astype(int)
    + 2000 * (NONLINEAR_REL_SUMMARY["min_reversion_edge"] > 0).astype(int)
    + 2000 * (NONLINEAR_REL_SUMMARY["min_hit_rate"] > 0.55).astype(int)
    + 1000 * (NONLINEAR_REL_SUMMARY["mean_hit_rate"].fillna(0) - 0.5)
    + 1000 * (-NONLINEAR_REL_SUMMARY["mean_mr_corr"].fillna(0))
    + 5 * np.log1p(NONLINEAR_REL_SUMMARY["total_events"].fillna(0))
    + 50 * NONLINEAR_REL_SUMMARY["mean_test_r2"].fillna(0)
)

NONLINEAR_REL_SUMMARY = NONLINEAR_REL_SUMMARY.sort_values(
    ["nonlinear_score", "mean_reversion_edge", "mean_hit_rate"],
    ascending=[False, False, False],
).reset_index(drop=True)

NONLINEAR_REL_BEST_BY_RELATION = (
    NONLINEAR_REL_SUMMARY
    .sort_values(["relationship", "nonlinear_score"], ascending=[True, False])
    .groupby("relationship", as_index=False)
    .head(1)
    .sort_values("nonlinear_score", ascending=False)
    .reset_index(drop=True)
)

# Compare nonlinear models against linear baseline for same relationship/horizon/threshold.
base = NONLINEAR_REL_SUMMARY[NONLINEAR_REL_SUMMARY["model"] == "linear"][
    ["relationship", "horizon", "threshold", "nonlinear_score", "mean_hit_rate", "mean_reversion_edge", "mean_mr_corr"]
].rename(columns={
    "nonlinear_score": "linear_score",
    "mean_hit_rate": "linear_hit_rate",
    "mean_reversion_edge": "linear_reversion_edge",
    "mean_mr_corr": "linear_mr_corr",
})

NONLINEAR_VS_LINEAR = (
    NONLINEAR_REL_SUMMARY[NONLINEAR_REL_SUMMARY["model"].isin(["quadratic", "cubic", "loglinear", "ratio", "logratio"])]
    .merge(base, on=["relationship", "horizon", "threshold"], how="left")
)

NONLINEAR_VS_LINEAR["score_gain_vs_linear"] = NONLINEAR_VS_LINEAR["nonlinear_score"] - NONLINEAR_VS_LINEAR["linear_score"]
NONLINEAR_VS_LINEAR["hit_gain_vs_linear"] = NONLINEAR_VS_LINEAR["mean_hit_rate"] - NONLINEAR_VS_LINEAR["linear_hit_rate"]
NONLINEAR_VS_LINEAR["edge_gain_vs_linear"] = NONLINEAR_VS_LINEAR["mean_reversion_edge"] - NONLINEAR_VS_LINEAR["linear_reversion_edge"]

NONLINEAR_VS_LINEAR = NONLINEAR_VS_LINEAR.sort_values(
    ["score_gain_vs_linear", "nonlinear_score"],
    ascending=[False, False],
).reset_index(drop=True)

# -------------------------------------------------------------------------
# 6. Static shape-factor residual test
# This checks if prices fit latent shape factors better than raw pair baskets.
# -------------------------------------------------------------------------

shape_features = pd.DataFrame(index=MICRO_PRODUCTS)

shape_features["intercept"] = 1.0
shape_features["curved"] = [1, 1, 0, 0, 0]
shape_features["angular"] = [0, 0, 1, 1, 1]
shape_features["regular"] = [1, 0, 1, 0, 0]
shape_features["elongated"] = [0, 1, 0, 1, 0]
shape_features["quadrilateral"] = [0, 0, 1, 1, 0]
shape_features["triangle"] = [0, 0, 0, 0, 1]
shape_features["side_count"] = [0, 0, 4, 4, 3]
shape_features["side_count_sq"] = shape_features["side_count"] ** 2

FACTOR_MODELS = {
    "regular_elongated_triangle": ["intercept", "regular", "elongated", "triangle"],
    "curved_quads_triangle": ["intercept", "curved", "quadrilateral", "triangle"],
    "side_count_quadratic": ["intercept", "side_count", "side_count_sq"],
    "curved_elongated_triangle": ["intercept", "curved", "elongated", "triangle"],
    "regular_elongated_quadrilateral": ["intercept", "regular", "elongated", "quadrilateral"],
}

factor_rows = []

for factor_name, cols in FACTOR_MODELS.items():
    F = shape_features[cols].to_numpy(float)
    rank = np.linalg.matrix_rank(F)

    # Avoid exact 5D fit because then residuals are meaningless.
    if rank >= len(MICRO_PRODUCTS):
        print("Skipping exact-rank factor model:", factor_name)
        continue

    # Projection matrix across products.
    Proj = F @ np.linalg.pinv(F)

    print(f"Factor model: {factor_name}, cols={cols}, rank={rank}")

    for day, d in DAY_DATA.items():
        X = d["X"]
        n = len(X)
        train_idx = train_test_split_idx(n, train_frac=0.65)

        pred = X @ Proj.T
        resid_matrix = X - pred

        for j, product in enumerate(MICRO_PRODUCTS):
            resid = resid_matrix[:, j]

            for h in HORIZONS:
                for th in THRESHOLDS:
                    stats = residual_mr_stats(resid, train_idx, h, th)
                    if stats is None:
                        continue

                    factor_rows.append({
                        "factor_model": factor_name,
                        "factor_cols": ",".join(cols),
                        "factor_rank": rank,
                        "product_residual": product,
                        "day": day,
                        "horizon": h,
                        "threshold": th,
                        "train_resid_std": float(np.nanstd(resid[:train_idx])),
                        "test_resid_std": float(np.nanstd(resid[train_idx:])),
                        **stats,
                    })

SHAPE_FACTOR_DAY = pd.DataFrame(factor_rows)

fgcols = ["factor_model", "factor_cols", "product_residual", "horizon", "threshold"]

SHAPE_FACTOR_SUMMARY = (
    SHAPE_FACTOR_DAY
    .groupby(fgcols, dropna=False)
    .agg(
        rows=("day", "size"),
        active_days=("day", "nunique"),
        total_events=("event_count", "sum"),
        mean_events=("event_count", "mean"),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        mean_reversion_edge=("avg_reversion_edge", "mean"),
        min_reversion_edge=("avg_reversion_edge", "min"),
        mean_mr_corr=("mr_corr", "mean"),
        max_mr_corr=("mr_corr", "max"),
        mean_test_resid_std=("test_resid_std", "mean"),
        mean_abs_z=("avg_abs_z", "mean"),
    )
    .reset_index()
)

SHAPE_FACTOR_SUMMARY["positive_edge_days"] = (
    SHAPE_FACTOR_DAY.assign(pos=lambda d: d["avg_reversion_edge"] > 0)
    .groupby(fgcols)["pos"]
    .sum()
    .values
)

SHAPE_FACTOR_SUMMARY["shape_factor_score"] = (
    10000 * (SHAPE_FACTOR_SUMMARY["active_days"] == 3).astype(int)
    + 3000 * (SHAPE_FACTOR_SUMMARY["positive_edge_days"] == 3).astype(int)
    + 2000 * (SHAPE_FACTOR_SUMMARY["min_reversion_edge"] > 0).astype(int)
    + 2000 * (SHAPE_FACTOR_SUMMARY["min_hit_rate"] > 0.55).astype(int)
    + 1000 * (SHAPE_FACTOR_SUMMARY["mean_hit_rate"].fillna(0) - 0.5)
    + 1000 * (-SHAPE_FACTOR_SUMMARY["mean_mr_corr"].fillna(0))
    + 5 * np.log1p(SHAPE_FACTOR_SUMMARY["total_events"].fillna(0))
)

SHAPE_FACTOR_SUMMARY = SHAPE_FACTOR_SUMMARY.sort_values(
    ["shape_factor_score", "mean_reversion_edge", "mean_hit_rate"],
    ascending=[False, False, False],
).reset_index(drop=True)

# -------------------------------------------------------------------------
# 7. Save outputs
# -------------------------------------------------------------------------

NONLINEAR_REL_DAY.to_csv(OUT_DIR / "NONLINEAR_REL_DAY.csv", index=False)
NONLINEAR_REL_SUMMARY.to_csv(OUT_DIR / "NONLINEAR_REL_SUMMARY.csv", index=False)
NONLINEAR_REL_BEST_BY_RELATION.to_csv(OUT_DIR / "NONLINEAR_REL_BEST_BY_RELATION.csv", index=False)
NONLINEAR_VS_LINEAR.to_csv(OUT_DIR / "NONLINEAR_VS_LINEAR.csv", index=False)

SHAPE_FACTOR_DAY.to_csv(OUT_DIR / "SHAPE_FACTOR_DAY.csv", index=False)
SHAPE_FACTOR_SUMMARY.to_csv(OUT_DIR / "SHAPE_FACTOR_SUMMARY.csv", index=False)

print()
print("Saved CSV outputs to:", OUT_DIR)
print("Runtime:", round(time.time() - t0, 2), "seconds")

# -------------------------------------------------------------------------
# 8. Display useful summaries
# -------------------------------------------------------------------------

show_cols = [
    "relationship", "model", "horizon", "threshold",
    "active_days", "positive_edge_days", "total_events",
    "mean_hit_rate", "min_hit_rate",
    "mean_reversion_edge", "min_reversion_edge",
    "mean_mr_corr", "mean_test_r2",
    "nonlinear_score",
]

print("\nTop NONLINEAR_REL_SUMMARY:")
display(NONLINEAR_REL_SUMMARY[show_cols].head(50))

print("\nBest nonlinear model per relationship:")
display(NONLINEAR_REL_BEST_BY_RELATION[show_cols].head(30))

print("\nTop nonlinear improvements vs linear:")
display(
    NONLINEAR_VS_LINEAR[
        [
            "relationship", "model", "horizon", "threshold",
            "score_gain_vs_linear", "hit_gain_vs_linear", "edge_gain_vs_linear",
            "nonlinear_score", "linear_score",
            "mean_hit_rate", "linear_hit_rate",
            "mean_reversion_edge", "linear_reversion_edge",
        ]
    ].head(50)
)

print("\nTop SHAPE_FACTOR_SUMMARY:")
display(
    SHAPE_FACTOR_SUMMARY[
        [
            "factor_model", "product_residual", "horizon", "threshold",
            "active_days", "positive_edge_days", "total_events",
            "mean_hit_rate", "min_hit_rate",
            "mean_reversion_edge", "min_reversion_edge",
            "mean_mr_corr", "shape_factor_score",
        ]
    ].head(50)
)

Days: [2, 3, 4]
Products: ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE', 'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']
Aligned shape: (30000, 5)
Day 2: X=(10000, 5)
Day 3: X=(10000, 5)
Day 4: X=(10000, 5)
[   0.07s] relationship/model 1/84: oval_plus_triangle_vs_rectangle | spread_fixed
[   0.15s] relationship/model 10/84: rectangle_vs_oval_plus_triangle | logratio
[   0.24s] relationship/model 20/84: curved_triangle_vs_quads | cubic
[   0.32s] relationship/model 30/84: regular_vs_elongated | ratio
[   0.41s] relationship/model 40/84: elongated_vs_regular | quadratic
[   0.50s] relationship/model 50/84: quads_avg_vs_triangle | spread_fixed
[   0.58s] relationship/model 60/84: circle_vs_oval | linear
[   0.67s] relationship/model 70/84: square_vs_rectangle | loglinear
[   0.76s] relationship/model 80/84: square_vs_triangle | logratio
NONLINEAR_REL_DAY: (6048, 16)
Factor model: regular_elongated_triangle, cols=['intercept', 'regular', 'elongated', 'triangle'], rank=3
Factor mo

,relationship,model,horizon,threshold,active_days,positive_edge_days,total_events,mean_hit_rate,min_hit_rate,mean_reversion_edge,min_reversion_edge,mean_mr_corr,mean_test_r2,nonlinear_score
0,circle_vs_oval,logratio,1500,2.5,3,3,1028,0.902192,0.706577,0.046257,0.006423,-0.842386,NaN,18279.260282
1,curved_triangle_vs_quads,logratio,1000,2.0,3,3,1255,0.940503,0.821508,0.029797,0.020838,-0.786174,NaN,18262.355233
2,quads_vs_curved_triangle,logratio,1000,2.0,3,3,1255,0.940503,0.821508,0.029797,0.020838,-0.786174,NaN,18262.355233
3,circle_vs_oval,spread_fixed,1500,2.5,3,3,1238,0.855626,0.566879,420.786702,48.435032,-0.848489,NaN,18239.725606
4,curved_triangle_vs_quads,spread_fixed,1000,2.0,3,3,1133,0.931395,0.794186,677.072967,414.072674,-0.769420,NaN,18235.983297
5,quads_vs_curved_triangle,spread_fixed,1000,2.0,3,3,1133,0.931395,0.794186,677.072967,414.072674,-0.769420,NaN,18235.983297
6,circle_vs_oval,logratio,1000,2.5,3,3,1047,0.925381,0.776144,0.043490,0.012895,-0.763826,NaN,18223.980175
7,circle_vs_oval,spread_fixed,1000,2.5,3,3,1461,0.890873,0.672619,387.174562,59.077381,-0.767842,NaN,18195.152556
8,square_vs_triangle,logratio,1000,1.5,3,3,2345,0.849950,0.709270,0.049560,0.033597,-0.778045,NaN,18166.797060
9,regular_vs_elongated,linear,1500,2.0,3,3,1264,0.986767,0.960302,1283.862763,820.307854,-0.888044,-5.703507,18125.350758



Best nonlinear model per relationship:


,relationship,model,horizon,threshold,active_days,positive_edge_days,total_events,mean_hit_rate,min_hit_rate,mean_reversion_edge,min_reversion_edge,mean_mr_corr,mean_test_r2,nonlinear_score
0,circle_vs_oval,logratio,1500,2.5,3,3,1028,0.902192,0.706577,0.046257,0.006423,-0.842386,NaN,18279.260282
1,curved_triangle_vs_quads,logratio,1000,2.0,3,3,1255,0.940503,0.821508,0.029797,0.020838,-0.786174,NaN,18262.355233
2,quads_vs_curved_triangle,logratio,1000,2.0,3,3,1255,0.940503,0.821508,0.029797,0.020838,-0.786174,NaN,18262.355233
3,square_vs_triangle,logratio,1000,1.5,3,3,2345,0.849950,0.709270,0.049560,0.033597,-0.778045,NaN,18166.797060
4,regular_vs_elongated,linear,1500,2.0,3,3,1264,0.986767,0.960302,1283.862763,820.307854,-0.888044,-5.703507,18125.350758
5,square_vs_rectangle,cubic,1500,2.0,3,3,719,0.999491,0.998473,444.232121,232.488855,-0.759595,-5.000275,18041.968773
6,square_vs_circle,linear,1500,2.0,3,3,1080,0.952548,0.920145,670.712114,142.277034,-0.807583,-10.126681,17788.725136
7,quads_avg_vs_triangle,cubic,2500,2.0,3,3,1080,0.987094,0.961283,794.148042,385.479282,-0.729598,-10.374830,17732.878882
8,elongated_vs_regular,quadratic,1500,2.5,3,3,2577,0.869286,0.802788,391.414140,190.379588,-0.807281,-12.443519,17593.665625
9,rectangle_vs_oval_plus_triangle,linear,1500,2.0,3,3,1954,0.931296,0.793889,365.848638,147.329685,-0.460323,-9.126161,17473.202091



Top nonlinear improvements vs linear:


,relationship,model,horizon,threshold,score_gain_vs_linear,hit_gain_vs_linear,edge_gain_vs_linear,nonlinear_score,linear_score,mean_hit_rate,linear_hit_rate,mean_reversion_edge,linear_reversion_edge
0,curved_triangle_vs_quads,logratio,1000,2.0,8279.101414,0.940503,1198.752288,18262.355233,9983.253820,0.940503,0.000000,0.029797,-1198.722491
1,square_vs_triangle,logratio,500,1.5,8151.343325,0.198831,-175.098500,17844.146854,9692.803529,0.725912,0.527081,0.038655,175.137154
2,quads_avg_vs_triangle,cubic,2500,2.0,7899.335301,0.446981,776.870139,17732.878882,9833.543580,0.987094,0.540113,794.148042,17.277903
3,quads_avg_vs_triangle,cubic,2500,1.5,7766.358520,0.313744,611.662614,17716.407445,9950.048925,0.968509,0.654764,736.892295,125.229681
4,curved_triangle_vs_quads,ratio,1000,2.0,7745.749352,0.951578,1198.760879,17729.003171,9983.253820,0.951578,0.000000,0.038388,-1198.722491
5,quads_avg_vs_triangle,cubic,100,2.5,7480.713241,0.157134,37.125817,16982.978740,9502.265499,0.696420,0.539286,69.017874,31.892057
6,square_vs_triangle,cubic,100,3.0,7473.357734,0.323928,770.899009,17013.425618,9540.067883,0.938739,0.614811,906.297605,135.398596
7,quads_avg_vs_triangle,cubic,100,2.0,7456.801706,0.131926,50.951969,16950.510509,9493.708803,0.661643,0.529717,71.192724,20.240756
8,quads_avg_vs_triangle,cubic,250,2.0,7430.986481,0.156855,71.767750,17133.831579,9702.845098,0.720878,0.564023,140.539436,68.771687
9,quads_avg_vs_triangle,cubic,250,2.5,7429.099271,0.156286,56.615999,17212.051158,9782.951888,0.801452,0.645166,180.059099,123.443100



Top SHAPE_FACTOR_SUMMARY:


,factor_model,product_residual,horizon,threshold,active_days,positive_edge_days,total_events,mean_hit_rate,min_hit_rate,mean_reversion_edge,min_reversion_edge,mean_mr_corr,shape_factor_score
0,curved_quads_triangle,MICROCHIP_OVAL,1500,2.5,3,3,1238,0.855626,0.566879,210.393351,24.217516,-0.848489,18239.725606
1,side_count_quadratic,MICROCHIP_CIRCLE,1500,2.5,3,3,1238,0.855626,0.566879,210.393351,24.217516,-0.848489,18239.725606
2,side_count_quadratic,MICROCHIP_OVAL,1500,2.5,3,3,1238,0.855626,0.566879,210.393351,24.217516,-0.848489,18239.725606
3,curved_quads_triangle,MICROCHIP_CIRCLE,1500,2.5,3,3,1238,0.855626,0.566879,210.393351,24.217516,-0.848489,18239.725606
4,side_count_quadratic,MICROCHIP_OVAL,1000,2.5,3,3,1461,0.890873,0.672619,193.587281,29.538690,-0.767842,18195.152556
5,curved_quads_triangle,MICROCHIP_OVAL,1000,2.5,3,3,1461,0.890873,0.672619,193.587281,29.538690,-0.767842,18195.152556
6,curved_quads_triangle,MICROCHIP_CIRCLE,1000,2.5,3,3,1461,0.890873,0.672619,193.587281,29.538690,-0.767842,18195.152556
7,side_count_quadratic,MICROCHIP_CIRCLE,1000,2.5,3,3,1461,0.890873,0.672619,193.587281,29.538690,-0.767842,18195.152556
8,side_count_quadratic,MICROCHIP_CIRCLE,250,2.5,3,3,2211,0.829509,0.572810,103.157498,30.597412,-0.490611,17858.628211
9,curved_quads_triangle,MICROCHIP_CIRCLE,250,2.5,3,3,2211,0.829509,0.572810,103.157498,30.597412,-0.490611,17858.628211


In [14]:
# ════════════════════════════════════════════════════════════════════════════
# MICROCHIPS — SIMPLE BEST BID / BEST ASK BACKTEST
# No extra fees/slippage. Execution only uses best bid/ask.
# Broad vectorised scan over strongest latest signals.
# ════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import time
from pathlib import Path
import warnings

t0 = time.time()

# --------------------------------------------------------------------------
# 0. Inputs / product order
# --------------------------------------------------------------------------

assert "prices" in globals(), "Expected a `prices` DataFrame to exist."

MICRO_PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

P_CIRCLE, P_OVAL, P_SQUARE, P_RECT, P_TRI = range(5)

day_col = "file_day" if "file_day" in prices.columns else "day"
required_cols = [day_col, "timestamp", "product", "bid_price_1", "ask_price_1", "mid_price"]
missing_cols = [c for c in required_cols if c not in prices.columns]
assert not missing_cols, f"Missing required price columns: {missing_cols}"

micro_prices = prices.loc[prices["product"].isin(MICRO_PRODUCTS), required_cols].copy()

# Pivot once.
mid = micro_prices.pivot_table(index=[day_col, "timestamp"], columns="product", values="mid_price", aggfunc="last")
bid = micro_prices.pivot_table(index=[day_col, "timestamp"], columns="product", values="bid_price_1", aggfunc="last")
ask = micro_prices.pivot_table(index=[day_col, "timestamp"], columns="product", values="ask_price_1", aggfunc="last")

# Enforce product order and aligned non-missing rows.
for p in MICRO_PRODUCTS:
    assert p in mid.columns, f"Missing product in prices: {p}"

mid = mid[MICRO_PRODUCTS]
bid = bid[MICRO_PRODUCTS]
ask = ask[MICRO_PRODUCTS]

valid_idx = mid.dropna().index.intersection(bid.dropna().index).intersection(ask.dropna().index)
mid = mid.loc[valid_idx].sort_index()
bid = bid.loc[valid_idx].sort_index()
ask = ask.loc[valid_idx].sort_index()

days = sorted(mid.index.get_level_values(day_col).unique())
print("Aligned shape:", mid.shape)
print("Days:", days)
print("Products:", MICRO_PRODUCTS)

DAY_DATA = {}
for d in days:
    X = mid.xs(d, level=day_col).to_numpy(float)
    B = bid.xs(d, level=day_col).to_numpy(float)
    A = ask.xs(d, level=day_col).to_numpy(float)
    ts = mid.xs(d, level=day_col).index.to_numpy()
    assert X.shape == B.shape == A.shape
    assert X.shape[1] == len(MICRO_PRODUCTS)
    assert np.isfinite(X).all() and np.isfinite(B).all() and np.isfinite(A).all()
    DAY_DATA[d] = {"X": X, "B": B, "A": A, "ts": ts}
    print(f"Day {d}: X={X.shape}")

# --------------------------------------------------------------------------
# 1. Optional simple trade-flow arrays
# --------------------------------------------------------------------------
# Uses trade price vs same-timestamp mid to sign quantity:
# price > mid => aggressive buy proxy
# price < mid => aggressive sell proxy
# This is optional and only runs if `trades` exists.

FLOW_DAY = {}

if "trades" in globals():
    try:
        trade_day_col = "file_day" if "file_day" in trades.columns else "day"
        tr = trades.loc[trades["product"].isin(MICRO_PRODUCTS)].copy()

        if {"timestamp", "product", "price", "quantity", trade_day_col}.issubset(tr.columns):
            mid_long = (
                mid.stack()
                .rename("mid_at_trade_ts")
                .reset_index()
                .rename(columns={day_col: "day_key", "level_2": "product"})
            )
            tr = tr.rename(columns={trade_day_col: "day_key"})
            tr = tr.merge(
                mid_long,
                left_on=["day_key", "timestamp", "product"],
                right_on=["day_key", "timestamp", "product"],
                how="left",
            )

            tr["signed_qty"] = np.where(
                tr["price"] > tr["mid_at_trade_ts"],
                tr["quantity"],
                np.where(tr["price"] < tr["mid_at_trade_ts"], -tr["quantity"], 0.0),
            )

            flow_piv = (
                tr.groupby(["day_key", "timestamp", "product"])["signed_qty"]
                .sum()
                .unstack("product")
                .reindex(columns=MICRO_PRODUCTS)
                .fillna(0.0)
            )

            # Reindex to price grid.
            target_idx = mid.index.rename(["day_key", "timestamp"])
            flow_piv = flow_piv.reindex(target_idx, fill_value=0.0)

            for d in days:
                day_flow = flow_piv.xs(d, level="day_key").to_numpy(float)
                FLOW_DAY[d] = day_flow

            print("Optional flow arrays built.")
        else:
            print("Trades exists, but required trade columns missing. Flow signals skipped.")
    except Exception as e:
        print("Flow signal setup failed, skipped:", repr(e))
else:
    print("No `trades` DataFrame found. Flow signals skipped.")

# --------------------------------------------------------------------------
# 2. Strongest latest microchip signals to test
# --------------------------------------------------------------------------

def q(arr):
    return np.array(arr, dtype=float)

CANDIDATES = [
    # Strong nonlinear / ratio discoveries
    {
        "name": "circle_oval_logratio_mr",
        "family": "curved_pair",
        "type": "logratio",
        "mode": "meanrev",
        "q_signal": q([1, -1, 0, 0, 0]),
        "q_trade": q([10, -10, 0, 0, 0]),
        "note": "log(CIRCLE / OVAL), strongest clean nonlinear pair relationship",
    },
    {
        "name": "curved_triangle_vs_quads_logratio_mr",
        "family": "shape_family",
        "type": "logratio",
        "mode": "meanrev",
        "q_signal": q([1, 1, -1, -1, 1]),
        "q_trade": q([10, 10, -10, -10, 10]),
        "note": "log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE))",
    },
    {
        "name": "regular_vs_elongated_linear_mr",
        "family": "regular_vs_elongated",
        "type": "linear",
        "mode": "meanrev",
        "q_signal": q([1, -1, 1, -1, 0]),
        "q_trade": q([10, -10, 10, -10, 0]),
        "note": "CIRCLE+SQUARE versus OVAL+RECTANGLE",
    },
    {
        "name": "circle_square_rect_vs_oval_triangle_mr",
        "family": "shape_group",
        "type": "linear",
        "mode": "meanrev",
        "q_signal": q([1, -1, 1, 1, -1]),
        "q_trade": q([10, -10, 10, 10, -10]),
        "note": "CIRCLE+SQUARE+RECTANGLE versus OVAL+TRIANGLE",
    },
    {
        "name": "oval_triangle_vs_rectangle_logratio_mr",
        "family": "stage1_interpretable",
        "type": "logratio",
        "mode": "meanrev",
        "q_signal": q([0, 1, 0, -1, 1]),
        "q_trade": q([0, 10, 0, -10, 10]),
        "note": "log((OVAL+TRIANGLE)/RECTANGLE)",
    },
    {
        "name": "triangle_vs_quads_logratio_mr",
        "family": "triangle_vs_quads",
        "type": "logratio",
        "mode": "meanrev",
        "q_signal": q([0, 0, -0.5, -0.5, 1]),
        "q_trade": q([0, 0, -5, -5, 10]),
        "note": "log(TRIANGLE / average(SQUARE,RECTANGLE))",
    },
    {
        "name": "circle_rect_vs_oval_2tri_mr",
        "family": "integer_shape",
        "type": "linear",
        "mode": "meanrev",
        "q_signal": q([1, -1, 0, 1, -2]),
        "q_trade": q([5, -5, 0, 5, -10]),
        "note": "CIRCLE+RECTANGLE versus OVAL+2*TRIANGLE",
    },

    # Nonlinear residuals
    {
        "name": "square_rectangle_cubic_resid_mr",
        "family": "quadrilateral_pair",
        "type": "poly_resid",
        "mode": "meanrev",
        "degree": 3,
        "x_weights": q([0, 0, 0, 1, 0]),      # RECTANGLE
        "y_weights": q([0, 0, 1, 0, 0]),      # SQUARE
        "q_trade": q([0, 0, 10, -10, 0]),
        "note": "SQUARE residual versus cubic function of RECTANGLE",
    },
    {
        "name": "square_triangle_logratio_mr",
        "family": "square_vs_triangle",
        "type": "logratio",
        "mode": "meanrev",
        "q_signal": q([0, 0, 1, 0, -1]),
        "q_trade": q([0, 0, 10, 0, -10]),
        "note": "log(SQUARE/TRIANGLE)",
    },
    {
        "name": "square_triangle_cubic_resid_mr",
        "family": "square_vs_triangle",
        "type": "poly_resid",
        "mode": "meanrev",
        "degree": 3,
        "x_weights": q([0, 0, 0, 0, 1]),      # TRIANGLE
        "y_weights": q([0, 0, 1, 0, 0]),      # SQUARE
        "q_trade": q([0, 0, 10, 0, -10]),
        "note": "SQUARE residual versus cubic function of TRIANGLE",
    },
    {
        "name": "quads_avg_triangle_cubic_resid_mr",
        "family": "quads_vs_triangle",
        "type": "poly_resid",
        "mode": "meanrev",
        "degree": 3,
        "x_weights": q([0, 0, 0, 0, 1]),      # TRIANGLE
        "y_weights": q([0, 0, 0.5, 0.5, 0]),  # avg(SQUARE,RECTANGLE)
        "q_trade": q([0, 0, 5, 5, -10]),
        "note": "Quadrilateral average residual versus cubic function of TRIANGLE",
    },

    # Own-move / lead-lag signals that looked viable
    {
        "name": "own_square_momentum_100_follow",
        "family": "own_move",
        "type": "own_move",
        "mode": "follow",
        "product_idx": P_SQUARE,
        "past_h": 100,
        "q_trade": q([0, 0, 10, 0, 0]),
        "note": "SQUARE own momentum after 100-row move",
    },
    {
        "name": "own_rectangle_reversal_500",
        "family": "own_move",
        "type": "own_move",
        "mode": "inverse",
        "product_idx": P_RECT,
        "past_h": 500,
        "q_trade": q([0, 0, 0, 10, 0]),
        "note": "RECTANGLE own reversal after 500-row move",
    },
    {
        "name": "lead_circle_to_square_inverse_50",
        "family": "leadlag",
        "type": "leadlag",
        "mode": "inverse",
        "leader_idx": P_CIRCLE,
        "past_h": 50,
        "q_trade": q([0, 0, 10, 0, 0]),
        "note": "CIRCLE move predicts opposite SQUARE move",
    },
    {
        "name": "lead_oval_to_triangle_follow_500",
        "family": "leadlag",
        "type": "leadlag",
        "mode": "follow",
        "leader_idx": P_OVAL,
        "past_h": 500,
        "q_trade": q([0, 0, 0, 0, 10]),
        "note": "OVAL move predicts TRIANGLE follow",
    },
]

# Optional flow candidates.
if FLOW_DAY:
    CANDIDATES.extend([
        {
            "name": "flow_square_follow_5000",
            "family": "trade_flow",
            "type": "flow",
            "mode": "follow",
            "product_idx": P_SQUARE,
            "flow_window_time": 5000,
            "q_trade": q([0, 0, 10, 0, 0]),
            "note": "SQUARE follows signed trade-flow imbalance over 5000 timestamp units",
        },
        {
            "name": "flow_square_follow_10000",
            "family": "trade_flow",
            "type": "flow",
            "mode": "follow",
            "product_idx": P_SQUARE,
            "flow_window_time": 10000,
            "q_trade": q([0, 0, 10, 0, 0]),
            "note": "SQUARE follows signed trade-flow imbalance over 10000 timestamp units",
        },
        {
            "name": "flow_triangle_follow_10000",
            "family": "trade_flow",
            "type": "flow",
            "mode": "follow",
            "product_idx": P_TRI,
            "flow_window_time": 10000,
            "q_trade": q([0, 0, 0, 0, 10]),
            "note": "TRIANGLE follows signed trade-flow imbalance over 10000 timestamp units",
        },
    ])

print("Candidates:", len(CANDIDATES))
for c in CANDIDATES:
    print(f"- {c['name']} | {c['family']} | {c['note']}")

# --------------------------------------------------------------------------
# 3. Helpers
# --------------------------------------------------------------------------

def rolling_z(raw, window):
    s = pd.Series(raw, dtype="float64")
    mean = s.rolling(window=window, min_periods=max(20, window // 4)).mean()
    std = s.rolling(window=window, min_periods=max(20, window // 4)).std()
    z = (s - mean) / std
    return z.to_numpy(float)

def compute_raw_signal(candidate, X, day=None):
    typ = candidate["type"]

    if typ == "linear":
        return X @ candidate["q_signal"]

    if typ == "logratio":
        qs = candidate["q_signal"]
        pos = np.clip(qs, 0, None)
        neg = np.clip(-qs, 0, None)
        numerator = X @ pos
        denominator = X @ neg
        raw = np.full(X.shape[0], np.nan)
        valid = (numerator > 0) & (denominator > 0)
        raw[valid] = np.log(numerator[valid] / denominator[valid])
        return raw

    if typ == "poly_resid":
        x = X @ candidate["x_weights"]
        y = X @ candidate["y_weights"]
        deg = int(candidate["degree"])
        valid = np.isfinite(x) & np.isfinite(y)
        raw = np.full(X.shape[0], np.nan)
        if valid.sum() <= deg + 5:
            return raw

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", np.RankWarning if hasattr(np, "RankWarning") else Warning)
            coeff = np.polyfit(x[valid], y[valid], deg=deg)

        raw[valid] = y[valid] - np.polyval(coeff, x[valid])
        return raw

    if typ == "own_move":
        idx = candidate["product_idx"]
        past = int(candidate["past_h"])
        raw = np.full(X.shape[0], np.nan)
        raw[past:] = X[past:, idx] - X[:-past, idx]
        return raw

    if typ == "leadlag":
        idx = candidate["leader_idx"]
        past = int(candidate["past_h"])
        raw = np.full(X.shape[0], np.nan)
        raw[past:] = X[past:, idx] - X[:-past, idx]
        return raw

    if typ == "flow":
        assert day in FLOW_DAY, f"Flow array missing for day {day}"
        idx = candidate["product_idx"]
        flow = FLOW_DAY[day][:, idx].astype(float)

        # Convert timestamp-window to approximate row-window.
        # Dataset timestamps usually step by 100.
        flow_window_time = int(candidate["flow_window_time"])
        row_window = max(1, flow_window_time // 100)

        raw = pd.Series(flow).rolling(row_window, min_periods=max(5, row_window // 4)).sum().to_numpy(float)
        return raw

    raise ValueError(f"Unknown signal type: {typ}")

def direction_from_z(z, mode):
    if mode in ("meanrev", "inverse"):
        return -np.sign(z)
    if mode in ("follow", "breakout"):
        return np.sign(z)
    raise ValueError(f"Unknown mode: {mode}")

def evaluate_config(candidate, day, window, horizon, threshold, collect_events=False):
    X = DAY_DATA[day]["X"]
    B = DAY_DATA[day]["B"]
    A = DAY_DATA[day]["A"]
    ts = DAY_DATA[day]["ts"]

    n = X.shape[0]
    h = int(horizon)
    if n <= h + window + 5:
        return None, None

    raw = compute_raw_signal(candidate, X, day=day)
    z = rolling_z(raw, int(window))

    z0 = z[:-h]
    valid = np.isfinite(z0) & (np.abs(z0) >= threshold)

    event_count = int(valid.sum())
    if event_count == 0:
        row = {
            "candidate": candidate["name"],
            "family": candidate["family"],
            "signal_type": candidate["type"],
            "note": candidate["note"],
            "mode": candidate["mode"],
            "day": day,
            "window": window,
            "horizon": horizon,
            "threshold": threshold,
            "event_count": 0,
            "win_events": 0,
            "total_exec_pnl": 0.0,
            "total_mid_pnl": 0.0,
            "crossing_loss": 0.0,
            "avg_exec_pnl": np.nan,
            "median_exec_pnl": np.nan,
            "hit_rate": np.nan,
            "avg_abs_z": np.nan,
            "q_trade": candidate["q_trade"].tolist(),
        }
        return row, None

    side = direction_from_z(z0[valid], candidate["mode"])
    q_base = candidate["q_trade"].astype(float)
    q_events = side[:, None] * q_base[None, :]

    X0 = X[:-h][valid]
    X1 = X[h:][valid]
    B0 = B[:-h][valid]
    A0 = A[:-h][valid]
    B1 = B[h:][valid]
    A1 = A[h:][valid]

    # Best bid/ask execution:
    # Long entry buys at ask, long exit sells at bid.
    # Short entry sells at bid, short exit buys at ask.
    entry_px = np.where(q_events > 0, A0, B0)
    exit_px = np.where(q_events > 0, B1, A1)

    exec_pnl = ((exit_px - entry_px) * q_events).sum(axis=1)
    mid_pnl = ((X1 - X0) * q_events).sum(axis=1)

    crossing_loss = mid_pnl - exec_pnl
    assert np.allclose(mid_pnl - exec_pnl, crossing_loss, equal_nan=False)
    assert np.isfinite(exec_pnl).all()

    row = {
        "candidate": candidate["name"],
        "family": candidate["family"],
        "signal_type": candidate["type"],
        "note": candidate["note"],
        "mode": candidate["mode"],
        "day": day,
        "window": window,
        "horizon": horizon,
        "threshold": threshold,
        "event_count": event_count,
        "win_events": int((exec_pnl > 0).sum()),
        "total_exec_pnl": float(exec_pnl.sum()),
        "total_mid_pnl": float(mid_pnl.sum()),
        "crossing_loss": float(crossing_loss.sum()),
        "avg_exec_pnl": float(exec_pnl.mean()),
        "median_exec_pnl": float(np.median(exec_pnl)),
        "hit_rate": float((exec_pnl > 0).mean()),
        "avg_abs_z": float(np.nanmean(np.abs(z0[valid]))),
        "q_trade": q_base.tolist(),
    }

    event_df = None
    if collect_events:
        idx_all = np.arange(n - h)[valid]
        event_df = pd.DataFrame({
            "candidate": candidate["name"],
            "family": candidate["family"],
            "signal_type": candidate["type"],
            "mode": candidate["mode"],
            "day": day,
            "window": window,
            "horizon": horizon,
            "threshold": threshold,
            "entry_idx": idx_all,
            "exit_idx": idx_all + h,
            "entry_ts": ts[idx_all],
            "exit_ts": ts[idx_all + h],
            "side": side,
            "z": z0[valid],
            "exec_pnl": exec_pnl,
            "mid_pnl": mid_pnl,
            "crossing_loss": crossing_loss,
        })

    return row, event_df

# --------------------------------------------------------------------------
# 4. Broad simple scan
# --------------------------------------------------------------------------

WINDOWS = [250, 500, 1000, 2500, 5000]
HORIZONS = [250, 500, 1000, 1500, 2500]
THRESHOLDS = [1.5, 2.0, 2.5, 3.0]

rows = []

total_configs = len(CANDIDATES) * len(days) * len(WINDOWS) * len(HORIZONS) * len(THRESHOLDS)
print("Total config/day combinations:", total_configs)

counter = 0
for ci, cand in enumerate(CANDIDATES, start=1):
    print(f"[{time.time()-t0:7.2f}s] Candidate {ci}/{len(CANDIDATES)}: {cand['name']}")
    for d in days:
        for w in WINDOWS:
            for h in HORIZONS:
                for th in THRESHOLDS:
                    counter += 1
                    row, _ = evaluate_config(cand, d, w, h, th, collect_events=False)
                    if row is not None:
                        rows.append(row)

MICRO_SIMPLE_DAY_RESULTS = pd.DataFrame(rows)
print("MICRO_SIMPLE_DAY_RESULTS:", MICRO_SIMPLE_DAY_RESULTS.shape)

# --------------------------------------------------------------------------
# 5. Summary / robust ranking
# --------------------------------------------------------------------------

group_cols = ["candidate", "family", "signal_type", "note", "mode", "window", "horizon", "threshold"]

MICRO_SIMPLE_SUMMARY = (
    MICRO_SIMPLE_DAY_RESULTS
    .groupby(group_cols, dropna=False)
    .agg(
        days=("day", "nunique"),
        active_days=("event_count", lambda s: int((s > 0).sum())),
        event_count=("event_count", "sum"),
        total_exec_pnl=("total_exec_pnl", "sum"),
        total_mid_pnl=("total_mid_pnl", "sum"),
        total_crossing_loss=("crossing_loss", "sum"),
        worst_day_exec_pnl=("total_exec_pnl", "min"),
        best_day_exec_pnl=("total_exec_pnl", "max"),
        positive_days=("total_exec_pnl", lambda s: int((s > 0).sum())),
        mean_day_pnl=("total_exec_pnl", "mean"),
        avg_trade_pnl=("avg_exec_pnl", "mean"),
        median_trade_pnl=("median_exec_pnl", "mean"),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        avg_abs_z=("avg_abs_z", "mean"),
    )
    .reset_index()
)

MICRO_SIMPLE_SUMMARY["pnl_per_event"] = (
    MICRO_SIMPLE_SUMMARY["total_exec_pnl"] / MICRO_SIMPLE_SUMMARY["event_count"].replace(0, np.nan)
)

MICRO_SIMPLE_SUMMARY["pnl_per_crossing_loss"] = (
    MICRO_SIMPLE_SUMMARY["total_exec_pnl"] / MICRO_SIMPLE_SUMMARY["total_crossing_loss"].replace(0, np.nan)
)

# Robust score favours all 3 days positive, high worst-day PnL, and decent hit rate.
MICRO_SIMPLE_SUMMARY["robust_score"] = (
    10000 * MICRO_SIMPLE_SUMMARY["positive_days"]
    + 0.20 * MICRO_SIMPLE_SUMMARY["worst_day_exec_pnl"]
    + 0.05 * MICRO_SIMPLE_SUMMARY["total_exec_pnl"]
    + 5000 * (MICRO_SIMPLE_SUMMARY["mean_hit_rate"].fillna(0) - 0.5)
)

MICRO_SIMPLE_SUMMARY = MICRO_SIMPLE_SUMMARY.sort_values(
    ["positive_days", "worst_day_exec_pnl", "total_exec_pnl", "mean_hit_rate", "pnl_per_event"],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)

MICRO_SIMPLE_BEST_BY_CANDIDATE = (
    MICRO_SIMPLE_SUMMARY
    .sort_values(["candidate", "positive_days", "worst_day_exec_pnl", "total_exec_pnl"], ascending=[True, False, False, False])
    .groupby("candidate", as_index=False)
    .head(1)
    .sort_values(["positive_days", "worst_day_exec_pnl", "total_exec_pnl"], ascending=[False, False, False])
    .reset_index(drop=True)
)

# Detailed day table for best configs.
top_keys = MICRO_SIMPLE_SUMMARY.head(20)[group_cols]
MICRO_SIMPLE_TOP_DAY_BREAKDOWN = MICRO_SIMPLE_DAY_RESULTS.merge(
    top_keys,
    on=group_cols,
    how="inner",
).sort_values(["candidate", "window", "horizon", "threshold", "day"])

# --------------------------------------------------------------------------
# 6. Collect event-level logs only for top configs
# --------------------------------------------------------------------------

event_logs = []
cand_by_name = {c["name"]: c for c in CANDIDATES}

for _, r in MICRO_SIMPLE_SUMMARY.head(15).iterrows():
    cand = cand_by_name[r["candidate"]]
    for d in days:
        _, ev = evaluate_config(
            cand,
            d,
            int(r["window"]),
            int(r["horizon"]),
            float(r["threshold"]),
            collect_events=True,
        )
        if ev is not None and len(ev):
            event_logs.append(ev)

MICRO_SIMPLE_TOP_TRADE_LOG = (
    pd.concat(event_logs, ignore_index=True)
    if event_logs else
    pd.DataFrame()
)

# --------------------------------------------------------------------------
# 7. Save outputs
# --------------------------------------------------------------------------

OUT_DIR = Path("analysis_outputs/microchips_simple_backtest")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MICRO_SIMPLE_DAY_RESULTS.to_csv(OUT_DIR / "micro_simple_day_results.csv", index=False)
MICRO_SIMPLE_SUMMARY.to_csv(OUT_DIR / "micro_simple_summary.csv", index=False)
MICRO_SIMPLE_BEST_BY_CANDIDATE.to_csv(OUT_DIR / "micro_simple_best_by_candidate.csv", index=False)
MICRO_SIMPLE_TOP_DAY_BREAKDOWN.to_csv(OUT_DIR / "micro_simple_top_day_breakdown.csv", index=False)
MICRO_SIMPLE_TOP_TRADE_LOG.to_csv(OUT_DIR / "micro_simple_top_trade_log.csv", index=False)

print("\nSaved outputs to:", OUT_DIR)
print("Runtime:", round(time.time() - t0, 2), "seconds")

# --------------------------------------------------------------------------
# 8. Display key outputs
# --------------------------------------------------------------------------

print("\nTop MICRO_SIMPLE_SUMMARY:")
display(MICRO_SIMPLE_SUMMARY.head(80))

print("\nMICRO_SIMPLE_BEST_BY_CANDIDATE:")
display(MICRO_SIMPLE_BEST_BY_CANDIDATE)

print("\nMICRO_SIMPLE_TOP_DAY_BREAKDOWN:")
display(MICRO_SIMPLE_TOP_DAY_BREAKDOWN.head(120))

print("\nMICRO_SIMPLE_TOP_TRADE_LOG:")
display(MICRO_SIMPLE_TOP_TRADE_LOG.head(50))

Aligned shape: (30000, 5)
Days: [2, 3, 4]
Products: ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE', 'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']
Day 2: X=(10000, 5)
Day 3: X=(10000, 5)
Day 4: X=(10000, 5)
Optional flow arrays built.
Candidates: 18
- circle_oval_logratio_mr | curved_pair | log(CIRCLE / OVAL), strongest clean nonlinear pair relationship
- curved_triangle_vs_quads_logratio_mr | shape_family | log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE))
- regular_vs_elongated_linear_mr | regular_vs_elongated | CIRCLE+SQUARE versus OVAL+RECTANGLE
- circle_square_rect_vs_oval_triangle_mr | shape_group | CIRCLE+SQUARE+RECTANGLE versus OVAL+TRIANGLE
- oval_triangle_vs_rectangle_logratio_mr | stage1_interpretable | log((OVAL+TRIANGLE)/RECTANGLE)
- triangle_vs_quads_logratio_mr | triangle_vs_quads | log(TRIANGLE / average(SQUARE,RECTANGLE))
- circle_rect_vs_oval_2tri_mr | integer_shape | CIRCLE+RECTANGLE versus OVAL+2*TRIANGLE
- square_rectangle_cubic_resid_mr | quadrilateral_pair 

,candidate,family,signal_type,note,mode,window,horizon,threshold,days,active_days,...,positive_days,mean_day_pnl,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_abs_z,pnl_per_event,pnl_per_crossing_loss,robust_score
0,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,5000,1000,1.5,3,3,...,3,1.292197e+07,5731.680279,5385.000000,0.851828,0.819772,2.102468,5627.217303,12.801251,4.284660e+06
1,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2500,1000,1.5,3,3,...,3,1.127798e+07,4419.472204,5080.000000,0.778018,0.718876,2.174373,4410.629644,10.011789,3.641209e+06
2,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,5000,1000,2.0,3,3,...,3,8.505923e+06,7411.543791,6590.000000,0.955481,0.945319,2.457297,7372.947125,16.727205,2.939612e+06
3,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2500,1000,2.0,3,3,...,3,8.561280e+06,6002.498967,6396.666667,0.834759,0.764485,2.509917,6013.542496,13.612309,2.838500e+06
4,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE,RECTANGLE))",meanrev,5000,1500,1.5,3,3,...,3,6.855772e+06,3975.938718,3952.500000,0.787019,0.613161,2.002451,3983.597715,21.579332,2.192570e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE,RECTANGLE))",meanrev,500,2500,1.5,3,3,...,3,2.556873e+06,1033.514733,1220.000000,0.533655,0.471247,2.064978,1017.323607,5.492009,6.510793e+05
76,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,1000,1000,2.0,3,3,...,3,1.946237e+06,1609.596430,891.666667,0.590479,0.536068,2.440135,1658.724432,8.091787,5.590599e+05
77,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE,RECTANGLE))",meanrev,250,1000,1.5,3,3,...,3,3.509978e+06,1133.242448,1097.500000,0.572009,0.496087,2.036428,1169.473012,6.328533,7.903028e+05
78,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,2500,1000,2.0,3,3,...,3,3.281687e+06,3772.623965,3176.666667,0.720067,0.599407,2.467706,3418.423611,16.631995,7.567153e+05



MICRO_SIMPLE_BEST_BY_CANDIDATE:


,candidate,family,signal_type,note,mode,window,horizon,threshold,days,active_days,...,positive_days,mean_day_pnl,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_abs_z,pnl_per_event,pnl_per_crossing_loss,robust_score
0,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,5000,1000,1.5,3,3,...,3,1.292197e+07,5731.680279,5385.000000,0.851828,0.819772,2.102468,5627.217303,12.801251,4.284660e+06
1,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE,RECTANGLE))",meanrev,5000,1500,1.5,3,3,...,3,6.855772e+06,3975.938718,3952.500000,0.787019,0.613161,2.002451,3983.597715,21.579332,2.192570e+06
2,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,1000,1000,1.5,3,3,...,3,6.117737e+06,2253.612613,941.666667,0.596866,0.528736,2.027632,2206.975709,10.790801,2.078649e+06
3,quads_avg_triangle_cubic_resid_mr,quads_vs_triangle,poly_resid,Quadrilateral average residual versus cubic fu...,meanrev,2500,1000,1.5,3,3,...,3,4.295790e+06,1912.522906,2500.833333,0.695518,0.624214,2.180046,1898.272205,10.273405,1.513010e+06
4,square_triangle_cubic_resid_mr,square_vs_triangle,poly_resid,SQUARE residual versus cubic function of TRIANGLE,meanrev,5000,500,1.5,3,3,...,3,3.182143e+06,1825.907908,1726.666667,0.598668,0.548942,2.041256,1453.033486,7.032983,9.758708e+05
5,square_rectangle_cubic_resid_mr,quadrilateral_pair,poly_resid,SQUARE residual versus cubic function of RECTA...,meanrev,250,1500,2.0,3,3,...,3,2.637000e+06,2135.571657,3425.000000,0.607946,0.580220,2.457175,2127.184727,10.779618,8.767717e+05
6,oval_triangle_vs_rectangle_logratio_mr,stage1_interpretable,logratio,log((OVAL+TRIANGLE)/RECTANGLE),meanrev,1000,500,2.0,3,3,...,3,3.380830e+06,2284.739736,2466.666667,0.681032,0.665035,2.448400,2355.431955,9.762673,9.099617e+05
7,lead_oval_to_triangle_follow_500,leadlag,leadlag,OVAL move predicts TRIANGLE follow,follow,500,500,1.5,3,3,...,3,2.806307e+06,879.451751,726.666667,0.605769,0.582670,2.022442,870.712587,10.102987,8.033468e+05
8,circle_square_rect_vs_oval_triangle_mr,shape_group,linear,CIRCLE+SQUARE+RECTANGLE versus OVAL+TRIANGLE,meanrev,2500,1500,2.5,3,3,...,3,2.086873e+06,4584.595143,6831.666667,0.753663,0.530220,2.815642,4218.746631,9.662345,6.250613e+05
9,circle_rect_vs_oval_2tri_mr,integer_shape,linear,CIRCLE+RECTANGLE versus OVAL+2*TRIANGLE,meanrev,250,1000,1.5,3,3,...,3,3.760095e+06,1225.574834,967.500000,0.569420,0.417522,2.043547,1228.120305,6.019531,8.018714e+05



MICRO_SIMPLE_TOP_DAY_BREAKDOWN:


,candidate,family,signal_type,note,mode,day,window,horizon,threshold,event_count,win_events,total_exec_pnl,total_mid_pnl,crossing_loss,avg_exec_pnl,median_exec_pnl,hit_rate,avg_abs_z,q_trade
0,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2,500,1500,1.5,2754,1552,4283020.0,5511125.0,1228105.0,1555.199710,1040.0,0.563544,2.065292,"[10.0, 10.0, -10.0, -10.0, 10.0]"
9,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,3,500,1500,1.5,2751,1569,6104600.0,7339785.0,1235185.0,2219.047619,1080.0,0.570338,2.026222,"[10.0, 10.0, -10.0, -10.0, 10.0]"
18,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,4,500,1500,1.5,2567,1470,3522130.0,4616475.0,1094345.0,1372.080249,1200.0,0.572653,2.070776,"[10.0, 10.0, -10.0, -10.0, 10.0]"
1,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2,1000,1000,1.5,2989,1988,7691830.0,9020970.0,1329140.0,2573.379057,3460.0,0.665105,2.084135,"[10.0, 10.0, -10.0, -10.0, 10.0]"
10,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,3,1000,1000,1.5,2517,2012,17246300.0,18377730.0,1131430.0,6851.926897,6320.0,0.799364,2.152786,"[10.0, 10.0, -10.0, -10.0, 10.0]"
19,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,4,1000,1000,1.5,3135,1861,5147800.0,6482435.0,1334635.0,1642.041467,2750.0,0.593620,2.042965,"[10.0, 10.0, -10.0, -10.0, 10.0]"
2,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2,2500,1000,1.5,2858,2238,12775060.0,14051535.0,1276475.0,4469.930021,6575.0,0.783065,2.133594,"[10.0, 10.0, -10.0, -10.0, 10.0]"
11,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,3,2500,1000,1.5,2323,1933,11468270.0,12514825.0,1046555.0,4936.835988,5190.0,0.832114,2.227690,"[10.0, 10.0, -10.0, -10.0, 10.0]"
20,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,4,2500,1000,1.5,2490,1790,9590610.0,10646990.0,1056380.0,3851.650602,3475.0,0.718876,2.161837,"[10.0, 10.0, -10.0, -10.0, 10.0]"
3,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2,2500,1000,2.0,1507,1306,9425080.0,10098170.0,673090.0,6254.200398,8050.0,0.866622,2.479722,"[10.0, 10.0, -10.0, -10.0, 10.0]"



MICRO_SIMPLE_TOP_TRADE_LOG:


,candidate,family,signal_type,mode,day,window,horizon,threshold,entry_idx,exit_idx,entry_ts,exit_ts,side,z,exec_pnl,mid_pnl,crossing_loss
0,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1249,2249,124900,224900,1.0,-2.248915,-11170.0,-10715.0,455.0
1,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1250,2250,125000,225000,1.0,-2.180532,-11060.0,-10610.0,450.0
2,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1251,2251,125100,225100,1.0,-2.179420,-11300.0,-10845.0,455.0
3,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1252,2252,125200,225200,1.0,-2.099110,-12130.0,-11670.0,460.0
4,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1253,2253,125300,225300,1.0,-1.914280,-12350.0,-11900.0,450.0
5,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1254,2254,125400,225400,1.0,-1.858287,-12670.0,-12225.0,445.0
6,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1255,2255,125500,225500,1.0,-1.939745,-12180.0,-11735.0,445.0
7,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1256,2256,125600,225600,1.0,-2.024609,-12140.0,-11700.0,440.0
8,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1257,2257,125700,225700,1.0,-2.111078,-12180.0,-11735.0,445.0
9,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,meanrev,2,5000,1000,1.5,1258,2258,125800,225800,1.0,-2.186916,-11540.0,-11085.0,455.0


In [15]:
import numpy as np
import pandas as pd
from time import perf_counter

t0 = perf_counter()

MICRO_PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

# Main strategy:
# log((CIRCLE + OVAL + TRIANGLE) / (SQUARE + RECTANGLE))
Q_TRADE = np.array([10.0, 10.0, -10.0, -10.0, 10.0])
Q_SIGNAL = np.array([1.0, 1.0, -1.0, -1.0, 1.0])

# Use micro_prices if already filtered, otherwise filter prices
px = micro_prices.copy() if "micro_prices" in globals() else prices[prices["product"].isin(MICRO_PRODUCTS)].copy()

day_col = "file_day" if "file_day" in px.columns else "day"
days = sorted(px[day_col].unique())
print("Days:", days)

def build_day_arrays(px_day):
    mids = px_day.pivot(index="timestamp", columns="product", values="mid_price").sort_index()
    bids = px_day.pivot(index="timestamp", columns="product", values="bid_price_1").sort_index()
    asks = px_day.pivot(index="timestamp", columns="product", values="ask_price_1").sort_index()

    mids = mids[MICRO_PRODUCTS]
    bids = bids[MICRO_PRODUCTS]
    asks = asks[MICRO_PRODUCTS]

    valid = ~(mids.isna().any(axis=1) | bids.isna().any(axis=1) | asks.isna().any(axis=1))

    return (
        mids.index[valid].to_numpy(),
        mids.loc[valid].to_numpy(dtype=float),
        bids.loc[valid].to_numpy(dtype=float),
        asks.loc[valid].to_numpy(dtype=float),
    )

def compute_signal(mid):
    c = mid[:, 0]
    o = mid[:, 1]
    s = mid[:, 2]
    r = mid[:, 3]
    t = mid[:, 4]

    numerator = c + o + t
    denominator = s + r

    return np.log(numerator / denominator)

def zscore_signal(signal, window):
    ser = pd.Series(signal)
    # shifted rolling stats avoids using the current tick inside its own z-score baseline
    mu = ser.rolling(window, min_periods=window).mean().shift(1)
    sd = ser.rolling(window, min_periods=window).std(ddof=0).shift(1)
    z = (ser - mu) / sd
    return z.to_numpy(dtype=float)

def trade_to_target(current_pos, target_pos, bid, ask):
    """
    Crosses the spread to move from current_pos to target_pos.
    Buys at ask, sells at bid.
    Returns cash_delta.
    """
    delta = target_pos - current_pos
    cash_delta = 0.0

    buys = delta > 0
    sells = delta < 0

    cash_delta -= np.sum(delta[buys] * ask[buys])
    cash_delta -= np.sum(delta[sells] * bid[sells])  # delta negative => adds cash

    return cash_delta

def backtest_one_day(day, timestamps, mid, bid, ask, window, entry_z, exit_z, max_hold):
    signal = compute_signal(mid)
    z = zscore_signal(signal, window)

    pos = np.zeros(len(MICRO_PRODUCTS), dtype=float)
    cash = 0.0

    in_trade = False
    entry_i = None
    entry_cash = None
    entry_side = None
    entry_z_val = None

    trades = []

    for i in range(len(signal)):
        zi = z[i]
        if not np.isfinite(zi):
            continue

        if not in_trade:
            if abs(zi) >= entry_z:
                # mean reversion:
                # z high => numerator expensive => short Q_SIGNAL basket
                # z low  => numerator cheap     => long Q_SIGNAL basket
                side = -np.sign(zi)
                target = side * Q_TRADE

                cash += trade_to_target(pos, target, bid[i], ask[i])
                pos = target.copy()

                in_trade = True
                entry_i = i
                entry_cash = cash
                entry_side = side
                entry_z_val = zi

        else:
            hold = i - entry_i
            should_exit = (abs(zi) <= exit_z) or (hold >= max_hold) or (i == len(signal) - 1)

            if should_exit:
                cash_before_exit = cash
                cash += trade_to_target(pos, np.zeros_like(pos), bid[i], ask[i])
                pos[:] = 0.0

                exec_pnl = cash - entry_cash

                trades.append({
                    "day": day,
                    "entry_idx": entry_i,
                    "exit_idx": i,
                    "entry_ts": timestamps[entry_i],
                    "exit_ts": timestamps[i],
                    "side": entry_side,
                    "entry_z": entry_z_val,
                    "exit_z": zi,
                    "hold": hold,
                    "exec_pnl": exec_pnl,
                })

                in_trade = False
                entry_i = None
                entry_cash = None
                entry_side = None
                entry_z_val = None

    return trades

# Focus grid around the configs that looked strongest before
windows = [500, 1000, 2500, 5000]
entry_zs = [1.5, 2.0, 2.5, 3.0]
exit_zs = [0.0, 0.25, 0.5, 1.0]
max_holds = [250, 500, 1000, 1500, 2500]

all_results = []
all_trades = []

day_arrays = {}
for day in days:
    ts, mid, bid, ask = build_day_arrays(px[px[day_col] == day])
    day_arrays[day] = (ts, mid, bid, ask)
    print(f"Day {day}: {mid.shape}")

total = len(windows) * len(entry_zs) * len(exit_zs) * len(max_holds)
k = 0

for window in windows:
    for entry_z in entry_zs:
        for exit_z in exit_zs:
            for max_hold in max_holds:
                k += 1
                if k % 25 == 0 or k == 1:
                    print(f"[{perf_counter()-t0:7.2f}s] config {k}/{total}: window={window}, entry={entry_z}, exit={exit_z}, hold={max_hold}")

                config_trades = []

                for day in days:
                    ts, mid, bid, ask = day_arrays[day]
                    trades = backtest_one_day(
                        day=day,
                        timestamps=ts,
                        mid=mid,
                        bid=bid,
                        ask=ask,
                        window=window,
                        entry_z=entry_z,
                        exit_z=exit_z,
                        max_hold=max_hold,
                    )
                    config_trades.extend(trades)

                    if trades:
                        day_pnl = sum(t["exec_pnl"] for t in trades)
                        hit_rate = np.mean([t["exec_pnl"] > 0 for t in trades])
                        avg_trade = np.mean([t["exec_pnl"] for t in trades])
                        median_trade = np.median([t["exec_pnl"] for t in trades])
                    else:
                        day_pnl = 0.0
                        hit_rate = np.nan
                        avg_trade = np.nan
                        median_trade = np.nan

                    all_results.append({
                        "candidate": "curved_triangle_vs_quads_logratio_mr",
                        "day": day,
                        "window": window,
                        "entry_z": entry_z,
                        "exit_z": exit_z,
                        "max_hold": max_hold,
                        "trade_count": len(trades),
                        "day_pnl": day_pnl,
                        "hit_rate": hit_rate,
                        "avg_trade_pnl": avg_trade,
                        "median_trade_pnl": median_trade,
                    })

                for tr in config_trades:
                    tr.update({
                        "candidate": "curved_triangle_vs_quads_logratio_mr",
                        "window": window,
                        "entry_z": entry_z,
                        "exit_z": exit_z,
                        "max_hold": max_hold,
                    })
                all_trades.extend(config_trades)

MICRO_POSITION_DAY_RESULTS = pd.DataFrame(all_results)
MICRO_POSITION_TRADE_LOG = pd.DataFrame(all_trades)

summary = (
    MICRO_POSITION_DAY_RESULTS
    .groupby(["candidate", "window", "entry_z", "exit_z", "max_hold"], as_index=False)
    .agg(
        days=("day", "nunique"),
        active_days=("trade_count", lambda x: int((x > 0).sum())),
        positive_days=("day_pnl", lambda x: int((x > 0).sum())),
        total_pnl=("day_pnl", "sum"),
        mean_day_pnl=("day_pnl", "mean"),
        min_day_pnl=("day_pnl", "min"),
        total_trades=("trade_count", "sum"),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        avg_trade_pnl=("avg_trade_pnl", "mean"),
        median_trade_pnl=("median_trade_pnl", "median"),
    )
)

summary["pnl_per_trade"] = summary["total_pnl"] / summary["total_trades"].replace(0, np.nan)

# Conservative robust score: rewards PnL, penalizes one bad day dependency
summary["one_day_dependency"] = (
    MICRO_POSITION_DAY_RESULTS
    .groupby(["candidate", "window", "entry_z", "exit_z", "max_hold"])["day_pnl"]
    .apply(lambda x: x.max() / x.sum() if x.sum() > 0 else np.nan)
    .to_numpy()
)

summary["robust_score"] = (
    summary["total_pnl"]
    * (summary["positive_days"] / summary["days"])
    * (1 - summary["one_day_dependency"].clip(0, 1))
)

MICRO_POSITION_SUMMARY = summary.sort_values(
    ["positive_days", "robust_score", "total_pnl"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("\nTop MICRO_POSITION_SUMMARY:")
display(MICRO_POSITION_SUMMARY.head(50))

print("\nTop day breakdown for best config:")
best = MICRO_POSITION_SUMMARY.iloc[0]
mask = (
    (MICRO_POSITION_DAY_RESULTS["window"] == best["window"]) &
    (MICRO_POSITION_DAY_RESULTS["entry_z"] == best["entry_z"]) &
    (MICRO_POSITION_DAY_RESULTS["exit_z"] == best["exit_z"]) &
    (MICRO_POSITION_DAY_RESULTS["max_hold"] == best["max_hold"])
)
display(MICRO_POSITION_DAY_RESULTS[mask].sort_values("day"))

print("\nTrades for best config:")
display(MICRO_POSITION_TRADE_LOG[
    (MICRO_POSITION_TRADE_LOG["window"] == best["window"]) &
    (MICRO_POSITION_TRADE_LOG["entry_z"] == best["entry_z"]) &
    (MICRO_POSITION_TRADE_LOG["exit_z"] == best["exit_z"]) &
    (MICRO_POSITION_TRADE_LOG["max_hold"] == best["max_hold"])
].head(100))

print(f"\nRuntime: {perf_counter() - t0:.2f}s")

Days: [np.int64(2), np.int64(3), np.int64(4)]
Day 2: (10000, 5)
Day 3: (10000, 5)
Day 4: (10000, 5)
[   0.07s] config 1/320: window=500, entry=1.5, exit=0.0, hold=250
[   0.42s] config 25/320: window=500, entry=2.0, exit=0.0, hold=2500
[   0.82s] config 50/320: window=500, entry=2.5, exit=0.25, hold=2500
[   1.14s] config 75/320: window=500, entry=3.0, exit=0.5, hold=2500
[   1.49s] config 100/320: window=1000, entry=1.5, exit=1.0, hold=2500
[   1.82s] config 125/320: window=1000, entry=2.5, exit=0.0, hold=2500
[   2.14s] config 150/320: window=1000, entry=3.0, exit=0.25, hold=2500
[   2.45s] config 175/320: window=2500, entry=1.5, exit=0.5, hold=2500
[   2.77s] config 200/320: window=2500, entry=2.0, exit=1.0, hold=2500
[   3.07s] config 225/320: window=2500, entry=3.0, exit=0.0, hold=2500
[   3.38s] config 250/320: window=5000, entry=1.5, exit=0.25, hold=2500
[   3.68s] config 275/320: window=5000, entry=2.0, exit=0.5, hold=2500
[   3.97s] config 300/320: window=5000, entry=2.5, exit

,candidate,window,entry_z,exit_z,max_hold,days,active_days,positive_days,total_pnl,mean_day_pnl,min_day_pnl,total_trades,mean_hit_rate,min_hit_rate,avg_trade_pnl,median_trade_pnl,pnl_per_trade,one_day_dependency,robust_score
0,curved_triangle_vs_quads_logratio_mr,2500,2.0,0.00,2500,3,3,3,136780.0,45593.333333,170.0,9,0.666667,0.333333,15197.777778,16110.0,15197.777778,0.539772,62950.000000
1,curved_triangle_vs_quads_logratio_mr,5000,2.0,0.00,1500,3,3,3,106470.0,35490.000000,290.0,7,0.722222,0.500000,14518.333333,24050.0,15210.000000,0.545506,48390.000000
2,curved_triangle_vs_quads_logratio_mr,5000,2.0,0.00,2500,3,3,3,34830.0,11610.000000,480.0,6,0.666667,0.500000,5805.000000,3625.0,5805.000000,0.778065,7730.000000
3,curved_triangle_vs_quads_logratio_mr,1000,1.5,1.00,250,3,3,2,1373400.0,457800.000000,-171040.0,115,0.607530,0.404762,12557.485419,42795.0,11942.608696,0.703480,271493.333333
4,curved_triangle_vs_quads_logratio_mr,500,1.5,0.00,250,3,3,2,991010.0,330336.666667,-129300.0,94,0.585598,0.387097,10422.051157,35805.0,10542.659574,0.634867,241233.333333
5,curved_triangle_vs_quads_logratio_mr,2500,1.5,0.50,1000,3,3,2,698110.0,232703.333333,-27770.0,23,0.809524,0.571429,29470.211640,44520.0,30352.608696,0.529028,219193.333333
6,curved_triangle_vs_quads_logratio_mr,500,1.5,1.00,250,3,3,2,1061080.0,353693.333333,-298990.0,204,0.563088,0.413333,5910.295314,35940.0,5201.372549,0.699448,212606.666667
7,curved_triangle_vs_quads_logratio_mr,500,1.5,0.25,250,3,3,2,744260.0,248086.666667,-70290.0,126,0.577582,0.488889,6262.136364,37930.0,5906.825397,0.573697,211520.000000
8,curved_triangle_vs_quads_logratio_mr,500,1.5,0.50,250,3,3,2,875090.0,291696.666667,-109600.0,139,0.568889,0.460000,6625.181197,37930.0,6295.611511,0.644379,207466.666667
9,curved_triangle_vs_quads_logratio_mr,2500,1.5,0.25,500,3,3,2,674270.0,224756.666667,-49350.0,27,0.717593,0.375000,23391.194444,41180.0,24972.962963,0.541905,205920.000000



Top day breakdown for best config:


,candidate,day,window,entry_z,exit_z,max_hold,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl
552,curved_triangle_vs_quads_logratio_mr,2,2500,2.0,0.0,2500,3,62780.0,0.666667,20926.666667,69360.0
553,curved_triangle_vs_quads_logratio_mr,3,2500,2.0,0.0,2500,3,73830.0,1.000000,24610.000000,16110.0
554,curved_triangle_vs_quads_logratio_mr,4,2500,2.0,0.0,2500,3,170.0,0.333333,56.666667,-8810.0



Trades for best config:


,day,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_z,exit_z,hold,exec_pnl,candidate,window,max_hold
8753,2,2517,5017,251700,501700,1.0,2.0,0.0,2500,75040.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8754,2,5484,7984,548400,798400,-1.0,2.0,0.0,2500,-81620.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8755,2,8375,9999,837500,999900,1.0,2.0,0.0,1624,69360.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8756,3,3908,6408,390800,640800,1.0,2.0,0.0,2500,45510.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8757,3,7143,9643,714300,964300,1.0,2.0,0.0,2500,16110.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8758,3,9644,9999,964400,999900,1.0,2.0,0.0,355,12210.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8759,4,2512,5012,251200,501200,1.0,2.0,0.0,2500,-8810.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8760,4,5013,7513,501300,751300,1.0,2.0,0.0,2500,23630.0,curved_triangle_vs_quads_logratio_mr,2500,2500
8761,4,8785,9999,878500,999900,-1.0,2.0,0.0,1214,-14650.0,curved_triangle_vs_quads_logratio_mr,2500,2500



Runtime: 4.27s


In [21]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

# ============================================================
# MICROCHIPS: strict position backtest for top 5 viable signals
# One position at a time, best bid/ask execution, no extra fees.
# ============================================================

t0 = time.time()

PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

CIRCLE, OVAL, SQUARE, RECTANGLE, TRIANGLE = range(5)

# Expected inputs:
# mid_by_day[day] -> np array shape (N, 5)
# bid_by_day[day] -> np array shape (N, 5)
# ask_by_day[day] -> np array shape (N, 5)
# ts_by_day[day]  -> np array shape (N,)
#
# If your variable names differ, map them here.
required_names = ["mid_by_day", "bid_by_day", "ask_by_day"]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(
        f"Missing required variables: {missing}. "
        "Map your existing arrays to mid_by_day, bid_by_day, ask_by_day first."
    )

if "ts_by_day" not in globals():
    ts_by_day = {d: np.arange(len(mid_by_day[d])) * 100 for d in mid_by_day.keys()}

days = sorted(mid_by_day.keys())

print("Days:", days)
for d in days:
    print(f"Day {d}: mid={mid_by_day[d].shape}, bid={bid_by_day[d].shape}, ask={ask_by_day[d].shape}")


# -----------------------------
# Utility functions
# -----------------------------

def safe_log_ratio(num, den, eps=1e-9):
    return np.log(np.maximum(num, eps) / np.maximum(den, eps))


def rolling_zscore(x, window):
    """
    Rolling z-score using previous window only.
    No current-row lookahead.
    """
    s = pd.Series(x)
    mu = s.shift(1).rolling(window, min_periods=window).mean()
    sd = s.shift(1).rolling(window, min_periods=window).std(ddof=0)
    z = (s - mu) / sd.replace(0, np.nan)
    return z.to_numpy(dtype=float)


def rolling_poly_resid(y, x, degree=3, window=2500):
    """
    Rolling polynomial residual:
      residual[t] = y[t] - poly_fit(x[t])
    Fit uses only rows [t-window, t), so no lookahead.

    This is intentionally simple. N is only ~10k/day, so it is fine.
    """
    n = len(y)
    resid = np.full(n, np.nan, dtype=float)

    for t in range(window, n):
        xs = x[t-window:t]
        ys = y[t-window:t]

        ok = np.isfinite(xs) & np.isfinite(ys)
        if ok.sum() < degree + 2:
            continue

        try:
            coeff = np.polyfit(xs[ok], ys[ok], deg=degree)
            pred = np.polyval(coeff, x[t])
            resid[t] = y[t] - pred
        except np.linalg.LinAlgError:
            continue

    return resid


def open_close_pnl(q, entry_bid, entry_ask, exit_bid, exit_ask):
    """
    q is signed position vector.
      q > 0: buy at ask on entry, sell at bid on exit.
      q < 0: sell at bid on entry, buy at ask on exit.
    """
    q = np.asarray(q, dtype=float)

    entry_px = np.where(q > 0, entry_ask, entry_bid)
    exit_px = np.where(q > 0, exit_bid, exit_ask)

    return float(np.nansum(q * (exit_px - entry_px)))


def backtest_one_day_position(
    signal,
    bid,
    ask,
    timestamps,
    q_trade,
    window,
    entry_z,
    exit_z,
    max_hold,
    mode="meanrev",
):
    """
    One-position-at-a-time backtest.

    For mean reversion:
      z > +entry => short spread => side = -1
      z < -entry => long spread  => side = +1

    Position = side * q_trade.
    """
    z = rolling_zscore(signal, window)
    n = len(signal)

    trades = []
    i = 0

    while i < n - 1:
        zi = z[i]

        if not np.isfinite(zi) or abs(zi) < entry_z:
            i += 1
            continue

        if mode == "meanrev":
            side = -np.sign(zi)
        elif mode == "follow":
            side = np.sign(zi)
        elif mode == "inverse":
            side = -np.sign(zi)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        q = side * np.asarray(q_trade, dtype=float)

        entry_idx = i
        forced_exit_idx = min(n - 1, entry_idx + max_hold)

        exit_idx = forced_exit_idx
        for j in range(entry_idx + 1, forced_exit_idx + 1):
            zj = z[j]
            if np.isfinite(zj) and abs(zj) <= exit_z:
                exit_idx = j
                break

        pnl = open_close_pnl(
            q=q,
            entry_bid=bid[entry_idx],
            entry_ask=ask[entry_idx],
            exit_bid=bid[exit_idx],
            exit_ask=ask[exit_idx],
        )

        trades.append({
            "entry_idx": entry_idx,
            "exit_idx": exit_idx,
            "entry_ts": timestamps[entry_idx],
            "exit_ts": timestamps[exit_idx],
            "side": side,
            "entry_z_seen": zi,
            "exit_z_seen": z[exit_idx],
            "hold": exit_idx - entry_idx,
            "exec_pnl": pnl,
        })

        # Strict one-position logic: do not overlap.
        i = exit_idx + 1

    return trades


# -----------------------------
# Top 5 candidates to test
# -----------------------------

CANDIDATES = [
    {
        "candidate": "curved_triangle_vs_quads_logratio_mr",
        "family": "shape_family",
        "signal_type": "logratio",
        "mode": "meanrev",
        "note": "log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE))",
        "q_trade": np.array([10, 10, -10, -10, 10], dtype=float),
        "signal_func": lambda X, window: safe_log_ratio(
            X[:, CIRCLE] + X[:, OVAL] + X[:, TRIANGLE],
            X[:, SQUARE] + X[:, RECTANGLE],
        ),
    },
    {
        "candidate": "triangle_vs_quads_logratio_mr",
        "family": "triangle_vs_quads",
        "signal_type": "logratio",
        "mode": "meanrev",
        "note": "log(TRIANGLE / average(SQUARE, RECTANGLE))",
        "q_trade": np.array([0, 0, -5, -5, 10], dtype=float),
        "signal_func": lambda X, window: safe_log_ratio(
            X[:, TRIANGLE],
            0.5 * (X[:, SQUARE] + X[:, RECTANGLE]),
        ),
    },
    {
        "candidate": "square_triangle_logratio_mr",
        "family": "square_vs_triangle",
        "signal_type": "logratio",
        "mode": "meanrev",
        "note": "log(SQUARE/TRIANGLE)",
        "q_trade": np.array([0, 0, 10, 0, -10], dtype=float),
        "signal_func": lambda X, window: safe_log_ratio(
            X[:, SQUARE],
            X[:, TRIANGLE],
        ),
    },
    {
        "candidate": "quads_avg_triangle_cubic_resid_mr",
        "family": "quads_vs_triangle",
        "signal_type": "poly_resid",
        "mode": "meanrev",
        "note": "average(SQUARE, RECTANGLE) residual vs cubic function of TRIANGLE",
        "q_trade": np.array([0, 0, 5, 5, -10], dtype=float),
        "signal_func": lambda X, window: rolling_poly_resid(
            y=0.5 * (X[:, SQUARE] + X[:, RECTANGLE]),
            x=X[:, TRIANGLE],
            degree=3,
            window=window,
        ),
    },
    {
        "candidate": "oval_triangle_vs_rectangle_logratio_mr",
        "family": "stage1_interpretable",
        "signal_type": "logratio",
        "mode": "meanrev",
        "note": "log((OVAL+TRIANGLE)/RECTANGLE)",
        "q_trade": np.array([0, 10, 0, -10, 10], dtype=float),
        "signal_func": lambda X, window: safe_log_ratio(
            X[:, OVAL] + X[:, TRIANGLE],
            X[:, RECTANGLE],
        ),
    },
]


# -----------------------------
# Config grid
# -----------------------------

WINDOWS = [500, 1000, 2500, 5000]
ENTRY_ZS = [1.5, 2.0, 2.5, 3.0]
EXIT_ZS = [0.0, 0.25, 0.5, 1.0]
MAX_HOLDS = [250, 500, 1000, 1500, 2500]

total_configs = len(CANDIDATES) * len(WINDOWS) * len(ENTRY_ZS) * len(EXIT_ZS) * len(MAX_HOLDS)
print("Candidates:", len(CANDIDATES))
print("Total candidate/config combos:", total_configs)


# -----------------------------
# Run backtest
# -----------------------------

day_rows = []
trade_rows = []

cfg_i = 0
signal_cache = {}

for cand in CANDIDATES:
    cname = cand["candidate"]

    for window in WINDOWS:
        # Cache signal per candidate/day/window because poly residual depends on window.
        for d in days:
            key = (cname, d, window)
            if key not in signal_cache:
                X = np.asarray(mid_by_day[d], dtype=float)
                signal_cache[key] = cand["signal_func"](X, window)

        for entry_z in ENTRY_ZS:
            for exit_z in EXIT_ZS:
                for max_hold in MAX_HOLDS:
                    cfg_i += 1

                    if cfg_i == 1 or cfg_i % 50 == 0:
                        print(
                            f"[{time.time()-t0:7.2f}s] "
                            f"config {cfg_i}/{total_configs}: "
                            f"{cname}, window={window}, entry={entry_z}, exit={exit_z}, hold={max_hold}"
                        )

                    for d in days:
                        signal = signal_cache[(cname, d, window)]
                        bid = np.asarray(bid_by_day[d], dtype=float)
                        ask = np.asarray(ask_by_day[d], dtype=float)
                        ts = np.asarray(ts_by_day[d])

                        trades = backtest_one_day_position(
                            signal=signal,
                            bid=bid,
                            ask=ask,
                            timestamps=ts,
                            q_trade=cand["q_trade"],
                            window=window,
                            entry_z=entry_z,
                            exit_z=exit_z,
                            max_hold=max_hold,
                            mode=cand["mode"],
                        )

                        pnl_values = [tr["exec_pnl"] for tr in trades]
                        trade_count = len(pnl_values)
                        day_pnl = float(np.sum(pnl_values)) if trade_count else 0.0
                        hit_rate = float(np.mean(np.array(pnl_values) > 0)) if trade_count else np.nan
                        avg_trade_pnl = float(np.mean(pnl_values)) if trade_count else np.nan
                        median_trade_pnl = float(np.median(pnl_values)) if trade_count else np.nan

                        row_base = {
                            "candidate": cname,
                            "family": cand["family"],
                            "signal_type": cand["signal_type"],
                            "note": cand["note"],
                            "mode": cand["mode"],
                            "day": d,
                            "window": window,
                            "entry_z": entry_z,
                            "exit_z": exit_z,
                            "max_hold": max_hold,
                            "trade_count": trade_count,
                            "day_pnl": day_pnl,
                            "hit_rate": hit_rate,
                            "avg_trade_pnl": avg_trade_pnl,
                            "median_trade_pnl": median_trade_pnl,
                            "q_trade": cand["q_trade"].tolist(),
                        }
                        day_rows.append(row_base)

                        for tr in trades:
                            trade_rows.append({
                                **row_base,
                                **tr,
                            })


day_results = pd.DataFrame(day_rows)
trade_log = pd.DataFrame(trade_rows)

print("Day results:", day_results.shape)
print("Trade log:", trade_log.shape)


# -----------------------------
# Summary + robustness filter
# -----------------------------

group_cols = [
    "candidate",
    "family",
    "signal_type",
    "note",
    "mode",
    "window",
    "entry_z",
    "exit_z",
    "max_hold",
]

summary_rows = []

for keys, g in day_results.groupby(group_cols, dropna=False):
    total_pnl = float(g["day_pnl"].sum())
    mean_day_pnl = float(g["day_pnl"].mean())
    min_day_pnl = float(g["day_pnl"].min())
    max_day_pnl = float(g["day_pnl"].max())
    total_trades = int(g["trade_count"].sum())
    active_days = int((g["trade_count"] > 0).sum())
    positive_days = int((g["day_pnl"] > 0).sum())

    valid_hit = g["hit_rate"].dropna()
    mean_hit_rate = float(valid_hit.mean()) if len(valid_hit) else np.nan
    min_hit_rate = float(valid_hit.min()) if len(valid_hit) else np.nan

    pnl_per_trade = total_pnl / total_trades if total_trades else np.nan

    if total_pnl > 0:
        one_day_dependency = max_day_pnl / total_pnl
    else:
        one_day_dependency = np.inf

    robust_pass = (
        active_days == len(days)
        and positive_days == len(days)
        and min_day_pnl > 10_000
        and total_trades >= 6
        and one_day_dependency <= 0.65
    )

    # Score intentionally rewards robustness more than raw PnL.
    robust_score = (
        total_pnl
        + 2.0 * min_day_pnl
        - 0.5 * max(0, one_day_dependency - 0.50) * total_pnl
    )

    summary_rows.append({
        **dict(zip(group_cols, keys)),
        "days": len(g),
        "active_days": active_days,
        "positive_days": positive_days,
        "total_pnl": total_pnl,
        "mean_day_pnl": mean_day_pnl,
        "min_day_pnl": min_day_pnl,
        "max_day_pnl": max_day_pnl,
        "total_trades": total_trades,
        "mean_hit_rate": mean_hit_rate,
        "min_hit_rate": min_hit_rate,
        "pnl_per_trade": pnl_per_trade,
        "one_day_dependency": one_day_dependency,
        "robust_pass": robust_pass,
        "robust_score": robust_score,
    })

summary = pd.DataFrame(summary_rows)

summary_sorted = summary.sort_values(
    ["robust_pass", "robust_score", "total_pnl"],
    ascending=[False, False, False],
).reset_index(drop=True)

robust_only = summary_sorted[summary_sorted["robust_pass"]].reset_index(drop=True)


# -----------------------------
# Outputs
# -----------------------------

out_dir = Path("analysis_outputs/microchips_strict_position_top5")
out_dir.mkdir(parents=True, exist_ok=True)

day_results.to_csv(out_dir / "MICRO_STRICT_DAY_RESULTS.csv", index=False)
trade_log.to_csv(out_dir / "MICRO_STRICT_TRADE_LOG.csv", index=False)
summary_sorted.to_csv(out_dir / "MICRO_STRICT_SUMMARY.csv", index=False)
robust_only.to_csv(out_dir / "MICRO_STRICT_ROBUST_ONLY.csv", index=False)

print("\nSaved outputs to:", out_dir)
print(f"Runtime: {time.time()-t0:.2f} seconds")

print("\nTop MICRO_STRICT_SUMMARY:")
display(summary_sorted.head(50))

print("\nMICRO_STRICT_ROBUST_ONLY:")
display(robust_only.head(50))

if len(robust_only):
    best = robust_only.iloc[0]
    mask = np.ones(len(day_results), dtype=bool)
    for col in ["candidate", "window", "entry_z", "exit_z", "max_hold"]:
        mask &= day_results[col].eq(best[col])

    print("\nBest robust config day breakdown:")
    display(day_results.loc[mask].sort_values("day"))

    tmask = np.ones(len(trade_log), dtype=bool)
    for col in ["candidate", "window", "entry_z", "exit_z", "max_hold"]:
        tmask &= trade_log[col].eq(best[col])

    print("\nTrades for best robust config:")
    display(trade_log.loc[tmask].sort_values(["day", "entry_idx"]).head(100))
else:
    print("\nNo config passed the strict robustness filter.")
    print("Closest configs by robust_score are shown in MICRO_STRICT_SUMMARY above.")

Days: [np.int64(2), np.int64(3), np.int64(4)]
Day 2: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 3: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 4: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Candidates: 5
Total candidate/config combos: 1600
[   0.16s] config 1/1600: curved_triangle_vs_quads_logratio_mr, window=500, entry=1.5, exit=0.0, hold=250
[   0.79s] config 50/1600: curved_triangle_vs_quads_logratio_mr, window=500, entry=2.5, exit=0.25, hold=2500
[   1.41s] config 100/1600: curved_triangle_vs_quads_logratio_mr, window=1000, entry=1.5, exit=1.0, hold=2500
[   2.01s] config 150/1600: curved_triangle_vs_quads_logratio_mr, window=1000, entry=3.0, exit=0.25, hold=2500
[   2.61s] config 200/1600: curved_triangle_vs_quads_logratio_mr, window=2500, entry=2.0, exit=1.0, hold=2500
[   3.20s] config 250/1600: curved_triangle_vs_quads_logratio_mr, window=5000, entry=1.5, exit=0.25, hold=2500
[   3.78s] config 300/1600: curved_triangle_vs_quads_logratio_mr, window=5000, entry=

,candidate,family,signal_type,note,mode,window,entry_z,exit_z,max_hold,days,...,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,1000,2.5,0.00,1000,3,...,20100.000000,17385.0,23480.0,17,0.588889,0.500000,3547.058824,0.389386,True,95070.0
1,quads_avg_triangle_cubic_resid_mr,quads_vs_triangle,poly_resid,"average(SQUARE, RECTANGLE) residual vs cubic f...",meanrev,2500,2.0,0.00,2500,3,...,17606.666667,13495.0,22435.0,6,1.000000,1.000000,8803.333333,0.424744,True,79810.0
2,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,1000,2.0,0.00,1000,3,...,17776.666667,11420.0,25790.0,24,0.666667,0.625000,2222.083333,0.483593,True,76170.0
3,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,2500,1.5,0.00,1000,3,...,16410.000000,11800.0,25070.0,18,0.666667,0.500000,2735.000000,0.509242,True,72602.5
4,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,2500,1.5,0.00,2500,3,...,15410.000000,11210.0,18110.0,9,0.666667,0.666667,5136.666667,0.391737,True,68650.0
5,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,500,1.5,0.00,2500,3,...,14511.666667,10410.0,16570.0,12,0.666667,0.500000,3627.916667,0.380613,True,64355.0
6,quads_avg_triangle_cubic_resid_mr,quads_vs_triangle,poly_resid,"average(SQUARE, RECTANGLE) residual vs cubic f...",meanrev,2500,1.5,0.00,1500,3,...,13063.333333,10010.0,19010.0,10,0.694444,0.666667,3919.000000,0.485073,True,59210.0
7,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,1000,3.0,0.00,1500,3,...,12296.666667,10050.0,16010.0,7,0.888889,0.666667,5270.000000,0.433993,True,56990.0
8,curved_triangle_vs_quads_logratio_mr,shape_family,logratio,log((CIRCLE+OVAL+TRIANGLE)/(SQUARE+RECTANGLE)),meanrev,2500,2.5,0.00,500,3,...,-3.333333,-3230.0,2070.0,9,0.361111,0.250000,-1.111111,inf,False,inf
9,oval_triangle_vs_rectangle_logratio_mr,stage1_interpretable,logratio,log((OVAL+TRIANGLE)/RECTANGLE),meanrev,2500,3.0,0.25,250,3,...,-20.000000,-2700.0,3950.0,9,0.388889,0.000000,-6.666667,inf,False,inf



MICRO_STRICT_ROBUST_ONLY:


,candidate,family,signal_type,note,mode,window,entry_z,exit_z,max_hold,days,...,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,1000,2.5,0.0,1000,3,...,20100.000000,17385.0,23480.0,17,0.588889,0.500000,3547.058824,0.389386,True,95070.0
1,quads_avg_triangle_cubic_resid_mr,quads_vs_triangle,poly_resid,"average(SQUARE, RECTANGLE) residual vs cubic f...",meanrev,2500,2.0,0.0,2500,3,...,17606.666667,13495.0,22435.0,6,1.000000,1.000000,8803.333333,0.424744,True,79810.0
2,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,1000,2.0,0.0,1000,3,...,17776.666667,11420.0,25790.0,24,0.666667,0.625000,2222.083333,0.483593,True,76170.0
3,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,2500,1.5,0.0,1000,3,...,16410.000000,11800.0,25070.0,18,0.666667,0.500000,2735.000000,0.509242,True,72602.5
4,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,2500,1.5,0.0,2500,3,...,15410.000000,11210.0,18110.0,9,0.666667,0.666667,5136.666667,0.391737,True,68650.0
5,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,500,1.5,0.0,2500,3,...,14511.666667,10410.0,16570.0,12,0.666667,0.500000,3627.916667,0.380613,True,64355.0
6,quads_avg_triangle_cubic_resid_mr,quads_vs_triangle,poly_resid,"average(SQUARE, RECTANGLE) residual vs cubic f...",meanrev,2500,1.5,0.0,1500,3,...,13063.333333,10010.0,19010.0,10,0.694444,0.666667,3919.000000,0.485073,True,59210.0
7,square_triangle_logratio_mr,square_vs_triangle,logratio,log(SQUARE/TRIANGLE),meanrev,1000,3.0,0.0,1500,3,...,12296.666667,10050.0,16010.0,7,0.888889,0.666667,5270.000000,0.433993,True,56990.0



Best robust config day breakdown:


,candidate,family,signal_type,note,mode,day,window,entry_z,exit_z,max_hold,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl,q_trade
1326,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,6,17385.0,0.666667,2897.500000,2642.5,"[0.0, 0.0, -5.0, -5.0, 10.0]"
1327,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,3,1000,2.5,0.0,1000,6,23480.0,0.500000,3913.333333,1985.0,"[0.0, 0.0, -5.0, -5.0, 10.0]"
1328,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,4,1000,2.5,0.0,1000,5,19435.0,0.600000,3887.000000,5130.0,"[0.0, 0.0, -5.0, -5.0, 10.0]"



Trades for best robust config:


,candidate,family,signal_type,note,mode,day,window,entry_z,exit_z,max_hold,...,q_trade,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_z_seen,exit_z_seen,hold,exec_pnl
17225,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",1701,2701,170100,270100,-1.0,2.516924,-0.441498,1000,9120.0
17226,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",3175,4175,317500,417500,-1.0,2.682764,0.739344,1000,-1165.0
17227,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",4643,5643,464300,564300,1.0,-2.670577,1.738142,1000,5385.0
17228,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",5681,6681,568100,668100,-1.0,2.708747,-2.961475,1000,4560.0
17229,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",6682,7682,668200,768200,1.0,-2.914221,1.280970,1000,725.0
17230,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,2,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",9295,9999,929500,999900,-1.0,2.503122,-0.098440,704,-1240.0
17231,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,3,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",1289,2289,128900,228900,-1.0,2.706103,-1.541482,1000,9585.0
17232,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,3,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",3916,4916,391600,491600,1.0,-2.553229,-1.581073,1000,-2795.0
17233,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,3,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",5360,6360,536000,636000,-1.0,2.540700,1.535683,1000,-2480.0
17234,triangle_vs_quads_logratio_mr,triangle_vs_quads,logratio,"log(TRIANGLE / average(SQUARE, RECTANGLE))",meanrev,3,1000,2.5,0.0,1000,...,"[0.0, 0.0, -5.0, -5.0, 10.0]",6379,7379,637900,737900,-1.0,2.527621,-2.084582,1000,4425.0


In [18]:
import numpy as np
import pandas as pd

array_dict_candidates = []

for name, obj in list(globals().items()):   # <- key fix
    if isinstance(obj, dict):
        vals = list(obj.values())
        if not vals:
            continue

        good_vals = []
        for v in vals:
            try:
                arr = np.asarray(v)
                if arr.ndim == 2 and arr.shape[1] == 5:
                    good_vals.append(arr.shape)
            except Exception:
                pass

        if len(good_vals) >= 3:
            array_dict_candidates.append((name, list(obj.keys())[:5], good_vals[:3]))

print("Possible day -> 5-product array dicts:")
for name, keys, shapes in array_dict_candidates:
    print(f"{name}: keys={keys}, shapes={shapes}")

Possible day -> 5-product array dicts:
FLOW_DAY: keys=[2, 3, 4], shapes=[(10000, 5), (10000, 5), (10000, 5)]


In [19]:
import numpy as np
import pandas as pd

PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

# 1) Find likely market/orderbook DataFrames
df_candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)
        has_product = any(c in cols for c in ["product", "symbol"])
        has_time = any(c in cols for c in ["timestamp", "time"])
        has_bid = any("bid_price" in c or c == "bid" for c in cols)
        has_ask = any("ask_price" in c or c == "ask" for c in cols)

        if has_product and has_time and has_bid and has_ask:
            df_candidates.append(name)
            print(f"\n{name}: shape={obj.shape}")
            print(list(obj.columns))

print("\nLikely market dfs:", df_candidates)


prices: shape=(1500000, 19)
['day', 'timestamp', 'product', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss', 'file_day', 'global_ts']

prices_f: shape=(1500000, 31)
['day', 'timestamp', 'product', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss', 'file_day', 'global_ts', 'mid', 'spread', 'bid_depth_l1', 'ask_depth_l1', 'bid_depth_total', 'ask_depth_total', 'depth_total', 'imbalance_l1', 'imbalance_total', 'microprice_l1', 'microprice_edge', 'microprice_edge_norm']

micro_prices: shape=(150000, 6)
['file_day', 'timestamp', 'product', 'bid_price_1', 'ask_price_1', 'mid_price']

micro_trade_q: shape=(2845, 15)
['timestamp', 'buyer', 'seller', 'product', 'cu

In [20]:
market_df = px.copy()   # or micro_prices.copy()

PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]

day_col = "file_day" if "file_day" in market_df.columns else "day"
time_col = "timestamp"

DAYS = sorted(market_df[day_col].unique())

bid_by_day = {}
ask_by_day = {}
mid_by_day = {}

for day in DAYS:
    d = market_df[market_df[day_col] == day].copy()

    bid_pivot = (
        d.pivot(index=time_col, columns="product", values="bid_price_1")
         .sort_index()[PRODUCTS]
    )

    ask_pivot = (
        d.pivot(index=time_col, columns="product", values="ask_price_1")
         .sort_index()[PRODUCTS]
    )

    mid_pivot = (
        d.pivot(index=time_col, columns="product", values="mid_price")
         .sort_index()[PRODUCTS]
    )

    bid_by_day[day] = bid_pivot.to_numpy(float)
    ask_by_day[day] = ask_pivot.to_numpy(float)
    mid_by_day[day] = mid_pivot.to_numpy(float)

print("Built arrays:")
print("bid_by_day:", {k: v.shape for k, v in bid_by_day.items()})
print("ask_by_day:", {k: v.shape for k, v in ask_by_day.items()})
print("mid_by_day:", {k: v.shape for k, v in mid_by_day.items()})

Built arrays:
bid_by_day: {np.int64(2): (10000, 5), np.int64(3): (10000, 5), np.int64(4): (10000, 5)}
ask_by_day: {np.int64(2): (10000, 5), np.int64(3): (10000, 5), np.int64(4): (10000, 5)}
mid_by_day: {np.int64(2): (10000, 5), np.int64(3): (10000, 5), np.int64(4): (10000, 5)}


In [22]:
import numpy as np
import pandas as pd
import itertools, time, math
from collections import defaultdict

# ============================================================
# MICROCHIPS PORTFOLIO STRICT POSITION BACKTEST
# Uses: mid_by_day, bid_by_day, ask_by_day from your previous cell
# Products order expected:
# ['MICROCHIP_CIRCLE','MICROCHIP_OVAL','MICROCHIP_SQUARE','MICROCHIP_RECTANGLE','MICROCHIP_TRIANGLE']
# ============================================================

PRODUCTS = ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE', 'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE']
P_CIRCLE, P_OVAL, P_SQUARE, P_RECT, P_TRI = range(5)

OUTDIR = "analysis_outputs/microchips_portfolio_top3"
import os
os.makedirs(OUTDIR, exist_ok=True)

# Conservative default; adjust if actual limit differs
POSITION_LIMIT = 100

# Each unit of q_trade below is actual position units.
# q_trade [0,0,-5,-5,+10] means +10 triangle, -5 square, -5 rectangle.
CANDIDATES = [
    {
        "name": "triangle_vs_quads_logratio_mr",
        "family": "triangle_vs_quads",
        "q_trade": np.array([0, 0, -5, -5, 10], dtype=float),
        "type": "logratio",
        "params": {"window": 1000, "entry_z": 2.5, "exit_z": 0.0, "max_hold": 1000},
    },
    {
        "name": "quads_avg_triangle_cubic_resid_mr",
        "family": "quads_vs_triangle",
        "q_trade": np.array([0, 0, 5, 5, -10], dtype=float),
        "type": "cubic_resid",
        "params": {"window": 2500, "entry_z": 2.0, "exit_z": 0.0, "max_hold": 2500},
    },
    {
        "name": "square_triangle_logratio_mr",
        "family": "square_vs_triangle",
        "q_trade": np.array([0, 0, 10, 0, -10], dtype=float),
        "type": "logratio_square_triangle",
        "params": {"window": 1000, "entry_z": 2.0, "exit_z": 0.0, "max_hold": 1000},
    },
]

def rolling_mean_std(x, window):
    s = pd.Series(x)
    mu = s.rolling(window, min_periods=window).mean().to_numpy()
    sd = s.rolling(window, min_periods=window).std(ddof=0).to_numpy()
    sd = np.where(sd <= 1e-12, np.nan, sd)
    return mu, sd

def fit_cubic_resid(y, x, window):
    """
    Rolling cubic residual y - f(x).
    More efficient enough for 10k rows; fits on trailing window only.
    """
    n = len(y)
    resid = np.full(n, np.nan)
    for i in range(window, n):
        xs = x[i-window:i]
        ys = y[i-window:i]
        if np.any(~np.isfinite(xs)) or np.any(~np.isfinite(ys)):
            continue
        coefs = np.polyfit(xs, ys, 3)
        pred = np.polyval(coefs, x[i])
        resid[i] = y[i] - pred
    return resid

def build_signal(mid, cand):
    eps = 1e-9
    typ = cand["type"]
    w = cand["params"]["window"]

    circle = mid[:, P_CIRCLE]
    oval   = mid[:, P_OVAL]
    square = mid[:, P_SQUARE]
    rect   = mid[:, P_RECT]
    tri    = mid[:, P_TRI]

    if typ == "logratio":
        # log(TRIANGLE / average(SQUARE, RECTANGLE))
        raw = np.log((tri + eps) / ((square + rect) / 2 + eps))

    elif typ == "logratio_square_triangle":
        # log(SQUARE / TRIANGLE)
        raw = np.log((square + eps) / (tri + eps))

    elif typ == "cubic_resid":
        # average(SQUARE, RECTANGLE) residual vs cubic(TRIANGLE)
        quads_avg = (square + rect) / 2
        raw = fit_cubic_resid(quads_avg, tri, w)

    else:
        raise ValueError(f"Unknown signal type: {typ}")

    mu, sd = rolling_mean_std(raw, w)
    z = (raw - mu) / sd
    return raw, z

def can_add_position(current_pos, delta, limit=POSITION_LIMIT):
    new_pos = current_pos + delta
    return np.all(np.abs(new_pos) <= limit + 1e-9)

def execution_pnl_for_leg(side, q_trade, entry_bid, entry_ask, exit_bid, exit_ask):
    """
    side = +1 means enter +q_trade basket.
    For positive qty: buy at ask, sell at bid.
    For negative qty: sell at bid, buy back at ask.
    """
    qty = side * q_trade
    pnl = 0.0

    for j, q in enumerate(qty):
        if abs(q) < 1e-12:
            continue

        if q > 0:
            entry_px = entry_ask[j]
            exit_px = exit_bid[j]
            pnl += q * (exit_px - entry_px)
        else:
            q_abs = -q
            entry_px = entry_bid[j]
            exit_px = exit_ask[j]
            pnl += q_abs * (entry_px - exit_px)

    return pnl

def backtest_portfolio_for_day(day, active_candidates, mid, bid, ask):
    n = len(mid)
    signals = {}

    for cand in active_candidates:
        _, z = build_signal(mid, cand)
        signals[cand["name"]] = z

    open_trades = []
    trade_log = []
    current_pos = np.zeros(5, dtype=float)

    for t in range(n):
        # 1) Exit open trades first
        still_open = []
        for tr in open_trades:
            cand = tr["candidate_obj"]
            z = signals[cand["name"]][t]
            exit_z = cand["params"]["exit_z"]
            max_hold = cand["params"]["max_hold"]

            should_exit = False
            if np.isfinite(z):
                # Mean reversion exit: z crosses back towards/through exit band
                if tr["side"] == -1 and z <= exit_z:
                    should_exit = True
                elif tr["side"] == +1 and z >= -exit_z:
                    should_exit = True

            if (t - tr["entry_idx"]) >= max_hold:
                should_exit = True

            if t == n - 1:
                should_exit = True

            if should_exit:
                pnl = execution_pnl_for_leg(
                    tr["side"], cand["q_trade"],
                    tr["entry_bid"], tr["entry_ask"],
                    bid[t], ask[t]
                )
                current_pos -= tr["side"] * cand["q_trade"]

                trade_log.append({
                    "day": day,
                    "candidate": cand["name"],
                    "entry_idx": tr["entry_idx"],
                    "exit_idx": t,
                    "entry_ts": tr["entry_idx"] * 100,
                    "exit_ts": t * 100,
                    "side": tr["side"],
                    "entry_z": tr["entry_z"],
                    "exit_z": z,
                    "hold": t - tr["entry_idx"],
                    "exec_pnl": pnl,
                    "position_after_exit": current_pos.copy().tolist(),
                })
            else:
                still_open.append(tr)

        open_trades = still_open

        # 2) Entries after exits
        for cand in active_candidates:
            name = cand["name"]
            z = signals[name][t]
            if not np.isfinite(z):
                continue

            # Do not open duplicate same candidate while already open
            if any(tr["candidate"] == name for tr in open_trades):
                continue

            entry_z = cand["params"]["entry_z"]

            # mean reversion:
            # z high => basket rich => short q_trade basket: side=-1
            # z low  => basket cheap => long q_trade basket: side=+1
            side = 0
            if z >= entry_z:
                side = -1
            elif z <= -entry_z:
                side = +1

            if side == 0:
                continue

            delta_pos = side * cand["q_trade"]
            if not can_add_position(current_pos, delta_pos):
                continue

            current_pos += delta_pos
            open_trades.append({
                "candidate": name,
                "candidate_obj": cand,
                "entry_idx": t,
                "entry_z": z,
                "side": side,
                "entry_bid": bid[t].copy(),
                "entry_ask": ask[t].copy(),
            })

    day_pnl = sum(x["exec_pnl"] for x in trade_log)
    return day_pnl, trade_log

def summarize_trade_log(trades, combo_name):
    if len(trades) == 0:
        return None

    df = pd.DataFrame(trades)
    by_day = df.groupby("day")["exec_pnl"].sum()
    hit_by_day = df.groupby("day")["exec_pnl"].apply(lambda x: (x > 0).mean())

    return {
        "combo": combo_name,
        "days": len(by_day),
        "positive_days": int((by_day > 0).sum()),
        "total_pnl": float(df["exec_pnl"].sum()),
        "mean_day_pnl": float(by_day.mean()),
        "min_day_pnl": float(by_day.min()),
        "max_day_pnl": float(by_day.max()),
        "total_trades": int(len(df)),
        "mean_hit_rate": float(hit_by_day.mean()),
        "min_hit_rate": float(hit_by_day.min()),
        "pnl_per_trade": float(df["exec_pnl"].mean()),
        "one_day_dependency": float(by_day.max() / max(abs(df["exec_pnl"].sum()), 1e-9)),
    }

# ============================================================
# Run all useful combinations
# ============================================================

days = sorted(mid_by_day.keys())
combo_results = []
all_day_rows = []
all_trades = []

start = time.time()

# individual, pairs, and all 3
candidate_indices = list(range(len(CANDIDATES)))
combos = []
for r in [1, 2, 3]:
    combos.extend(list(itertools.combinations(candidate_indices, r)))

print(f"Days: {days}")
print(f"Testing {len(combos)} portfolio combinations")

for ci, combo in enumerate(combos, 1):
    active = [CANDIDATES[i] for i in combo]
    combo_name = " + ".join(c["name"] for c in active)
    print(f"[{time.time()-start:7.2f}s] combo {ci}/{len(combos)}: {combo_name}")

    combo_trades = []
    for day in days:
        pnl, trades = backtest_portfolio_for_day(
            day=day,
            active_candidates=active,
            mid=mid_by_day[day],
            bid=bid_by_day[day],
            ask=ask_by_day[day],
        )

        all_day_rows.append({
            "combo": combo_name,
            "day": day,
            "day_pnl": pnl,
            "trade_count": len(trades),
        })

        for tr in trades:
            tr["combo"] = combo_name
        combo_trades.extend(trades)
        all_trades.extend(trades)

    summary = summarize_trade_log(combo_trades, combo_name)
    if summary is not None:
        combo_results.append(summary)

summary_df = pd.DataFrame(combo_results)
day_df = pd.DataFrame(all_day_rows)
trade_df = pd.DataFrame(all_trades)

if len(summary_df):
    summary_df["robust_pass"] = (
        (summary_df["positive_days"] == summary_df["days"]) &
        (summary_df["min_day_pnl"] > 0)
    )
    summary_df["robust_score"] = (
        summary_df["total_pnl"]
        + 2.0 * summary_df["min_day_pnl"]
        - 0.5 * summary_df["max_day_pnl"] * summary_df["one_day_dependency"]
    )

    summary_df = summary_df.sort_values(
        ["robust_pass", "robust_score", "total_pnl"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

print("\nTop PORTFOLIO_SUMMARY:")
display(summary_df.head(20))

print("\nDay breakdown for best combo:")
best_combo = summary_df.iloc[0]["combo"]
display(day_df[day_df["combo"] == best_combo].sort_values("day"))

print("\nTrades for best combo:")
display(trade_df[trade_df["combo"] == best_combo].sort_values(["day", "entry_idx"]).head(100))

summary_df.to_csv(f"{OUTDIR}/portfolio_summary.csv", index=False)
day_df.to_csv(f"{OUTDIR}/portfolio_day_breakdown.csv", index=False)
trade_df.to_csv(f"{OUTDIR}/portfolio_trade_log.csv", index=False)

print(f"\nSaved outputs to: {OUTDIR}")
print(f"Runtime: {time.time()-start:.2f} seconds")

Days: [np.int64(2), np.int64(3), np.int64(4)]
Testing 7 portfolio combinations
[   0.00s] combo 1/7: triangle_vs_quads_logratio_mr
[   0.05s] combo 2/7: quads_avg_triangle_cubic_resid_mr
[   2.65s] combo 3/7: square_triangle_logratio_mr
[   2.67s] combo 4/7: triangle_vs_quads_logratio_mr + quads_avg_triangle_cubic_resid_mr
[   5.33s] combo 5/7: triangle_vs_quads_logratio_mr + square_triangle_logratio_mr
[   5.38s] combo 6/7: quads_avg_triangle_cubic_resid_mr + square_triangle_logratio_mr
[   8.07s] combo 7/7: triangle_vs_quads_logratio_mr + quads_avg_triangle_cubic_resid_mr + square_triangle_logratio_mr

Top PORTFOLIO_SUMMARY:


,combo,days,positive_days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,triangle_vs_quads_logratio_mr + quads_avg_tria...,3,3,34965.0,11655.000000,2405.0,28395.0,76,0.699321,0.629630,460.065789,0.812098,True,28245.241313
1,triangle_vs_quads_logratio_mr,3,3,27385.0,9128.333333,2940.0,18295.0,27,0.730556,0.625000,1014.259259,0.668066,True,27153.862060
2,triangle_vs_quads_logratio_mr + square_triangl...,3,2,40345.0,13448.333333,-2730.0,27685.0,61,0.736508,0.700000,661.393443,0.686206,False,25386.186950
3,triangle_vs_quads_logratio_mr + quads_avg_tria...,3,2,22005.0,7335.000000,-6835.0,19005.0,42,0.670452,0.533333,523.928571,0.863667,False,128.001022
4,square_triangle_logratio_mr,3,2,12960.0,4320.000000,-5670.0,9390.0,34,0.734848,0.727273,381.176471,0.724537,False,-1781.701389
5,quads_avg_triangle_cubic_resid_mr + square_tri...,3,2,7580.0,2526.666667,-3745.0,10100.0,49,0.677193,0.631579,154.693878,1.332454,False,-6638.891821
6,quads_avg_triangle_cubic_resid_mr,3,2,-5380.0,-1793.333333,-12985.0,6895.0,15,0.559524,0.428571,-358.666667,1.281599,False,-35768.310874



Day breakdown for best combo:


,combo,day,day_pnl,trade_count
18,triangle_vs_quads_logratio_mr + quads_avg_tria...,2,2405.0,27
19,triangle_vs_quads_logratio_mr + quads_avg_tria...,3,28395.0,25
20,triangle_vs_quads_logratio_mr + quads_avg_tria...,4,4165.0,24



Trades for best combo:


,day,candidate,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_z,exit_z,hold,exec_pnl,position_after_exit,combo
228,2,square_triangle_logratio_mr,1103,1339,110300,133900,-1,2.016438,-0.014010,236,3300.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
229,2,square_triangle_logratio_mr,1470,1804,147000,180400,1,-2.139108,0.015734,334,3140.0,"[0.0, 0.0, 5.0, 5.0, -10.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
230,2,triangle_vs_quads_logratio_mr,1701,1811,170100,181100,-1,2.511452,-0.162868,110,2955.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
232,2,square_triangle_logratio_mr,1843,2809,184300,280900,-1,2.032937,-0.161655,966,-7130.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
231,2,triangle_vs_quads_logratio_mr,1953,2746,195300,274600,1,-2.546936,0.131294,793,-1295.0,"[0.0, 0.0, -10.0, 0.0, 10.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,4,square_triangle_logratio_mr,8595,9050,859500,905000,1,-2.170465,0.008262,455,1840.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
299,4,triangle_vs_quads_logratio_mr,8668,8902,866800,890200,-1,2.530804,-0.041536,234,2980.0,"[0.0, 0.0, 10.0, 0.0, -10.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
302,4,square_triangle_logratio_mr,9572,9720,957200,972000,-1,2.182137,-0.113385,148,1510.0,"[0.0, 0.0, 0.0, 0.0, 0.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...
301,4,quads_avg_triangle_cubic_resid_mr,9596,9719,959600,971900,-1,2.026274,-0.010311,123,1355.0,"[0.0, 0.0, -10.0, 0.0, 10.0]",triangle_vs_quads_logratio_mr + quads_avg_tria...



Saved outputs to: analysis_outputs/microchips_portfolio_top3
Runtime: 10.76 seconds


In [23]:
import numpy as np
import pandas as pd
import time
from pathlib import Path

t0 = time.time()

PRODUCTS = [
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",
]
CIRCLE, OVAL, SQUARE, RECTANGLE, TRIANGLE = range(5)

# ------------------------------------------------------------
# 1) Build/normalise mid/bid/ask arrays if needed
# ------------------------------------------------------------

def build_price_arrays_from_df(df):
    day_col = "file_day" if "file_day" in df.columns else "day"
    out_mid, out_bid, out_ask = {}, {}, {}

    use = df[df["product"].isin(PRODUCTS)].copy()
    for d in sorted(use[day_col].unique()):
        dd = use[use[day_col] == d].copy()

        mid = dd.pivot(index="timestamp", columns="product", values="mid_price").sort_index()
        bid = dd.pivot(index="timestamp", columns="product", values="bid_price_1").sort_index()
        ask = dd.pivot(index="timestamp", columns="product", values="ask_price_1").sort_index()

        mid = mid.reindex(columns=PRODUCTS)
        bid = bid.reindex(columns=PRODUCTS)
        ask = ask.reindex(columns=PRODUCTS)

        out_mid[int(d)] = mid.to_numpy(float)
        out_bid[int(d)] = bid.to_numpy(float)
        out_ask[int(d)] = ask.to_numpy(float)

    return out_mid, out_bid, out_ask


if not all(name in globals() for name in ["mid_by_day", "bid_by_day", "ask_by_day"]):
    if "px" in globals():
        mid_by_day, bid_by_day, ask_by_day = build_price_arrays_from_df(px)
    elif "micro_prices" in globals():
        mid_by_day, bid_by_day, ask_by_day = build_price_arrays_from_df(micro_prices)
    else:
        raise NameError("Need mid_by_day/bid_by_day/ask_by_day, or px/micro_prices dataframe.")

mid_by_day = {int(k): np.asarray(v, float) for k, v in mid_by_day.items()}
bid_by_day = {int(k): np.asarray(v, float) for k, v in bid_by_day.items()}
ask_by_day = {int(k): np.asarray(v, float) for k, v in ask_by_day.items()}

DAYS = sorted(mid_by_day.keys())
print("Days:", DAYS)
for d in DAYS:
    print(f"Day {d}: mid={mid_by_day[d].shape}, bid={bid_by_day[d].shape}, ask={ask_by_day[d].shape}")


# ------------------------------------------------------------
# 2) Signal helpers
# ------------------------------------------------------------

def rolling_z(x, window):
    s = pd.Series(np.asarray(x, float))
    mu = s.rolling(window, min_periods=window).mean().shift(1)
    sd = s.rolling(window, min_periods=window).std(ddof=0).shift(1)
    z = (s - mu) / sd.replace(0, np.nan)
    return z.to_numpy(float)


def rolling_cubic_residual(x, y, fit_window):
    """
    Rolling residual y - cubic(x), fit on previous fit_window rows.
    Uses globally-normalised x for numerical stability.
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    n = len(x)
    out = np.full(n, np.nan)

    x_mu = np.nanmean(x)
    x_sd = np.nanstd(x)
    if not np.isfinite(x_sd) or x_sd == 0:
        return out

    xn = (x - x_mu) / x_sd

    powers = [np.ones(n)]
    for k in range(1, 7):
        powers.append(powers[-1] * xn)

    c_pow = [np.r_[0.0, np.cumsum(p)] for p in powers]
    c_y = [np.r_[0.0, np.cumsum((xn ** k) * y)] for k in range(4)]

    for t in range(fit_window, n):
        lo, hi = t - fit_window, t

        S = np.array([c_pow[k][hi] - c_pow[k][lo] for k in range(7)])
        Sy = np.array([c_y[k][hi] - c_y[k][lo] for k in range(4)])

        A = np.array([
            [S[0], S[1], S[2], S[3]],
            [S[1], S[2], S[3], S[4]],
            [S[2], S[3], S[4], S[5]],
            [S[3], S[4], S[5], S[6]],
        ])

        try:
            beta = np.linalg.solve(A, Sy)
        except np.linalg.LinAlgError:
            beta = np.linalg.lstsq(A, Sy, rcond=None)[0]

        xt = xn[t]
        pred = beta[0] + beta[1]*xt + beta[2]*(xt**2) + beta[3]*(xt**3)
        out[t] = y[t] - pred

    return out


def make_signals_for_day(mid):
    circle = mid[:, CIRCLE]
    oval = mid[:, OVAL]
    square = mid[:, SQUARE]
    rect = mid[:, RECTANGLE]
    tri = mid[:, TRIANGLE]
    quads_avg = 0.5 * (square + rect)

    # Main/base signal: TRIANGLE vs average(SQUARE, RECTANGLE)
    base_raw = np.log(tri / quads_avg)
    base_z = rolling_z(base_raw, window=1000)

    # Confirmation 1: SQUARE vs TRIANGLE
    square_tri_raw = np.log(square / tri)
    square_tri_z = rolling_z(square_tri_raw, window=1000)

    # Confirmation 2: average quadrilateral residual vs cubic(TRIANGLE)
    cubic_resid_raw = rolling_cubic_residual(tri, quads_avg, fit_window=2500)
    cubic_resid_z = rolling_z(cubic_resid_raw, window=2500)

    return {
        "base_triangle_vs_quads": base_z,
        "confirm_square_triangle": square_tri_z,
        "confirm_quads_cubic": cubic_resid_z,
    }


signals_by_day = {}
for d in DAYS:
    signals_by_day[d] = make_signals_for_day(mid_by_day[d])


# ------------------------------------------------------------
# 3) Strict position backtest with confirmation gates
# ------------------------------------------------------------

BASE_Q = np.array([0.0, 0.0, -5.0, -5.0, 10.0])
SQUARE_TRI_Q = np.array([0.0, 0.0, 10.0, 0.0, -10.0])
QUADS_CUBIC_Q = np.array([0.0, 0.0, 5.0, 5.0, -10.0])

BASE_ENTRY_Z = 2.5
BASE_EXIT_Z = 0.0
BASE_MAX_HOLD = 1000


def exec_cash(delta_pos, idx, bid, ask):
    delta_pos = np.asarray(delta_pos, float)
    b = bid[idx]
    a = ask[idx]

    # positive delta = buy at ask, negative delta = sell at bid
    cash = np.where(
        delta_pos > 0,
        -delta_pos * a,
        np.where(delta_pos < 0, -delta_pos * b, 0.0)
    )
    return float(np.nansum(cash))


def confirm_status(confirm_z, confirm_q, base_pos, thresh):
    if not np.isfinite(confirm_z) or abs(confirm_z) < thresh:
        return 0

    confirm_side = -np.sign(confirm_z)
    confirm_pos = confirm_side * confirm_q
    dot = float(np.dot(confirm_pos, base_pos))

    if dot > 1e-9:
        return 1
    if dot < -1e-9:
        return -1
    return 0


def gate_ok(mode, square_status, cubic_status):
    if mode == "base_only":
        return True
    if mode == "require_square_agree":
        return square_status == 1
    if mode == "require_cubic_agree":
        return cubic_status == 1
    if mode == "require_either_agree":
        return square_status == 1 or cubic_status == 1
    if mode == "require_both_agree":
        return square_status == 1 and cubic_status == 1
    if mode == "veto_square_conflict":
        return square_status != -1
    if mode == "veto_cubic_conflict":
        return cubic_status != -1
    if mode == "veto_any_conflict":
        return square_status != -1 and cubic_status != -1
    if mode == "either_agree_no_conflict":
        return (square_status == 1 or cubic_status == 1) and square_status != -1 and cubic_status != -1
    raise ValueError(f"Unknown gate mode: {mode}")


gate_specs = [("base_only", np.nan)]

for th in [0.0, 0.5, 1.0, 1.5, 2.0]:
    for mode in [
        "require_square_agree",
        "require_cubic_agree",
        "require_either_agree",
        "require_both_agree",
        "veto_square_conflict",
        "veto_cubic_conflict",
        "veto_any_conflict",
        "either_agree_no_conflict",
    ]:
        gate_specs.append((mode, th))

print("Gate specs:", len(gate_specs))


def run_day_gate(day, mode, confirm_thresh):
    mid = mid_by_day[day]
    bid = bid_by_day[day]
    ask = ask_by_day[day]

    z_base = signals_by_day[day]["base_triangle_vs_quads"]
    z_square = signals_by_day[day]["confirm_square_triangle"]
    z_cubic = signals_by_day[day]["confirm_quads_cubic"]

    n = len(mid)
    trades = []
    t = 0

    while t < n:
        bz = z_base[t]

        if not np.isfinite(bz) or abs(bz) < BASE_ENTRY_Z:
            t += 1
            continue

        base_side = -np.sign(bz)
        base_pos = base_side * BASE_Q

        if mode == "base_only":
            sq_status = 0
            cu_status = 0
        else:
            sq_status = confirm_status(z_square[t], SQUARE_TRI_Q, base_pos, confirm_thresh)
            cu_status = confirm_status(z_cubic[t], QUADS_CUBIC_Q, base_pos, confirm_thresh)

        if not gate_ok(mode, sq_status, cu_status):
            t += 1
            continue

        entry_idx = t
        entry_cash = exec_cash(base_pos, entry_idx, bid, ask)

        max_exit = min(n - 1, entry_idx + BASE_MAX_HOLD)
        exit_idx = max_exit

        for u in range(entry_idx + 1, max_exit + 1):
            zu = z_base[u]
            if np.isfinite(zu) and base_side * zu >= -BASE_EXIT_Z:
                exit_idx = u
                break

        exit_cash = exec_cash(-base_pos, exit_idx, bid, ask)
        pnl = entry_cash + exit_cash

        trades.append({
            "day": day,
            "gate_mode": mode,
            "confirm_threshold": confirm_thresh,
            "entry_idx": entry_idx,
            "exit_idx": exit_idx,
            "entry_ts": entry_idx * 100,
            "exit_ts": exit_idx * 100,
            "side": base_side,
            "entry_base_z": bz,
            "exit_base_z": z_base[exit_idx],
            "square_z": z_square[entry_idx],
            "cubic_z": z_cubic[entry_idx],
            "square_status": sq_status,
            "cubic_status": cu_status,
            "hold": exit_idx - entry_idx,
            "exec_pnl": pnl,
            "q_trade": BASE_Q.tolist(),
            "position": base_pos.tolist(),
        })

        t = exit_idx + 1

    return trades


all_day_rows = []
all_trades = []

for i, (mode, th) in enumerate(gate_specs, 1):
    print(f"[{time.time()-t0:7.2f}s] gate {i}/{len(gate_specs)}: {mode}, th={th}")

    for d in DAYS:
        trades = run_day_gate(d, mode, th)
        all_trades.extend(trades)

        pnls = [tr["exec_pnl"] for tr in trades]
        all_day_rows.append({
            "gate_mode": mode,
            "confirm_threshold": th,
            "day": d,
            "trade_count": len(trades),
            "day_pnl": float(np.sum(pnls)) if pnls else 0.0,
            "hit_rate": float(np.mean(np.array(pnls) > 0)) if pnls else np.nan,
            "avg_trade_pnl": float(np.mean(pnls)) if pnls else np.nan,
            "median_trade_pnl": float(np.median(pnls)) if pnls else np.nan,
        })

day_results = pd.DataFrame(all_day_rows)
trade_log = pd.DataFrame(all_trades)


# ------------------------------------------------------------
# 4) Summary + output
# ------------------------------------------------------------

summary_rows = []

for (mode, th), g in day_results.groupby(["gate_mode", "confirm_threshold"], dropna=False):
    days = len(g)
    active_days = int((g["trade_count"] > 0).sum())
    positive_days = int((g["day_pnl"] > 0).sum())
    total_pnl = float(g["day_pnl"].sum())
    total_trades = int(g["trade_count"].sum())
    min_day_pnl = float(g["day_pnl"].min())
    max_day_pnl = float(g["day_pnl"].max())

    if total_trades > 0:
        pnl_per_trade = total_pnl / total_trades
    else:
        pnl_per_trade = np.nan

    if total_pnl > 0:
        one_day_dependency = max_day_pnl / total_pnl
    else:
        one_day_dependency = np.inf

    robust_pass = positive_days == days and active_days == days and min_day_pnl > 0

    # simple score: reward total and min-day robustness, penalise over-dependence and trade count
    robust_score = (
        total_pnl
        + 2.0 * min_day_pnl
        - 0.25 * max_day_pnl
        - 25.0 * total_trades
    )

    summary_rows.append({
        "gate_mode": mode,
        "confirm_threshold": th,
        "days": days,
        "active_days": active_days,
        "positive_days": positive_days,
        "total_pnl": total_pnl,
        "mean_day_pnl": float(g["day_pnl"].mean()),
        "min_day_pnl": min_day_pnl,
        "max_day_pnl": max_day_pnl,
        "total_trades": total_trades,
        "mean_hit_rate": float(g["hit_rate"].mean(skipna=True)),
        "min_hit_rate": float(g["hit_rate"].min(skipna=True)),
        "pnl_per_trade": pnl_per_trade,
        "one_day_dependency": one_day_dependency,
        "robust_pass": robust_pass,
        "robust_score": robust_score,
    })

summary = pd.DataFrame(summary_rows)

summary_sorted = summary.sort_values(
    ["robust_pass", "total_pnl", "min_day_pnl", "pnl_per_trade"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

robust_only = summary_sorted[summary_sorted["robust_pass"]].reset_index(drop=True)

outdir = Path("analysis_outputs/microchips_confirmation_gate")
outdir.mkdir(parents=True, exist_ok=True)

summary_sorted.to_csv(outdir / "confirmation_summary.csv", index=False)
robust_only.to_csv(outdir / "confirmation_robust_only.csv", index=False)
day_results.to_csv(outdir / "confirmation_day_results.csv", index=False)
trade_log.to_csv(outdir / "confirmation_trade_log.csv", index=False)

print("\nTop CONFIRMATION SUMMARY:")
display(summary_sorted.head(50))

print("\nROBUST ONLY:")
display(robust_only.head(30))

if len(summary_sorted):
    best = summary_sorted.iloc[0]
    best_mode = best["gate_mode"]
    best_th = best["confirm_threshold"]

    best_day = day_results[
        (day_results["gate_mode"] == best_mode)
        & (
            (day_results["confirm_threshold"].isna() & pd.isna(best_th))
            | (day_results["confirm_threshold"] == best_th)
        )
    ]

    best_trades = trade_log[
        (trade_log["gate_mode"] == best_mode)
        & (
            (trade_log["confirm_threshold"].isna() & pd.isna(best_th))
            | (trade_log["confirm_threshold"] == best_th)
        )
    ]

    print("\nBest gate day breakdown:")
    display(best_day)

    print("\nBest gate trades:")
    display(best_trades.head(100))

print("\nSaved outputs to:", outdir)
print(f"Runtime: {time.time() - t0:.2f}s")

Days: [2, 3, 4]
Day 2: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 3: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 4: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Gate specs: 41
[   0.23s] gate 1/41: base_only, th=nan
[   0.25s] gate 2/41: require_square_agree, th=0.0
[   0.26s] gate 3/41: require_cubic_agree, th=0.0
[   0.27s] gate 4/41: require_either_agree, th=0.0
[   0.28s] gate 5/41: require_both_agree, th=0.0
[   0.30s] gate 6/41: veto_square_conflict, th=0.0
[   0.31s] gate 7/41: veto_cubic_conflict, th=0.0
[   0.32s] gate 8/41: veto_any_conflict, th=0.0
[   0.33s] gate 9/41: either_agree_no_conflict, th=0.0
[   0.34s] gate 10/41: require_square_agree, th=0.5
[   0.35s] gate 11/41: require_cubic_agree, th=0.5
[   0.37s] gate 12/41: require_either_agree, th=0.5
[   0.38s] gate 13/41: require_both_agree, th=0.5
[   0.39s] gate 14/41: veto_square_conflict, th=0.5
[   0.40s] gate 15/41: veto_cubic_conflict, th=0.5
[   0.42s] gate 16/41: veto_any_conflict, th=0.5
[   0

,gate_mode,confirm_threshold,days,active_days,positive_days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,base_only,NaN,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
1,either_agree_no_conflict,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
2,require_either_agree,0.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
3,require_either_agree,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
4,require_square_agree,0.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
5,require_square_agree,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
6,veto_any_conflict,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
7,veto_any_conflict,1.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
8,veto_any_conflict,1.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
9,veto_any_conflict,2.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00



ROBUST ONLY:


,gate_mode,confirm_threshold,days,active_days,positive_days,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,base_only,NaN,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
1,either_agree_no_conflict,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
2,require_either_agree,0.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
3,require_either_agree,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
4,require_square_agree,0.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
5,require_square_agree,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
6,veto_any_conflict,0.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
7,veto_any_conflict,1.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
8,veto_any_conflict,1.5,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00
9,veto_any_conflict,2.0,3,3,3,27110.0,9036.666667,2940.0,18160.0,27,0.730556,0.625,1004.074074,0.669864,True,27775.00



Best gate day breakdown:


,gate_mode,confirm_threshold,day,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl
0,base_only,NaN,2,8,6010.0,0.625000,751.250000,825.0
1,base_only,NaN,3,10,18160.0,0.900000,1816.000000,3567.5
2,base_only,NaN,4,9,2940.0,0.666667,326.666667,1080.0



Best gate trades:


,day,gate_mode,confirm_threshold,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_base_z,exit_base_z,square_z,cubic_z,square_status,cubic_status,hold,exec_pnl,q_trade,position
0,2,base_only,NaN,1701,1811,170100,181100,-1.0,2.516924,-0.163296,-1.574944,NaN,0,0,110,2955.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[-0.0, -0.0, 5.0, 5.0, -10.0]"
1,2,base_only,NaN,1953,2746,195300,274600,1.0,-2.557879,0.128124,3.307164,NaN,0,0,793,-1295.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[0.0, 0.0, -5.0, -5.0, 10.0]"
2,2,base_only,NaN,3175,3557,317500,355700,-1.0,2.682764,-0.046076,-1.511757,NaN,0,0,382,1490.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[-0.0, -0.0, 5.0, 5.0, -10.0]"
3,2,base_only,NaN,4643,4956,464300,495600,1.0,-2.670577,0.175206,2.160469,NaN,0,0,313,3255.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[0.0, 0.0, -5.0, -5.0, 10.0]"
4,2,base_only,NaN,5485,5552,548500,555200,-1.0,2.568288,-0.029867,-2.878163,-1.909830,0,0,67,2800.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[-0.0, -0.0, 5.0, 5.0, -10.0]"
5,2,base_only,NaN,5681,6378,568100,637800,-1.0,2.708747,-0.003748,-2.413861,-2.179235,0,0,697,160.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[-0.0, -0.0, 5.0, 5.0, -10.0]"
6,2,base_only,NaN,6585,7261,658500,726100,1.0,-2.719377,0.052595,3.238499,1.091424,0,0,676,-2000.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[0.0, 0.0, -5.0, -5.0, 10.0]"
7,2,base_only,NaN,9295,9995,929500,999500,-1.0,2.503122,-0.051640,-1.899529,-2.220576,0,0,700,-1355.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[-0.0, -0.0, 5.0, 5.0, -10.0]"
8,3,base_only,NaN,1289,1790,128900,179000,-1.0,2.706103,-0.059315,-1.222617,NaN,0,0,501,3485.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[-0.0, -0.0, 5.0, 5.0, -10.0]"
9,3,base_only,NaN,2128,2388,212800,238800,1.0,-2.582162,0.050768,2.451154,NaN,0,0,260,4525.0,"[0.0, 0.0, -5.0, -5.0, 10.0]","[0.0, 0.0, -5.0, -5.0, 10.0]"



Saved outputs to: analysis_outputs/microchips_confirmation_gate
Runtime: 0.80s
